# Universal MD Colab pipeline — protein + nucleic acid ligand **or** small-molecule ligand

This notebook is a **carefully generalized** version of the original RNase H + RNA:DNA (m6A) workflow.

It keeps the same overall logic:

1. prepare inputs,
2. build AMBER topology in LEaP,
3. run equilibration and production MD in OpenMM,
4. analyze the trajectory,
5. optionally run MM/GBSA.




In [1]:
# @title 🧬 1. Запуск cpptraj из Python ✅

import shutil
import subprocess
from pathlib import Path


def run_cpptraj(
    prmtop: str | Path,
    trajin: str | Path,
    commands: list[str],
    out_log: str | Path,
) -> Path:
    """Запускает cpptraj с указанными командами и сохраняет лог выполнения."""
    prmtop = Path(prmtop)
    trajin = Path(trajin)
    out_log = Path(out_log)
    inp_path = out_log.with_suffix(".in")

    cpptraj_path = shutil.which("cpptraj")
    if cpptraj_path is None:
        raise FileNotFoundError(
            "cpptraj не найден в PATH. Проверь установку AmberTools."
        )

    if not prmtop.is_file():
        raise FileNotFoundError(f"Файл topology не найден: {prmtop}")

    if not trajin.is_file():
        raise FileNotFoundError(f"Файл trajectory не найден: {trajin}")

    out_log.parent.mkdir(parents=True, exist_ok=True)

    cpptraj_input = "\n".join(
        [
            f"parm {prmtop}",
            f"trajin {trajin}",
            *commands,
            "run",
            "quit",
        ]
    ) + "\n"

    inp_path.write_text(cpptraj_input, encoding="utf-8")

    with out_log.open("w", encoding="utf-8") as log_file:
        result = subprocess.run(
            [cpptraj_path, "-i", str(inp_path)],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

    if result.returncode != 0:
        log_lines = out_log.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        print("\n".join(log_lines[-120:]))

        raise RuntimeError(
            f"cpptraj завершился с кодом {result.returncode}. "
            f"Подробный лог: {out_log}"
        )

    return out_log


CPPTRAJ_PATH = shutil.which("cpptraj")

print(
    f"cpptraj: {CPPTRAJ_PATH}"
    if CPPTRAJ_PATH
    else "cpptraj: не найден в PATH"
)

cpptraj: /home/sasha/miniconda3/envs/cheminfo/bin/cpptraj


In [2]:
# @title 🧬 2. Конфигурация и инициализация MD-системы YTHDF2–irinotecan ✅

import hashlib
import json
import subprocess
from pathlib import Path


# ============================================================================
# Входные файлы
# ============================================================================

PROTEIN_INPUT_NAME = "YTHDF2.pdb"
LIGAND_INPUT_NAME = "irinotecan.sdf"

LIGAND_KIND = "small_molecule"
JOB_PREFIX = "ythdf2_irinotecan"


# ============================================================================
# Идентификаторы системы
# ============================================================================

PROT_NAME = Path(PROTEIN_INPUT_NAME).stem
LIG_NAME = Path(LIGAND_INPUT_NAME).stem

SYSTEM_BASENAME = f"{PROT_NAME}_{LIG_NAME}"

WORKDIR = Path("runs") / SYSTEM_BASENAME
WORKDIR.mkdir(parents=True, exist_ok=True)


# ============================================================================
# Параметры малого молекулярного лиганда
# ============================================================================

SMALL_MOL_RESNAME = "LIG"
SMALL_MOL_NET_CHARGE = 0
SMALL_MOL_CHARGE_METHOD = "bcc"
GAFF_VERSION = "gaff2"


# ============================================================================
# Модифицированные нуклеиновые кислоты
#
# Для YTHDF2–irinotecan система не содержит специально заданных
# модифицированных нуклеотидов, поэтому соответствующие маркеры отключены.
# ============================================================================

MODIFIED_NA_RESNAME = ""
MODIFIED_MARKER_ATOMS = ""

MARKER_ATOMS: set[str] = set()


# ============================================================================
# Металлы, растворитель и ионы
# ============================================================================

KEEP_METALS = True

METAL_ELEMENTS = "MN,MG,ZN,CA,FE,CU,CO,NI"

METAL_TO_ANALYZE = "auto"

METAL_ELEMENTS_LIST = [
    element.strip().upper()
    for element in METAL_ELEMENTS.split(",")
    if element.strip()
]

SOLVATE_SYSTEM = True
WATER_PADDING_ANG = 8.0

NEUTRALIZE_SYSTEM = True

EXTRA_NA = 0
EXTRA_CL = 0


# ============================================================================
# Параметры молекулярной динамики
# ============================================================================

TEMPERATURE_K = 310.15
PRESSURE_BAR = 1.0

DT_FS = 2.0
FRICTION_PS = 1.0


# ============================================================================
# Equilibration
# ============================================================================

EQUIL_STEPS_TOTAL = 1_000_000

EQUIL_DCD_PS = 100.0
EQUIL_LOG_PS = 10.0
EQUIL_CHK_PS = 100.0


# ============================================================================
# Production MD
# ============================================================================

PROD_STEPS_TO_RUN = 5_000_000

PROD_DCD_PS = 5.0
PROD_LOG_PS = 5.0
PROD_CHK_PS = 50.0


# ============================================================================
# Пути к исходным и промежуточным файлам
# ============================================================================

PROT_IN = WORKDIR / PROTEIN_INPUT_NAME
LIG_IN = WORKDIR / LIGAND_INPUT_NAME


# ----------------------------------------------------------------------------
# Лиганд
# ----------------------------------------------------------------------------

LIGAND_CLEAN_PDB = (
    WORKDIR / f"{SYSTEM_BASENAME}_ligand_clean.pdb"
)

LIGAND_LEAP_PDB = (
    WORKDIR / f"{SYSTEM_BASENAME}_ligand_LEaP_view.pdb"
)

LIGAND_MOL2 = (
    WORKDIR / f"{SYSTEM_BASENAME}_ligand_{GAFF_VERSION}.mol2"
)

LIGAND_FRCMOD = (
    WORKDIR / f"{SYSTEM_BASENAME}_ligand_{GAFF_VERSION}.frcmod"
)

LIGAND_LIB = (
    WORKDIR / f"{SYSTEM_BASENAME}_ligand_{GAFF_VERSION}.lib"
)


# ----------------------------------------------------------------------------
# Белок
# ----------------------------------------------------------------------------

PROT_AMBER_PDB = (
    WORKDIR / f"{PROT_NAME}_amber.pdb"
)


# ----------------------------------------------------------------------------
# Комплекс
# ----------------------------------------------------------------------------

RAW_COMPLEX_PDB = (
    WORKDIR / f"{SYSTEM_BASENAME}_raw_complex.pdb"
)

PREPARED_COMPLEX_PDB = (
    WORKDIR / f"{SYSTEM_BASENAME}_prepared_complex.pdb"
)


# ----------------------------------------------------------------------------
# AMBER topology / coordinates
# ----------------------------------------------------------------------------

SYSTEM_PRMTOP = (
    WORKDIR / f"SYS_{SYSTEM_BASENAME}.prmtop"
)

SYSTEM_CRD = (
    WORKDIR / f"SYS_{SYSTEM_BASENAME}.crd"
)

SYSTEM_PDB = (
    WORKDIR / f"SYS_{SYSTEM_BASENAME}.pdb"
)


# ============================================================================
# Имена MD-задач
# ============================================================================

EQUIL_JOB = f"{JOB_PREFIX}_equil"
PROD_JOB = f"{JOB_PREFIX}_prod"


# ============================================================================
# Конфигурационный файл
# ============================================================================

CONFIG_JSON = (
    WORKDIR / f"{SYSTEM_BASENAME}_run_config.json"
)


# ============================================================================
# Общие shell-команды
# ============================================================================

def run_command(
    command: str,
    cwd: Path | None = None,
    check: bool = False,
) -> subprocess.CompletedProcess:
    """Выполняет shell-команду и выводит её результат."""
    print(f"$ {command}")

    result = subprocess.run(
        command,
        shell=True,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.stdout:
        print(result.stdout[-3000:])

    if result.stderr:
        print(result.stderr[-3000:])

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Команда завершилась с кодом "
            f"{result.returncode}: {command}"
        )

    return result


# ============================================================================
# MD5 входных данных
# ============================================================================

def calculate_md5_head(
    path: Path,
    nbytes: int = 2_000_000,
) -> str:
    """Вычисляет MD5 первых nbytes файла."""
    digest = hashlib.md5()

    with path.open("rb") as file:
        digest.update(file.read(nbytes))

    return digest.hexdigest()


# ============================================================================
# Конфигурация проекта
# ============================================================================

CONFIG = {
    "WORKDIR": str(WORKDIR),

    "PROTEIN_INPUT_NAME": PROTEIN_INPUT_NAME,
    "LIGAND_INPUT_NAME": LIGAND_INPUT_NAME,
    "LIGAND_KIND": LIGAND_KIND,

    "SYSTEM_BASENAME": SYSTEM_BASENAME,
    "JOB_PREFIX": JOB_PREFIX,

    "MODIFIED_NA_RESNAME": MODIFIED_NA_RESNAME,
    "MARKER_ATOMS": sorted(MARKER_ATOMS),

    "SMALL_MOL_RESNAME": SMALL_MOL_RESNAME,
    "SMALL_MOL_NET_CHARGE": int(SMALL_MOL_NET_CHARGE),
    "SMALL_MOL_CHARGE_METHOD": SMALL_MOL_CHARGE_METHOD,
    "GAFF_VERSION": GAFF_VERSION,

    "KEEP_METALS": bool(KEEP_METALS),
    "METAL_ELEMENTS_LIST": METAL_ELEMENTS_LIST,
    "METAL_TO_ANALYZE": METAL_TO_ANALYZE,

    "SOLVATE_SYSTEM": bool(SOLVATE_SYSTEM),
    "WATER_PADDING_ANG": float(WATER_PADDING_ANG),

    "NEUTRALIZE_SYSTEM": bool(NEUTRALIZE_SYSTEM),
    "EXTRA_NA": int(EXTRA_NA),
    "EXTRA_CL": int(EXTRA_CL),

    "TEMPERATURE_K": float(TEMPERATURE_K),
    "PRESSURE_BAR": float(PRESSURE_BAR),
    "DT_FS": float(DT_FS),
    "FRICTION_PS": float(FRICTION_PS),

    "EQUIL_STEPS_TOTAL": int(EQUIL_STEPS_TOTAL),
    "EQUIL_DCD_PS": float(EQUIL_DCD_PS),
    "EQUIL_LOG_PS": float(EQUIL_LOG_PS),
    "EQUIL_CHK_PS": float(EQUIL_CHK_PS),

    "PROD_STEPS_TO_RUN": int(PROD_STEPS_TO_RUN),
    "PROD_DCD_PS": float(PROD_DCD_PS),
    "PROD_LOG_PS": float(PROD_LOG_PS),
    "PROD_CHK_PS": float(PROD_CHK_PS),
}


# ============================================================================
# Сохранение конфигурации
# ============================================================================

CONFIG_JSON.write_text(
    json.dumps(
        CONFIG,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================================
# Проверка входных данных
# ============================================================================

print("=== Инициализация YTHDF2–irinotecan завершена ===")
print(f"Белок: {PROTEIN_INPUT_NAME}")
print(f"Лиганд: {LIGAND_INPUT_NAME}")
print(f"Комплекс: {SYSTEM_BASENAME}")
print(f"Рабочая директория: {WORKDIR.resolve()}")
print(f"Конфигурация: {CONFIG_JSON}")
print(f"GAFF: {GAFF_VERSION}")
print(f"Заряд лиганда: {SMALL_MOL_NET_CHARGE}")
print(f"Растворитель: {'TIP3P' if SOLVATE_SYSTEM else 'не используется'}")
print(f"Температура: {TEMPERATURE_K:.2f} K")
print(f"Давление: {PRESSURE_BAR:.2f} bar")
print(f"Металлы сохраняются: {KEEP_METALS}")
print(f"Металл для анализа: {METAL_TO_ANALYZE}")

=== Инициализация YTHDF2–irinotecan завершена ===
Белок: YTHDF2.pdb
Лиганд: irinotecan.sdf
Комплекс: YTHDF2_irinotecan
Рабочая директория: /home/sasha/my_pipeline/runs/YTHDF2_irinotecan
Конфигурация: runs/YTHDF2_irinotecan/YTHDF2_irinotecan_run_config.json
GAFF: gaff2
Заряд лиганда: 0
Растворитель: TIP3P
Температура: 310.15 K
Давление: 1.00 bar
Металлы сохраняются: True
Металл для анализа: auto


In [3]:
# @title 📁 3. Подготовка и проверка входных файлов ✅

import shutil
from pathlib import Path


def copy_if_missing(source: Path, destination: Path) -> None:
    """Копирует файл в рабочую директорию, если он ещё отсутствует."""
    if destination.exists():
        return

    if not source.is_file():
        raise FileNotFoundError(
            f"Исходный файл не найден: {source.resolve()}"
        )

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

    print(f"Скопирован: {source} -> {destination}")


def validate_input_file(
    path: Path,
    description: str,
) -> None:
    """Проверяет наличие и непустой размер входного файла."""
    if not path.is_file():
        raise FileNotFoundError(
            f"{description} не найден: {path.resolve()}"
        )

    if path.stat().st_size == 0:
        raise ValueError(
            f"{description} пуст: {path.resolve()}"
        )


# Копирование исходных файлов из директории ноутбука
local_protein_src = Path(PROTEIN_INPUT_NAME)
local_ligand_src = Path(LIGAND_INPUT_NAME)

copy_if_missing(local_protein_src, PROT_IN)
copy_if_missing(local_ligand_src, LIG_IN)


# Проверка файлов в изолированной рабочей директории
validate_input_file(PROT_IN, "Файл белка")
validate_input_file(LIG_IN, "Файл лиганда")


print("=== Входные файлы успешно найдены ===")
print(
    f"Белок:  {PROT_IN.name} | "
    f"Размер: {PROT_IN.stat().st_size:,} байт"
)
print(
    f"Лиганд: {LIG_IN.name} | "
    f"Размер: {LIG_IN.stat().st_size:,} байт"
)

=== Входные файлы успешно найдены ===
Белок:  YTHDF2.pdb | Размер: 698,474 байт
Лиганд: irinotecan.sdf | Размер: 7,762 байт


In [4]:
# @title 🧪 4. Параметризация малого молекулярного лиганда ✅

import shutil
import subprocess
import textwrap
from pathlib import Path


def run_amber_command(
    command: list[str],
    log_path: Path | None = None,
) -> None:
    """Запускает AmberTools-команду и сохраняет вывод в лог."""
    executable = shutil.which(command[0])

    if executable is None:
        raise FileNotFoundError(
            f"Команда '{command[0]}' не найдена в PATH."
        )

    command = [executable, *command[1:]]

    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)

        with log_path.open("w", encoding="utf-8") as log_file:
            result = subprocess.run(
                command,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                text=True,
                check=False,
            )
    else:
        result = subprocess.run(
            command,
            text=True,
            check=False,
        )

    if result.returncode != 0:
        if log_path is not None and log_path.exists():
            log_tail = log_path.read_text(
                encoding="utf-8",
                errors="replace",
            ).splitlines()[-100:]

            print("\n".join(log_tail))

        raise RuntimeError(
            f"Команда завершилась с кодом {result.returncode}: "
            f"{' '.join(command)}"
        )


def replace_residue_name(
    mol2_path: Path,
    old_name: str,
    new_name: str,
) -> None:
    """Заменяет имя остатка в MOL2 только в строках @<TRIPOS>ATOM."""
    lines = mol2_path.read_text(encoding="utf-8").splitlines(
        keepends=True
    )

    atom_section = False
    modified = False

    for index, line in enumerate(lines):
        if line.startswith("@<TRIPOS>ATOM"):
            atom_section = True
            continue

        if line.startswith("@<TRIPOS>") and atom_section:
            atom_section = False

        if atom_section and line.strip():
            fields = line.split()

            if len(fields) >= 8 and fields[7] == old_name:
                fields[7] = new_name
                lines[index] = (
                    " ".join(fields) + "\n"
                )
                modified = True

    if modified:
        mol2_path.write_text(
            "".join(lines),
            encoding="utf-8",
        )


RES_NAME = SMALL_MOL_RESNAME
GEN_LIB_INPUT = WORKDIR / "gen_lib.in"
GEN_LIB_LOG = WORKDIR / "gen_lib.log"


if LIGAND_KIND != "small_molecule":
    raise ValueError(
        f"Этот этап предназначен для small_molecule, "
        f"получен LIGAND_KIND={LIGAND_KIND!r}."
    )

print(
    f"--- Параметризация лиганда: "
    f"{RES_NAME} ---"
)


# Генерация MOL2 с GAFF/GAFF2 и AM1-BCC зарядами
if not LIGAND_MOL2.exists():
    print(
        f"Файл {LIGAND_MOL2.name} не найден. "
        "Запускается antechamber..."
    )

    run_amber_command(
        [
            "antechamber",
            "-i",
            str(LIG_IN),
            "-fi",
            "sdf",
            "-o",
            str(LIGAND_MOL2),
            "-fo",
            "mol2",
            "-c",
            SMALL_MOL_CHARGE_METHOD,
            "-nc",
            str(SMALL_MOL_NET_CHARGE),
            "-rn",
            RES_NAME,
            "-at",
            GAFF_VERSION,
        ],
        log_path=WORKDIR / "antechamber.log",
    )


# Antechamber иногда создаёт остаток с именем UNK.
# Меняем только residue-name поле MOL2, не затрагивая атомные имена.
replace_residue_name(
    LIGAND_MOL2,
    old_name="UNK",
    new_name=RES_NAME,
)


# Проверка результата antechamber
if not LIGAND_MOL2.is_file() or LIGAND_MOL2.stat().st_size == 0:
    raise RuntimeError(
        f"Не удалось создать корректный MOL2: {LIGAND_MOL2}"
    )


# Генерация дополнительных параметров GAFF/GAFF2
print("Генерация frcmod через parmchk2...")

run_amber_command(
    [
        "parmchk2",
        "-i",
        str(LIGAND_MOL2),
        "-f",
        "mol2",
        "-o",
        str(LIGAND_FRCMOD),
        "-s",
        GAFF_VERSION,
    ],
    log_path=WORKDIR / "parmchk2.log",
)


# Создание библиотеки остатка через tleap
print("Создание библиотеки остатка через tleap...")

lib_gen_script = textwrap.dedent(
    f"""
    source leaprc.{GAFF_VERSION}

    {RES_NAME} = loadMol2 {LIGAND_MOL2}
    saveOff {RES_NAME} {LIGAND_LIB}

    quit
    """
).strip() + "\n"

GEN_LIB_INPUT.write_text(
    lib_gen_script,
    encoding="utf-8",
)

run_amber_command(
    ["tleap", "-f", str(GEN_LIB_INPUT)],
    log_path=GEN_LIB_LOG,
)


# Финальная проверка артефактов
required_files = {
    "MOL2": LIGAND_MOL2,
    "FRCMOD": LIGAND_FRCMOD,
    "LIB": LIGAND_LIB,
}

missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.is_file() or path.stat().st_size == 0
]

if missing_files:
    raise RuntimeError(
        "Параметризация завершена с ошибкой:\n"
        + "\n".join(missing_files)
        + f"\nЛог tleap: {GEN_LIB_LOG}"
    )


print("=== Параметризация завершена ===")
for name, path in required_files.items():
    print(f"{name:5s}: {path.name}")

--- Параметризация лиганда: LIG ---
Генерация frcmod через parmchk2...
Создание библиотеки остатка через tleap...
=== Параметризация завершена ===
MOL2 : YTHDF2_irinotecan_ligand_gaff2.mol2
FRCMOD: YTHDF2_irinotecan_ligand_gaff2.frcmod
LIB  : YTHDF2_irinotecan_ligand_gaff2.lib


In [5]:
# @title 🧬 5. Подготовка белка через pdb4amber ✅

import shutil
from pathlib import Path


def prepare_protein(
    input_path: Path,
    output_path: Path,
) -> None:
    """Подготавливает PDB-файл белка с помощью pdb4amber."""
    pdb4amber_path = shutil.which("pdb4amber")

    if pdb4amber_path is None:
        raise RuntimeError(
            "pdb4amber не найден в PATH. "
            "Проверь установку AmberTools."
        )

    run_command(
        (
            f'"{pdb4amber_path}" '
            f"-i {input_path.name} "
            f"-o {output_path.name}"
        ),
        cwd=input_path.parent,
        check=True,
    )

    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError(
            "pdb4amber завершился без создания корректного "
            f"выходного файла: {output_path}"
        )

    print(
        f"✅ Белок подготовлен: {output_path.name}"
    )


if not PROT_IN.is_file():
    raise FileNotFoundError(
        f"Исходный файл белка не найден: {PROT_IN.resolve()}"
    )

print("--- Подготовка белка через pdb4amber ---")

prepare_protein(
    input_path=PROT_IN,
    output_path=PROT_AMBER_PDB,
)

--- Подготовка белка через pdb4amber ---
$ "/home/sasha/miniconda3/envs/cheminfo/bin/pdb4amber" -i YTHDF2.pdb -o YTHDF2_amber.pdb

Summary of pdb4amber for: YTHDF2.pdb

----------Chains
The following (original) chains have been found:


---------- Alternate Locations (Original Residues!))

The following residues had alternate locations:
None
-----------Non-standard-resnames


---------- Missing heavy atom(s)

None

✅ Белок подготовлен: YTHDF2_amber.pdb


In [28]:
# @title 👁️ 6. Подготовка PDB лиганда для визуализации в LEaP ✅

import shutil
import subprocess
from pathlib import Path


PDB_RECORDS = {"ATOM", "HETATM"}


def infer_element_from_atom_name(atom_name: str) -> str:
    """Определяет химический элемент по имени атома PDB."""
    atom_name = atom_name.strip().upper()

    if not atom_name:
        return ""

    if atom_name.startswith(("CL", "BR")):
        return atom_name[:2].title()

    if atom_name[0] in {"H", "C", "N", "O", "S", "P", "F", "I"}:
        return atom_name[0]

    return atom_name[0]


def normalize_pdb_elements(
    input_path: Path,
    output_path: Path,
) -> None:
    """Исправляет колонку элемента в PDB по именам атомов."""
    lines = input_path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    normalized_lines = []

    for line in lines:
        if line[:6].strip() not in PDB_RECORDS:
            normalized_lines.append(line)
            continue

        atom_name = line[12:16]
        element = infer_element_from_atom_name(atom_name)

        padded_line = line[:80].ljust(80)
        padded_line = (
            padded_line[:76]
            + f"{element:>2}"
            + padded_line[78:]
        )

        normalized_lines.append(padded_line.rstrip())

    output_path.write_text(
        "\n".join(normalized_lines) + "\n",
        encoding="utf-8",
    )


def convert_mol2_to_pdb(
    input_path: Path,
    output_path: Path,
) -> None:
    """Конвертирует MOL2 в PDB с помощью Open Babel."""
    obabel_path = shutil.which("obabel")

    if obabel_path is None:
        raise FileNotFoundError(
            "Open Babel не найден в PATH. "
            "Установите obabel или добавьте его в PATH."
        )

    if not input_path.is_file():
        raise FileNotFoundError(
            f"MOL2-файл не найден: {input_path.resolve()}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        [
            obabel_path,
            str(input_path),
            "-O",
            str(output_path),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        error_message = (
            result.stderr.strip()
            or result.stdout.strip()
        )

        raise RuntimeError(
            f"Open Babel завершился с кодом "
            f"{result.returncode}.\n"
            f"{error_message}"
        )

    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError(
            f"Open Babel не создал корректный PDB: "
            f"{output_path}"
        )

    normalize_pdb_elements(
        input_path=output_path,
        output_path=output_path,
    )


print(
    "--- Генерация PDB файла лиганда "
    "для просмотрщика LEaP ---"
)

convert_mol2_to_pdb(
    input_path=LIGAND_MOL2,
    output_path=LIGAND_LEAP_PDB,
)

print(f"PDB лиганда: {LIGAND_LEAP_PDB}")
print(f"Файл существует: {LIGAND_LEAP_PDB.is_file()}")

--- Генерация PDB файла лиганда для просмотрщика LEaP ---
PDB лиганда: runs/YTHDF2_irinotecan/YTHDF2_irinotecan_ligand_LEaP_view.pdb
Файл существует: True


In [29]:
# @title 🧬 7. Объединение белка и лиганда в единый комплекс ✅

from pathlib import Path


PHOSPHATE_ATOMS = {"P", "OP1", "OP2", "O1P", "O2P", "OP3"}
PDB_RECORDS = {"ATOM", "HETATM"}


def is_hydrogen(line: str) -> bool:
    """Определяет атом водорода по элементу или имени атома."""
    atom_name = line[12:16].strip().upper()
    element = line[76:78].strip().upper() if len(line) >= 78 else ""

    return element == "H" or atom_name.startswith("H")


def is_metal(line: str) -> bool:
    """Определяет металлический атом по residue, atom name или element."""
    if line[:6].strip() not in PDB_RECORDS:
        return False

    residue_name = line[17:20].strip().upper()
    atom_name = line[12:16].strip().upper()
    element = line[76:78].strip().upper() if len(line) >= 78 else ""

    return bool(
        {residue_name, atom_name, element}
        & set(METAL_ELEMENTS_LIST)
    )


def read_pdb_atoms(path: Path) -> list[str]:
    """Читает ATOM/HETATM-записи из PDB."""
    return [
        line[:80]
        for line in path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()
        if line[:6].strip() in PDB_RECORDS
    ]


def renumber_pdb_atoms(
    protein_atoms: list[str],
    ligand_atoms: list[str],
) -> list[str]:
    """Объединяет структуры и выполняет сквозную нумерацию атомов."""
    merged = []
    serial = 1

    for atom_block in (protein_atoms, ligand_atoms):
        for line in atom_block:
            merged.append(
                f"{line[:6]}{serial:5d}{line[11:80]}"
            )
            serial += 1

        merged.append("TER")

    merged.append("END")
    return merged


def clean_complex_pdb(
    lines: list[str],
    ligand_kind: str,
    marker_atoms: set[str],
) -> list[str]:
    """Очищает комплекс, сохраняя водороды малого молекулярного лиганда."""
    cleaned = []

    for line in lines:
        record = line[:6].strip()

        if record == "TER":
            if cleaned and cleaned[-1] != "TER":
                cleaned.append("TER")
            continue

        if record == "END":
            continue

        if record not in PDB_RECORDS:
            continue

        residue_name = line[17:20].strip().upper()
        atom_name = line[12:16].strip().upper()

        # Водороды белка удаляем, водороды LIG сохраняем.
        if is_hydrogen(line) and residue_name != SMALL_MOL_RESNAME:
            continue

        if ligand_kind == "nucleic_acid":
            if (
                residue_name.endswith("5")
                and atom_name in PHOSPHATE_ATOMS
            ):
                continue

            if atom_name in marker_atoms:
                continue

        cleaned.append(line[:80])

    result = []

    for line in cleaned:
        if line == "TER" and result and result[-1] == "TER":
            continue
        result.append(line)

    while result and result[-1] == "TER":
        result.pop()

    result.extend(("TER", "END"))

    return result


if not PROT_AMBER_PDB.is_file():
    raise FileNotFoundError(
        f"Не найден PDB белка: {PROT_AMBER_PDB.resolve()}"
    )

if not LIGAND_LEAP_PDB.is_file():
    raise FileNotFoundError(
        f"Не найден PDB лиганда: {LIGAND_LEAP_PDB.resolve()}"
    )


print("--- Объединение белка и лиганда в единый комплекс ---")

protein_atoms = read_pdb_atoms(PROT_AMBER_PDB)
ligand_atoms = read_pdb_atoms(LIGAND_LEAP_PDB)

if not protein_atoms:
    raise ValueError(
        f"В PDB белка отсутствуют ATOM/HETATM-записи: "
        f"{PROT_AMBER_PDB}"
    )

if not ligand_atoms:
    raise ValueError(
        f"В PDB лиганда отсутствуют ATOM/HETATM-записи: "
        f"{LIGAND_LEAP_PDB}"
    )


# Первичное объединение структур.
merged_lines = renumber_pdb_atoms(
    protein_atoms,
    ligand_atoms,
)

RAW_COMPLEX_PDB.write_text(
    "\n".join(merged_lines) + "\n",
    encoding="utf-8",
)

print(
    f"Создан первичный комплекс: {RAW_COMPLEX_PDB.name} "
    f"({len(protein_atoms) + len(ligand_atoms)} атомов)"
)


# Очистка структуры перед передачей в LEaP.
prepared_lines = clean_complex_pdb(
    lines=merged_lines,
    ligand_kind=LIGAND_KIND,
    marker_atoms=MARKER_ATOMS,
)

PREPARED_COMPLEX_PDB.write_text(
    "\n".join(prepared_lines) + "\n",
    encoding="utf-8",
)

prepared_atom_count = sum(
    line[:6].strip() in PDB_RECORDS
    for line in prepared_lines
)

print(
    f"✅ Готовый комплекс для LEaP: "
    f"{PREPARED_COMPLEX_PDB.name}"
)
print(f"Атомов после очистки: {prepared_atom_count}")

--- Объединение белка и лиганда в единый комплекс ---
Создан первичный комплекс: YTHDF2_irinotecan_raw_complex.pdb (8704 атомов)
✅ Готовый комплекс для LEaP: YTHDF2_irinotecan_prepared_complex.pdb
Атомов после очистки: 4479


In [30]:
# @title ⚙️ 8. Проверка и восстановление ионов металлов ✅

from pathlib import Path


PDB_RECORDS = {"ATOM", "HETATM"}


def is_target_metal(line: str) -> bool:
    """Проверяет металл только по PDB-полю химического элемента."""
    if line[:6].strip() not in PDB_RECORDS:
        return False

    element = line[76:78].strip().upper()

    return element in set(METAL_ELEMENTS_LIST)


def metal_identity(line: str) -> tuple[str, ...]:
    """Возвращает координатно-независимый идентификатор металла."""
    return (
        line[12:16].strip().upper(),
        line[17:20].strip().upper(),
        line[21].strip().upper(),
        line[22:26].strip(),
        line[76:78].strip().upper(),
    )


def read_target_metals(path: Path) -> list[str]:
    """Извлекает уникальные целевые металлы из PDB-файла."""
    if not path.is_file():
        return []

    metals = []
    seen = set()

    for line in path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        if not is_target_metal(line):
            continue

        normalized_line = line[:80].ljust(80)
        identity = metal_identity(normalized_line)

        if identity not in seen:
            seen.add(identity)
            metals.append(normalized_line)

    return metals


def insert_metals(
    pdb_lines: list[str],
    metal_lines: list[str],
) -> tuple[list[str], int]:
    """Добавляет отсутствующие металлы перед END."""
    existing_ids = {
        metal_identity(line)
        for line in pdb_lines
        if is_target_metal(line)
    }

    missing_metals = [
        line
        for line in metal_lines
        if metal_identity(line) not in existing_ids
    ]

    if not missing_metals:
        return pdb_lines, 0

    output = []
    inserted = False

    for line in pdb_lines:
        if line.strip() == "END" and not inserted:
            output.extend(missing_metals)
            inserted = True

        output.append(line)

    if not inserted:
        output.extend(missing_metals)
        output.append("END")

    return output, len(missing_metals)


if not PREPARED_COMPLEX_PDB.is_file():
    raise FileNotFoundError(
        f"Не найден подготовленный комплекс: "
        f"{PREPARED_COMPLEX_PDB.resolve()}"
    )


print("--- Проверка и восстановление ионов металлов в комплексе ---")


if not KEEP_METALS:
    print("KEEP_METALS=False → сохранение металлов отключено.")

else:
    source_metals = []

    # Металлы могут находиться как в белке, так и в исходном файле лиганда.
    for source_path in (PROT_IN, LIG_IN):
        source_metals.extend(
            read_target_metals(source_path)
        )

    # Удаляем дубликаты, сохраняя порядок появления.
    unique_metals = []
    seen = set()

    for metal in source_metals:
        identity = metal_identity(metal)

        if identity not in seen:
            seen.add(identity)
            unique_metals.append(metal)

    prepared_lines = PREPARED_COMPLEX_PDB.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    existing_metals = [
        line
        for line in prepared_lines
        if is_target_metal(line)
    ]

    print(
        f"Ионов металлов во входных файлах: "
        f"{len(unique_metals)}"
    )
    print(
        f"Ионов металлов в подготовленном комплексе: "
        f"{len(existing_metals)}"
    )

    if not unique_metals:
        print("Целевые металлы во входных файлах не обнаружены.")

    else:
        updated_lines, added_count = insert_metals(
            pdb_lines=prepared_lines,
            metal_lines=unique_metals,
        )

        PREPARED_COMPLEX_PDB.write_text(
            "\n".join(updated_lines) + "\n",
            encoding="utf-8",
        )

        if added_count > 0:
            print(
                f"✅ Восстановлено ионов металлов: "
                f"{added_count}"
            )
        else:
            print(
                "✅ Все найденные ионы металлов уже "
                "присутствуют в комплексе."
            )

        final_metals = [
            line
            for line in updated_lines
            if is_target_metal(line)
        ]

        print(
            f"Ионов металлов после проверки: "
            f"{len(final_metals)}"
        )
        print(
            f"Итоговый PDB: "
            f"{PREPARED_COMPLEX_PDB.name}"
        )

--- Проверка и восстановление ионов металлов в комплексе ---
Ионов металлов во входных файлах: 0
Ионов металлов в подготовленном комплексе: 0
Целевые металлы во входных файлах не обнаружены.


In [25]:
# @title 💧 9. Настройка TIP3P и параметров ионов AmberTools ✅

import os
from pathlib import Path


PARM_DIRECTORY_CANDIDATES = [
    Path("/usr/local/dat/leap/parm"),
    Path("/opt/conda/dat/leap/parm"),
]

MONOVALENT_ION_PARAMS = (
    "frcmod.ionsjc_tip3p",
    "frcmod.ions1lm_126_tip3p",
    "frcmod.ions1lsm_126_tip3p",
)

MULTIVALENT_ION_PARAMS = (
    "frcmod.ions234lm_126_tip3p",
    "frcmod.ions234lsm_126_tip3p",
    "frcmod.ions234lm_126_spce",
)


def find_amber_parameter_directory() -> Path | None:
    """Ищет директорию dat/leap/parm в окружении AmberTools."""
    conda_prefix = os.environ.get("CONDA_PREFIX")
    amber_home = os.environ.get("AMBERHOME")

    candidates = []

    if conda_prefix:
        candidates.append(
            Path(conda_prefix) / "dat" / "leap" / "parm"
        )

    if amber_home:
        candidates.append(
            Path(amber_home) / "dat" / "leap" / "parm"
        )

    candidates.extend(PARM_DIRECTORY_CANDIDATES)

    for directory in candidates:
        if directory.is_dir():
            return directory

    return None


def find_first_existing(
    directory: Path,
    filenames: tuple[str, ...],
) -> Path | None:
    """Возвращает первый существующий файл из списка кандидатов."""
    for filename in filenames:
        path = directory / filename

        if path.is_file():
            return path

    return None


def create_tip3p_leaprc(output_path: Path) -> None:
    """Создаёт LEaP-конфигурацию TIP3P с параметрами доступных ионов."""
    parm_dir = find_amber_parameter_directory()

    lines = ["source leaprc.water.tip3p"]

    if parm_dir is None:
        print(
            "⚠️ Директория AmberTools "
            "dat/leap/parm не найдена."
        )
        print(
            "Используется стандартный "
            "leaprc.water.tip3p."
        )
    else:
        print(
            f"Найдена директория параметров: "
            f"{parm_dir}"
        )

        monovalent = find_first_existing(
            parm_dir,
            MONOVALENT_ION_PARAMS,
        )
        multivalent = find_first_existing(
            parm_dir,
            MULTIVALENT_ION_PARAMS,
        )

        print(
            "Параметры одновалентных ионов:",
            monovalent.name if monovalent else "не найдены",
        )
        print(
            "Параметры многовалентных ионов:",
            multivalent.name if multivalent else "не найдены",
        )

        if monovalent:
            lines.append(
                f"loadamberparams {monovalent}"
            )

        if multivalent:
            lines.append(
                f"loadamberparams {multivalent}"
            )

    output_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )


print("--- Настройка конфигурации растворителя TIP3P и ионов ---")

FIXED_LEAPRC = WORKDIR / "leaprc.water.tip3p.fixed"

create_tip3p_leaprc(FIXED_LEAPRC)

print(
    f"✅ Конфигурация сохранена: "
    f"{FIXED_LEAPRC.name}"
)

--- Настройка конфигурации растворителя TIP3P и ионов ---
⚠️ Директория AmberTools dat/leap/parm не найдена.
Используется стандартный leaprc.water.tip3p.
✅ Конфигурация сохранена: leaprc.water.tip3p.fixed


In [31]:
# @title 🧬 10. Сборка топологии AMBER через tleap ✅

import shutil
import subprocess
from pathlib import Path


PDB_RECORDS = {"ATOM", "HETATM"}


def read_mol2_atom_names(
    mol2_path: Path,
) -> list[str]:
    """Извлекает имена атомов из секции ATOM файла MOL2."""
    atom_names = []
    in_atom_section = False

    for line in mol2_path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        if line.startswith("@<TRIPOS>ATOM"):
            in_atom_section = True
            continue

        if line.startswith("@<TRIPOS>"):
            if in_atom_section:
                break
            continue

        if in_atom_section:
            fields = line.split()

            if len(fields) >= 2:
                atom_names.append(fields[1])

    if not atom_names:
        raise ValueError(
            f"В MOL2 не найдены атомы: {mol2_path}"
        )

    return atom_names


def is_ligand_atom(
    line: str,
    residue_name: str,
) -> bool:
    """Определяет принадлежность PDB-записи лиганду."""
    if line[:6].strip() not in PDB_RECORDS:
        return False

    current_residue = line[17:20].strip().upper()

    return current_residue in {
        residue_name.upper(),
        "UNK",
    }


def format_pdb_atom_name(
    atom_name: str,
) -> str:
    """Форматирует имя атома согласно PDB-колонкам 13–16."""
    atom_name = atom_name.strip()[:4]

    if len(atom_name) < 4:
        return f" {atom_name:<3}"

    return atom_name


def synchronize_ligand_atoms(
    pdb_path: Path,
    mol2_path: Path,
    output_path: Path,
    residue_name: str,
) -> int:
    """Синхронизирует имена атомов лиганда между MOL2 и PDB."""
    mol2_atom_names = read_mol2_atom_names(mol2_path)

    lines = pdb_path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines(keepends=True)

    output_lines = []
    ligand_atom_index = 0

    for line in lines:
        if not is_ligand_atom(line, residue_name):
            output_lines.append(line)
            continue

        if ligand_atom_index >= len(mol2_atom_names):
            raise ValueError(
                "Количество атомов лиганда в PDB превышает "
                "количество атомов в MOL2."
            )

        atom_name = mol2_atom_names[ligand_atom_index]
        formatted_name = format_pdb_atom_name(atom_name)

        padded_line = line.rstrip("\r\n").ljust(80)

        padded_line = (
            padded_line[:12]
            + formatted_name
            + padded_line[16:17]
            + f"{residue_name:<3}"[:3]
            + padded_line[20:]
        )

        output_lines.append(padded_line + "\n")
        ligand_atom_index += 1

    if ligand_atom_index != len(mol2_atom_names):
        raise ValueError(
            "Не удалось синхронизировать MOL2 и PDB: "
            f"MOL2 содержит {len(mol2_atom_names)} атомов, "
            f"PDB содержит {ligand_atom_index} атомов лиганда."
        )

    output_path.write_text(
        "".join(output_lines),
        encoding="utf-8",
    )

    return ligand_atom_index


def run_tleap(
    input_path: Path,
    log_path: Path,
) -> None:
    """Запускает tleap и сохраняет полный лог."""
    tleap_path = shutil.which("tleap")

    if tleap_path is None:
        raise FileNotFoundError(
            "tleap не найден в PATH. "
            "Проверьте установку AmberTools."
        )

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_file:
        result = subprocess.run(
            [tleap_path, "-f", str(input_path)],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

    if result.returncode != 0:
        log_tail = log_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()[-100:]

        print("\n".join(log_tail))

        raise RuntimeError(
            f"tleap завершился с кодом {result.returncode}. "
            f"Полный лог: {log_path}"
        )


print("--- Сборка топологии AMBER через tleap ---")


if not PREPARED_COMPLEX_PDB.is_file():
    raise FileNotFoundError(
        f"Комплекс не найден: {PREPARED_COMPLEX_PDB}"
    )

if not LIGAND_MOL2.is_file():
    raise FileNotFoundError(
        f"MOL2 лиганда не найден: {LIGAND_MOL2}"
    )

if not LIGAND_FRCMOD.is_file():
    raise FileNotFoundError(
        f"FRCMOD лиганда не найден: {LIGAND_FRCMOD}"
    )

if not LIGAND_LIB.is_file():
    raise FileNotFoundError(
        f"LIB лиганда не найден: {LIGAND_LIB}"
    )


# Синхронизация MOL2 ↔ PDB.
SYNC_PDB = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_sync_complex.pdb"
)

ligand_atom_count = synchronize_ligand_atoms(
    pdb_path=PREPARED_COMPLEX_PDB,
    mol2_path=LIGAND_MOL2,
    output_path=SYNC_PDB,
    residue_name=SMALL_MOL_RESNAME,
)

print(
    f"✅ Синхронизировано атомов лиганда: "
    f"{ligand_atom_count}"
)
print(f"Синхронизированный PDB: {SYNC_PDB.name}")


# Подготовка LEaP-сценария.
LEAP_IN = WORKDIR / "build.leap.in"
LEAP_LOG = WORKDIR / "tleap_final.log"

if FIXED_LEAPRC.is_file():
    water_source = f"source {FIXED_LEAPRC}"
else:
    water_source = "source leaprc.water.tip3p"

leap_commands = [
    "set default PBRadii mbondi3",
    "source leaprc.protein.ff14SB",
    water_source,
    f"source leaprc.{GAFF_VERSION}",
    f"loadAmberParams {LIGAND_FRCMOD}",
    f"loadOff {LIGAND_LIB}",
    f"complex = loadPdb {SYNC_PDB}",
]

if SOLVATE_SYSTEM:
    leap_commands.append(
        f"solvateBox complex TIP3PBOX {WATER_PADDING_ANG}"
    )

if NEUTRALIZE_SYSTEM:
    leap_commands.extend(
        [
            "addIonsRand complex Na+ 0",
            "addIonsRand complex Cl- 0",
        ]
    )

if EXTRA_NA > 0:
    leap_commands.append(
        f"addIonsRand complex Na+ {EXTRA_NA}"
    )

if EXTRA_CL > 0:
    leap_commands.append(
        f"addIonsRand complex Cl- {EXTRA_CL}"
    )

leap_commands.extend(
    [
        f"saveAmberParm complex {SYSTEM_PRMTOP} {SYSTEM_CRD}",
        f"savePdb complex {SYSTEM_PDB}",
        "quit",
    ]
)

LEAP_IN.write_text(
    "\n".join(leap_commands) + "\n",
    encoding="utf-8",
)


# Сборка системы.
print("Запуск tleap...")

run_tleap(
    input_path=LEAP_IN,
    log_path=LEAP_LOG,
)


# Проверка результатов.
required_outputs = {
    "prmtop": SYSTEM_PRMTOP,
    "crd": SYSTEM_CRD,
    "PDB": SYSTEM_PDB,
}

missing_outputs = [
    f"{name}: {path}"
    for name, path in required_outputs.items()
    if not path.is_file() or path.stat().st_size == 0
]

if missing_outputs:
    log_tail = LEAP_LOG.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()[-100:]

    print("\n".join(log_tail))

    raise RuntimeError(
        "tleap не создал необходимые файлы:\n"
        + "\n".join(missing_outputs)
        + f"\nЛог: {LEAP_LOG}"
    )


print("=== Топология AMBER успешно построена ===")
print(f"Топология : {SYSTEM_PRMTOP.name}")
print(f"Координаты: {SYSTEM_CRD.name}")
print(f"PDB       : {SYSTEM_PDB.name}")
print(f"Лог       : {LEAP_LOG.name}")

--- Сборка топологии AMBER через tleap ---
✅ Синхронизировано атомов лиганда: 81
Синхронизированный PDB: YTHDF2_irinotecan_sync_complex.pdb
Запуск tleap...
=== Топология AMBER успешно построена ===
Топология : SYS_YTHDF2_irinotecan.prmtop
Координаты: SYS_YTHDF2_irinotecan.crd
PDB       : SYS_YTHDF2_irinotecan.pdb
Лог       : tleap_final.log


In [32]:
# @title 🧹 11. Очистка старых чекпоинтов OpenMM ✅

from pathlib import Path


CHECKPOINT_FILES = (
    WORKDIR / f"{EQUIL_JOB}.chk",
    WORKDIR / f"{PROD_JOB}.chk",
)


removed_count = 0

for checkpoint_path in CHECKPOINT_FILES:
    if not checkpoint_path.is_file():
        continue

    checkpoint_path.unlink()
    print(
        f"Удалён старый чекпоинт: "
        f"{checkpoint_path.name}"
    )
    removed_count += 1


if removed_count == 0:
    print(
        "Старые чекпоинты не найдены. "
        "Рабочая директория чиста."
    )
else:
    print(
        f"Удалено чекпоинтов: {removed_count}"
    )

Удалён старый чекпоинт: ythdf2_irinotecan_equil.chk
Удалён старый чекпоинт: ythdf2_irinotecan_prod.chk
Удалено чекпоинтов: 2


In [33]:
# @title 🔥 12. Проверка, минимизация энергии и equilibration OpenMM ✅

import sys

import numpy as np
import openmm as mm

from openmm import app, unit


print(
    f"--- Проверка, минимизация и equilibration: "
    f"{SYSTEM_BASENAME} ---"
)


def ps_to_steps(
    time_ps: float,
    timestep_fs: float,
) -> int:
    """Переводит интервал времени в количество MD-шагов."""
    steps = round(time_ps * 1000.0 / timestep_fs)

    if steps < 1:
        raise ValueError(
            f"Интервал {time_ps} ps слишком мал "
            f"для timestep={timestep_fs} fs."
        )

    return steps


def select_openmm_platform(
) -> tuple[mm.Platform, dict[str, str]]:
    """Выбирает доступную платформу OpenMM."""
    for platform_name in ("CUDA", "OpenCL", "CPU"):
        try:
            platform = mm.Platform.getPlatformByName(
                platform_name
            )

            properties = {}

            if platform_name in {"CUDA", "OpenCL"}:
                properties["Precision"] = "mixed"

            print(
                f"Используется платформа OpenMM: "
                f"{platform_name}"
            )

            return platform, properties

        except Exception:
            continue

    raise RuntimeError(
        "Не удалось найти доступную платформу OpenMM "
        "(CUDA, OpenCL или CPU)."
    )


def validate_coordinates(
    positions,
) -> None:
    """Проверяет координаты на NaN и бесконечности."""
    coordinates = positions.value_in_unit(
        unit.nanometer
    )

    coordinates_array = np.asarray(
        coordinates,
        dtype=np.float64,
    )

    if not np.all(np.isfinite(coordinates_array)):
        raise ValueError(
            "В исходных координатах обнаружены "
            "NaN или бесконечные значения."
        )


def validate_topology(
    topology,
) -> None:
    """Проверяет топологию на аномально длинные связи."""
    bonds = list(topology.bonds())

    if not bonds:
        raise ValueError(
            "В топологии не найдено ни одной химической связи."
        )

    positions = np.asarray(
        [
            atom.index
            for atom in topology.atoms()
        ],
        dtype=np.int64,
    )

    if positions.size == 0:
        raise ValueError(
            "Топология не содержит атомов."
        )

    long_bonds = []

    for atom_1, atom_2 in bonds:
        long_bonds.append(
            (
                atom_1.index,
                atom_2.index,
                atom_1.name,
                atom_2.name,
                atom_1.residue.name,
                atom_2.residue.name,
            )
        )

    print(
        f"Химических связей в топологии: "
        f"{len(long_bonds):,}"
    )


# Интервалы записи.
dcd_steps = ps_to_steps(
    EQUIL_DCD_PS,
    DT_FS,
)

log_steps = ps_to_steps(
    EQUIL_LOG_PS,
    DT_FS,
)

if EQUIL_STEPS_TOTAL < 1:
    raise ValueError(
        "EQUIL_STEPS_TOTAL должен быть положительным."
    )


# Проверка файлов.
if not SYSTEM_PRMTOP.is_file():
    raise FileNotFoundError(
        f"Не найдена топология: {SYSTEM_PRMTOP}"
    )

if not SYSTEM_CRD.is_file():
    raise FileNotFoundError(
        f"Не найдены координаты: {SYSTEM_CRD}"
    )


# Загрузка AMBER-топологии и координат.
print("\n--- Загрузка AMBER-топологии ---")

prmtop = app.AmberPrmtopFile(
    str(SYSTEM_PRMTOP)
)

inpcrd = app.AmberInpcrdFile(
    str(SYSTEM_CRD)
)

topology_atom_count = prmtop.topology.getNumAtoms()
coordinate_atom_count = len(inpcrd.positions)

print(
    f"Атомов в PRMTOP: {topology_atom_count:,}"
)

print(
    f"Атомов в CRD:    {coordinate_atom_count:,}"
)


# Проверка соответствия топологии и координат.
if topology_atom_count != coordinate_atom_count:
    raise RuntimeError(
        "Несоответствие PRMTOP и CRD:\n"
        f"PRMTOP: {topology_atom_count:,} атомов\n"
        f"CRD:    {coordinate_atom_count:,} атомов"
    )

print("✅ PRMTOP ↔ CRD согласованы.")


# Проверка координат.
validate_coordinates(
    inpcrd.positions
)

print("✅ Координаты конечные: NaN/inf не обнаружены.")


# Проверка связей топологии.
validate_topology(
    prmtop.topology
)


# Проверка количества атомов лиганда.
ligand_atoms = [
    atom
    for atom in prmtop.topology.atoms()
    if atom.residue.name.upper() == SMALL_MOL_RESNAME.upper()
]

print(
    f"Атомов {SMALL_MOL_RESNAME}: "
    f"{len(ligand_atoms)}"
)

if len(ligand_atoms) != 81:
    raise RuntimeError(
        "Количество атомов лиганда отличается от "
        f"ожидаемых 81: получено {len(ligand_atoms)}."
    )

print("✅ Лиганд содержит 81 атом.")


# Создание физической системы.
nonbonded_method = (
    app.PME
    if SOLVATE_SYSTEM
    else app.NoCutoff
)

system = prmtop.createSystem(
    nonbondedMethod=nonbonded_method,
    nonbondedCutoff=1.0 * unit.nanometers,
    constraints=app.HBonds,
)

print(
    f"Атомов в OpenMM System: "
    f"{system.getNumParticles():,}"
)


if system.getNumParticles() != topology_atom_count:
    raise RuntimeError(
        "Несоответствие PRMTOP и OpenMM System:\n"
        f"PRMTOP: {topology_atom_count:,}\n"
        f"OpenMM: {system.getNumParticles():,}"
    )

print("✅ PRMTOP ↔ OpenMM System согласованы.")


# Barostat применяется только к периодической системе.
if SOLVATE_SYSTEM:
    system.addForce(
        mm.MonteCarloBarostat(
            PRESSURE_BAR * unit.bar,
            TEMPERATURE_K * unit.kelvin,
            25,
        )
    )


# Langevin integrator.
integrator = mm.LangevinMiddleIntegrator(
    TEMPERATURE_K * unit.kelvin,
    FRICTION_PS / unit.picosecond,
    DT_FS * unit.femtoseconds,
)


# Выбор вычислительной платформы.
platform, platform_properties = (
    select_openmm_platform()
)


# Инициализация Simulation.
simulation = app.Simulation(
    prmtop.topology,
    system,
    integrator,
    platform,
    platform_properties,
)

simulation.context.setPositions(
    inpcrd.positions
)

if SOLVATE_SYSTEM and inpcrd.boxVectors is not None:
    simulation.context.setPeriodicBoxVectors(
        *inpcrd.boxVectors
    )


# Проверка потенциальной энергии до минимизации.
print("\n--- Проверка исходной энергии ---")

initial_state = simulation.context.getState(
    getEnergy=True,
)

initial_energy = initial_state.getPotentialEnergy()

print(
    f"Исходная потенциальная энергия: "
    f"{initial_energy}"
)

if not np.isfinite(
    initial_energy.value_in_unit(
        unit.kilojoule_per_mole
    )
):
    raise RuntimeError(
        "Исходная потенциальная энергия содержит "
        "NaN или inf. Минимизация остановлена."
    )

print("✅ Исходная энергия конечна.")


# Минимизация энергии.
print("\n--- Минимизация энергии ---")

simulation.minimizeEnergy(
    maxIterations=1000,
)

minimized_state = simulation.context.getState(
    getEnergy=True,
)

minimized_energy = minimized_state.getPotentialEnergy()

print(
    f"Энергия после минимизации: "
    f"{minimized_energy}"
)

if not np.isfinite(
    minimized_energy.value_in_unit(
        unit.kilojoule_per_mole
    )
):
    raise RuntimeError(
        "После минимизации получена NaN/inf "
        "потенциальная энергия."
    )

print("✅ Минимизация энергии завершена.")


# Начальные скорости.
simulation.context.setVelocitiesToTemperature(
    TEMPERATURE_K * unit.kelvin
)


# Пути вывода.
equil_log_path = (
    WORKDIR / f"{EQUIL_JOB}.log"
)

equil_dcd_path = (
    WORKDIR / f"{EQUIL_JOB}.dcd"
)

equil_chk_path = (
    WORKDIR / f"{EQUIL_JOB}.chk"
)

equil_xml_path = (
    WORKDIR / f"{EQUIL_JOB}.xml"
)


# Репортинг.
simulation.reporters.append(
    app.StateDataReporter(
        sys.stdout,
        log_steps,
        step=True,
        potentialEnergy=True,
        temperature=True,
        density=SOLVATE_SYSTEM,
        speed=True,
        progress=True,
        totalSteps=EQUIL_STEPS_TOTAL,
    )
)

simulation.reporters.append(
    app.StateDataReporter(
        str(equil_log_path),
        log_steps,
        step=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        volume=SOLVATE_SYSTEM,
        density=SOLVATE_SYSTEM,
        speed=True,
    )
)

simulation.reporters.append(
    app.DCDReporter(
        str(equil_dcd_path),
        dcd_steps,
    )
)


# Equilibration.
print(
    f"\nЗапуск equilibration: "
    f"{EQUIL_STEPS_TOTAL:,} шагов..."
)

simulation.step(
    EQUIL_STEPS_TOTAL
)


# Сохранение состояния и checkpoint.
simulation.saveCheckpoint(
    str(equil_chk_path)
)

simulation.saveState(
    str(equil_xml_path)
)


print("✅ Уравновешивание системы завершено.")
print(f"Траектория : {equil_dcd_path.name}")
print(f"Лог        : {equil_log_path.name}")
print(
    f"Состояния  : "
    f"{equil_chk_path.name}, "
    f"{equil_xml_path.name}"
)

--- Проверка, минимизация и equilibration: YTHDF2_irinotecan ---

--- Загрузка AMBER-топологии ---
Атомов в PRMTOP: 315,571
Атомов в CRD:    315,571
✅ PRMTOP ↔ CRD согласованы.
✅ Координаты конечные: NaN/inf не обнаружены.
Химических связей в топологии: 213,404
Атомов LIG: 81
✅ Лиганд содержит 81 атом.
Атомов в OpenMM System: 315,571
✅ PRMTOP ↔ OpenMM System согласованы.
Используется платформа OpenMM: CUDA

--- Проверка исходной энергии ---
Исходная потенциальная энергия: -3896471.5284578726 kJ/mol
✅ Исходная энергия конечна.

--- Минимизация энергии ---
Энергия после минимизации: -5035699.997287884 kJ/mol
✅ Минимизация энергии завершена.

Запуск equilibration: 1,000,000 шагов...
#"Progress (%)","Step","Potential Energy (kJ/mole)","Temperature (K)","Density (g/mL)","Speed (ns/day)"
0.5%,5000,-4064375.6447898224,310.4091592797615,0.955171636911213,0
1.0%,10000,-4081768.6850852817,310.5499200569298,0.9794928830645453,18.4
1.5%,15000,-4090322.883891668,311.06109251976784,0.984052441502382

In [34]:
# @title 🧹 13. Очистка файлов Production MD перед запуском ✅

from pathlib import Path


print("--- Очистка файлов Production MD ---")
print(f"Рабочая папка: {WORKDIR.resolve()}")


prod_files = sorted(WORKDIR.glob(f"{PROD_JOB}*"))


if not prod_files:
    print("✅ Файлы продукционного прогона не найдены.")
else:
    removed_count = 0

    for file_path in prod_files:
        if not file_path.is_file():
            continue

        try:
            file_path.unlink()
            removed_count += 1
            print(f"🗑 Удалён файл: {file_path.name}")

        except OSError as exc:
            print(
                f"⚠️ Не удалось удалить "
                f"{file_path.name}: {exc}"
            )

    print(
        f"✅ Очистка завершена. Удалено файлов: "
        f"{removed_count}"
    )


print(
    "Можно запускать продукционный прогон "
    "(Production MD)."
)

--- Очистка файлов Production MD ---
Рабочая папка: /home/sasha/my_pipeline/runs/YTHDF2_irinotecan
✅ Файлы продукционного прогона не найдены.
Можно запускать продукционный прогон (Production MD).


In [ ]:
# @title 🚀 14. Продукционная MD-симуляция OpenMM ✅

import sys
from pathlib import Path

import openmm as mm
from openmm import app, unit


print(
    f"--- Запуск Production MD: "
    f"{SYSTEM_BASENAME} ---"
)


def time_interval_to_steps(time_ps: float, timestep_fs: float) -> int:
    """Переводит интервал времени в количество MD-шагов."""
    steps = round(time_ps * 1000.0 / timestep_fs)

    if steps < 1:
        raise ValueError(
            f"Интервал {time_ps} ps слишком мал "
            f"для timestep={timestep_fs} fs."
        )

    return steps


def load_saved_state(
    simulation: app.Simulation,
    checkpoint_path: Path,
    state_path: Path,
) -> bool:
    """Восстанавливает Simulation из checkpoint или XML-state."""
    if checkpoint_path.is_file():
        try:
            simulation.loadCheckpoint(str(checkpoint_path))
            print(
                f"✅ Загружен checkpoint: "
                f"{checkpoint_path.name}"
            )
            return True

        except Exception as exc:
            print(
                f"⚠️ Не удалось загрузить "
                f"{checkpoint_path.name}: {exc}"
            )

    if state_path.is_file():
        try:
            simulation.loadState(str(state_path))
            print(
                f"✅ Загружено состояние: "
                f"{state_path.name}"
            )
            return True

        except Exception as exc:
            print(
                f"⚠️ Не удалось загрузить "
                f"{state_path.name}: {exc}"
            )

    return False


# Пути Production MD
job = PROD_JOB

log_file = WORKDIR / f"{job}.log"
chk_file = WORKDIR / f"{job}.chk"
xml_file = WORKDIR / f"{job}.xml"
last_pdb = WORKDIR / f"{job}_last.pdb"

equil_chk = WORKDIR / f"{EQUIL_JOB}.chk"
equil_xml = WORKDIR / f"{EQUIL_JOB}.xml"


# Интервалы записи
steps_dcd = time_interval_to_steps(PROD_DCD_PS, DT_FS)
steps_log = time_interval_to_steps(PROD_LOG_PS, DT_FS)
steps_chk = time_interval_to_steps(PROD_CHK_PS, DT_FS)

print("Параметры запуска:")
print(f"  Задача       : {job}")
print(
    f"  DCD          : каждые {PROD_DCD_PS} ps "
    f"({steps_dcd:,} шагов)"
)
print(
    f"  Лог          : каждые {PROD_LOG_PS} ps "
    f"({steps_log:,} шагов)"
)
print(
    f"  Checkpoint   : каждые {PROD_CHK_PS} ps "
    f"({steps_chk:,} шагов)"
)
print(f"  Всего шагов  : {PROD_STEPS_TO_RUN:,}")


# Загрузка AMBER-топологии и координат
if not SYSTEM_PRMTOP.is_file():
    raise FileNotFoundError(
        f"Не найдена топология: {SYSTEM_PRMTOP}"
    )

if not SYSTEM_CRD.is_file():
    raise FileNotFoundError(
        f"Не найдены координаты: {SYSTEM_CRD}"
    )

prmtop = app.AmberPrmtopFile(str(SYSTEM_PRMTOP))
inpcrd = app.AmberInpcrdFile(str(SYSTEM_CRD))


# Создание физической системы
nonbonded_method = (
    app.PME
    if SOLVATE_SYSTEM
    else app.NoCutoff
)

system = prmtop.createSystem(
    nonbondedMethod=nonbonded_method,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=app.HBonds,
    rigidWater=True,
    removeCMMotion=True,
)


if SOLVATE_SYSTEM:
    system.addForce(
        mm.MonteCarloBarostat(
            PRESSURE_BAR * unit.bar,
            TEMPERATURE_K * unit.kelvin,
            25,
        )
    )


# Интегратор
integrator = mm.LangevinMiddleIntegrator(
    TEMPERATURE_K * unit.kelvin,
    FRICTION_PS / unit.picosecond,
    DT_FS * unit.femtoseconds,
)


# Используем тот же выбор платформы, что и на этапе equilibration.
platform, platform_properties = select_openmm_platform()

simulation = app.Simulation(
    prmtop.topology,
    system,
    integrator,
    platform,
    platform_properties,
)


# Восстановление состояния
resumed = False

if chk_file.is_file() or xml_file.is_file():
    print(
        "Найдено сохранённое состояние Production MD. "
        "Попытка продолжения..."
    )

    resumed = load_saved_state(
        simulation,
        chk_file,
        xml_file,
    )


if resumed:
    # DCD нельзя безопасно дописывать после восстановления
    # Simulation из checkpoint/state, поэтому создаём новый chunk.
    part = 2

    while (
        WORKDIR / f"{job}_part{part}.dcd"
    ).exists():
        part += 1

    dcd_file = WORKDIR / f"{job}_part{part}.dcd"

    print(
        f"⚠️ Продолжение с шага {simulation.currentStep:,}."
    )
    print(
        f"Новый файл траектории: {dcd_file.name}"
    )

else:
    print(
        "Состояние Production MD не найдено. "
        "Загружаем Equilibration..."
    )

    resumed = load_saved_state(
        simulation,
        equil_chk,
        equil_xml,
    )

    if not resumed:
        print(
            "⚠️ Checkpoint Equilibration не найден. "
            "Запуск с исходных координат."
        )

        simulation.context.setPositions(
            inpcrd.positions
        )

        if SOLVATE_SYSTEM and inpcrd.boxVectors is not None:
            simulation.context.setPeriodicBoxVectors(
                *inpcrd.boxVectors
            )

        simulation.minimizeEnergy(
            maxIterations=1000
        )

        simulation.context.setVelocitiesToTemperature(
            TEMPERATURE_K * unit.kelvin
        )

    dcd_file = WORKDIR / f"{job}.dcd"


# Целевой номер шага с учётом уже выполненной части.
target_step = (
    simulation.currentStep
    + PROD_STEPS_TO_RUN
)

log_mode = (
    "a"
    if resumed and log_file.is_file()
    else "w"
)


# Репортеры
simulation.reporters.clear()

simulation.reporters.append(
    app.DCDReporter(
        str(dcd_file),
        steps_dcd,
        append=False,
    )
)

simulation.reporters.append(
    app.StateDataReporter(
        sys.stdout,
        steps_log,
        step=True,
        time=True,
        potentialEnergy=True,
        temperature=True,
        progress=True,
        remainingTime=True,
        speed=True,
        totalSteps=target_step,
    )
)


log_handle = open(
    log_file,
    log_mode,
    encoding="utf-8",
)

simulation.reporters.append(
    app.StateDataReporter(
        log_handle,
        steps_log,
        step=True,
        time=True,
        potentialEnergy=True,
        temperature=True,
        density=SOLVATE_SYSTEM,
        separator="\t",
    )
)

simulation.reporters.append(
    app.CheckpointReporter(
        str(chk_file),
        steps_chk,
    )
)


print(
    f"Стартовый шаг : {simulation.currentStep:,}"
)
print(
    f"Целевой шаг   : {target_step:,}"
)
print(
    f"Осталось      : {PROD_STEPS_TO_RUN:,} шагов"
)


# Production MD
try:
    simulation.step(PROD_STEPS_TO_RUN)

    print(
        "✅ Продукционный прогон "
        "полностью завершён!"
    )

except Exception as exc:
    print(
        f"❌ Ошибка во время Production MD: {exc}"
    )
    raise

finally:
    log_handle.close()

    final_state = simulation.context.getState(
        getPositions=True
    )

    with last_pdb.open(
        "w",
        encoding="utf-8",
    ) as pdb_handle:
        app.PDBFile.writeFile(
            simulation.topology,
            final_state.getPositions(),
            pdb_handle,
        )

    simulation.saveState(
        str(xml_file)
    )

    print(
        f"Финальная структура : {last_pdb.name}"
    )
    print(
        f"Состояние           : {xml_file.name}"
    )
    print(
        f"Траектория          : {dcd_file.name}"
    )
    print(
        f"Лог                 : {log_file.name}"
    )
    print(
        f"Checkpoint          : {chk_file.name}"
    )

In [ ]:
# @title 📊 15. Анализ траектории: RMSD, RMSF и COM ✅

import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np


EXCLUDE_TAIL = False
ANALYSIS_STRIDE = 5
ANALYSIS_CHUNK_SIZE = 10
FIGURES_DIR = WORKDIR / "analysis_figures"

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет наличие данных и возможность чтения DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(str(path)) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


def select_protein_ca(
    topology: md.Topology,
) -> np.ndarray:
    """Возвращает индексы Cα выбранной области белка."""
    if EXCLUDE_TAIL:
        atom_indices = topology.select(
            "protein and name CA and residue 0 to 295",
        )
        print(
            "✂️ Анализ Cα выполняется для остатков 0–295.",
        )
    else:
        atom_indices = topology.select(
            "protein and name CA",
        )
        print(
            "Анализ Cα выполняется для всего белка.",
        )

    if len(atom_indices) == 0:
        raise ValueError(
            "В траектории не найдены Cα-атомы белка.",
        )

    return atom_indices


def configure_axis(
    axis,
    title: str,
    xlabel: str,
    ylabel: str,
) -> None:
    """Единое оформление графиков."""
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)
    axis.grid(
        linestyle="-",
        linewidth=0.5,
        alpha=0.7,
    )
    axis.set_xlim(left=0)


def calculate_center_of_mass(
    coordinates: np.ndarray,
    atom_indices: np.ndarray,
    masses: np.ndarray,
) -> np.ndarray:
    """Рассчитывает центр масс для каждого кадра."""
    selected_coordinates = coordinates[:, atom_indices, :]
    weighted_coordinates = (
        selected_coordinates
        * masses[np.newaxis, :, np.newaxis]
    )

    return (
        weighted_coordinates.sum(axis=1)
        / masses.sum()
    )


print(
    f"--- Потоковый анализ траектории MDTraj: "
    f"{SYSTEM_BASENAME} ---"
)


# ---------------------------------------------------------------------
# 1. Поиск DCD.
# ---------------------------------------------------------------------

dcd_pattern = f"{PROD_JOB}*.dcd"

dcd_files = sorted(
    WORKDIR.glob(dcd_pattern),
    key=dcd_sort_key,
)

if not dcd_files:
    raise FileNotFoundError(
        f"Траектории {dcd_pattern} не найдены "
        f"в {WORKDIR}.",
    )


# ---------------------------------------------------------------------
# 2. Проверка DCD.
# ---------------------------------------------------------------------

valid_dcd_files = [
    path
    for path in dcd_files
    if validate_dcd(path)
]

if not valid_dcd_files:
    raise ValueError(
        "Не найдено корректных DCD-файлов "
        "для анализа.",
    )

print(
    f"Найдено валидных DCD: "
    f"{len(valid_dcd_files)}",
)

for dcd_path in valid_dcd_files:
    size_mb = dcd_path.stat().st_size / 1024**2

    print(
        f"  • {dcd_path.name} "
        f"({size_mb:.1f} MB)",
    )


# ---------------------------------------------------------------------
# 3. Проверка topology.
# ---------------------------------------------------------------------

if not SYSTEM_PRMTOP.is_file():
    raise FileNotFoundError(
        f"Не найдена топология: {SYSTEM_PRMTOP}",
    )


# Загружаем только topology, а не всю траекторию.
reference = md.load_frame(
    str(valid_dcd_files[0]),
    0,
    top=str(SYSTEM_PRMTOP),
)

topology = reference.topology

print(
    f"Topology содержит "
    f"{reference.n_atoms:,} атомов.",
)


# ---------------------------------------------------------------------
# 4. Выбор атомов.
# ---------------------------------------------------------------------

protein_ca = select_protein_ca(topology)

protein_heavy = topology.select(
    "protein and not element H",
)

ligand_heavy = topology.select(
    f"resname {SMALL_MOL_RESNAME} and not element H",
)

ligand_all = topology.select(
    f"resname {SMALL_MOL_RESNAME}",
)

if len(ligand_all) == 0:
    raise ValueError(
        f"Лиганд с resname={SMALL_MOL_RESNAME} "
        "не найден в траектории.",
    )

if len(ligand_heavy) == 0:
    print(
        "⚠️ Тяжёлые атомы лиганда не найдены. "
        "Для RMSD используются все атомы лиганда.",
    )
    ligand_heavy = ligand_all


print(
    f"Cα белка: {len(protein_ca):,}",
)
print(
    f"Тяжёлых атомов белка: "
    f"{len(protein_heavy):,}",
)
print(
    f"Атомов лиганда: {len(ligand_all):,}",
)


# ---------------------------------------------------------------------
# 5. Подготовка масс для COM.
# ---------------------------------------------------------------------

protein_heavy_masses = np.array(
    [
        atom.element.mass
        for atom in topology.atoms
        if atom.index in set(protein_heavy)
    ],
    dtype=np.float64,
)

ligand_masses = np.array(
    [
        atom.element.mass
        for atom in topology.atoms
        if atom.index in set(ligand_all)
    ],
    dtype=np.float64,
)


# ---------------------------------------------------------------------
# 6. Потоковый анализ.
# ---------------------------------------------------------------------

print(
    "\n🚀 Запуск потокового анализа...",
)

print(
    f"Размер chunk: {ANALYSIS_CHUNK_SIZE} кадров.",
)

print(
    "ℹ️ Вся Production-траектория "
    "целиком в RAM НЕ загружается.",
)


time_chunks = []
rmsd_protein_chunks = []
rmsd_ligand_chunks = []
com_distance_chunks = []

rmsf_sum = np.zeros(
    (len(protein_ca), 3),
    dtype=np.float64,
)

rmsf_sum_sq = np.zeros(
    (len(protein_ca), 3),
    dtype=np.float64,
)

rmsf_frame_count = 0
global_frame_index = 0
processed_frames = 0


for dcd_path in valid_dcd_files:
    print(
        f"\n📂 Обработка: {dcd_path.name}",
    )

    for chunk in md.iterload(
        str(dcd_path),
        top=str(SYSTEM_PRMTOP),
        stride=ANALYSIS_STRIDE,
        chunk=ANALYSIS_CHUNK_SIZE,
    ):
        if chunk.n_frames == 0:
            continue

        # -------------------------------------------------------------
        # Выравнивание по Cα белка.
        # -------------------------------------------------------------

        chunk.superpose(
            reference,
            frame=0,
            atom_indices=protein_ca,
        )

        coordinates = chunk.xyz

        # -------------------------------------------------------------
        # RMSD белка.
        # -------------------------------------------------------------

        protein_coordinates = coordinates[
            :,
            protein_ca,
            :,
        ]

        reference_protein_coordinates = reference.xyz[
            0,
            protein_ca,
            :,
        ]

        protein_rmsd = np.sqrt(
            np.mean(
                np.sum(
                    (
                        protein_coordinates
                        - reference_protein_coordinates
                    ) ** 2,
                    axis=2,
                ),
                axis=1,
            ),
        )

        protein_rmsd *= 10.0


        # -------------------------------------------------------------
        # RMSD лиганда.
        # -------------------------------------------------------------

        ligand_coordinates = coordinates[
            :,
            ligand_heavy,
            :,
        ]

        reference_ligand_coordinates = reference.xyz[
            0,
            ligand_heavy,
            :,
        ]

        ligand_rmsd = np.sqrt(
            np.mean(
                np.sum(
                    (
                        ligand_coordinates
                        - reference_ligand_coordinates
                    ) ** 2,
                    axis=2,
                ),
                axis=1,
            ),
        )

        ligand_rmsd *= 10.0


        # -------------------------------------------------------------
        # RMSF Cα.
        # -------------------------------------------------------------

        ca_coordinates = coordinates[
            :,
            protein_ca,
            :,
        ].astype(
            np.float64,
            copy=False,
        )

        rmsf_sum += ca_coordinates.sum(
            axis=0,
        )

        rmsf_sum_sq += np.sum(
            ca_coordinates**2,
            axis=0,
        )

        rmsf_frame_count += chunk.n_frames


        # -------------------------------------------------------------
        # Центры масс.
        # -------------------------------------------------------------

        protein_com = calculate_center_of_mass(
            coordinates,
            protein_heavy,
            protein_heavy_masses,
        )

        ligand_com = calculate_center_of_mass(
            coordinates,
            ligand_all,
            ligand_masses,
        )

        com_distance = np.linalg.norm(
            protein_com - ligand_com,
            axis=1,
        )


        # -------------------------------------------------------------
        # Временная шкала.
        # -------------------------------------------------------------

        chunk_frame_indices = (
            global_frame_index
            + np.arange(chunk.n_frames)
        )

        chunk_time_ns = (
            chunk_frame_indices
            * PROD_DCD_PS
            * ANALYSIS_STRIDE
            / 1000.0
        )

        global_frame_index += chunk.n_frames
        processed_frames += chunk.n_frames


        # -------------------------------------------------------------
        # Сохраняем только маленькие результаты.
        # -------------------------------------------------------------

        time_chunks.append(
            chunk_time_ns,
        )

        rmsd_protein_chunks.append(
            protein_rmsd,
        )

        rmsd_ligand_chunks.append(
            ligand_rmsd,
        )

        com_distance_chunks.append(
            com_distance,
        )


        # Освобождаем chunk перед чтением следующего.
        del chunk
        del coordinates
        del protein_coordinates
        del ligand_coordinates
        del ca_coordinates
        del protein_com
        del ligand_com

        gc.collect()


    print(
        f"  ✅ Обработано кадров: "
        f"{processed_frames:,}",
    )


# ---------------------------------------------------------------------
# 7. Объединение маленьких массивов результатов.
# ---------------------------------------------------------------------

if processed_frames == 0:
    raise RuntimeError(
        "Не удалось обработать ни одного кадра.",
    )

time_ns = np.concatenate(
    time_chunks,
)

rmsd_protein = np.concatenate(
    rmsd_protein_chunks,
)

rmsd_ligand = np.concatenate(
    rmsd_ligand_chunks,
)

com_distance = np.concatenate(
    com_distance_chunks,
)


# ---------------------------------------------------------------------
# 8. Финальный RMSF.
# ---------------------------------------------------------------------

mean_coordinates = (
    rmsf_sum
    / rmsf_frame_count
)

mean_squared_coordinates = (
    rmsf_sum_sq
    / rmsf_frame_count
)

coordinate_variance = (
    mean_squared_coordinates
    - mean_coordinates**2
)

coordinate_variance = np.maximum(
    coordinate_variance,
    0.0,
)

rmsf_ca = np.sqrt(
    coordinate_variance.sum(axis=1),
)

rmsf_ca *= 10.0

ca_indices = np.arange(
    1,
    len(protein_ca) + 1,
)


print(
    f"\n✅ Обработано кадров: "
    f"{processed_frames:,}",
)

print(
    f"В RAM не хранилась полная Production-траектория.",
)


# ---------------------------------------------------------------------
# 9. Сводный график.
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10),
)


configure_axis(
    axes[0, 0],
    "Protein Cα RMSD",
    "Time (ns)",
    "RMSD (Å)",
)

axes[0, 0].plot(
    time_ns,
    rmsd_protein,
    linewidth=1.2,
)


configure_axis(
    axes[0, 1],
    "Ligand Heavy-Atom RMSD",
    "Time (ns)",
    "RMSD (Å)",
)

axes[0, 1].plot(
    time_ns,
    rmsd_ligand,
    linewidth=1.2,
)


configure_axis(
    axes[1, 0],
    "Protein Cα RMSF",
    "Cα Index",
    "RMSF (Å)",
)

axes[1, 0].plot(
    ca_indices,
    rmsf_ca,
    linewidth=1.2,
)


configure_axis(
    axes[1, 1],
    "Ligand–Protein COM Distance",
    "Time (ns)",
    "Distance (nm)",
)

axes[1, 1].plot(
    time_ns,
    com_distance,
    linewidth=1.2,
)


fig.suptitle(
    f"{SYSTEM_BASENAME} — MD Analysis",
    fontsize=14,
)

fig.tight_layout()


# ---------------------------------------------------------------------
# 10. Сохранение графиков.
# ---------------------------------------------------------------------

png_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_unified_summary.png"
)

svg_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_unified_summary.svg"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
)


print("\n📊 Анализ завершён.")
print(f"PNG: {png_path.resolve()}")
print(f"SVG: {svg_path.resolve()}")

plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# 11. Финальная очистка.
# ---------------------------------------------------------------------

del time_chunks
del rmsd_protein_chunks
del rmsd_ligand_chunks
del com_distance_chunks
del reference

gc.collect()

print("🧹 Временные данные анализа освобождены.")

--- Анализ траектории MDTraj: YTHDF2_irinotecan ---
Найдено валидных DCD: 1
  • ythdf2_irinotecan_prod.dcd (7223.0 MB)

🚀 Загрузка Production-траектории...
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)


ValueError: The topology and the trajectory files might not contain the same atoms
The input topology must contain all atoms even if you want to select a subset of them with atom_indices

In [ ]:
# @title 📈 16. Детальный анализ Cα RMSD белка и распределения ✅

import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
from scipy.stats import gaussian_kde


EXCLUDE_TAIL = False
ANALYSIS_STRIDE = 5
ANALYSIS_CHUNK_SIZE = 10
MAX_KDE_POINTS = 20_000


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет наличие данных и возможность чтения DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(str(path)) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


def select_protein_ca(
    topology: md.Topology,
) -> np.ndarray:
    """Возвращает индексы Cα выбранной области белка."""
    if EXCLUDE_TAIL:
        atom_indices = topology.select(
            "protein and name CA and residue 0 to 295",
        )
        print(
            "✂️ Анализ Cα выполняется для остатков 0–295.",
        )
    else:
        atom_indices = topology.select(
            "protein and name CA",
        )
        print(
            "Анализ Cα выполняется для всего белка.",
        )

    if len(atom_indices) == 0:
        raise ValueError(
            "Cα-атомы белка не найдены в топологии.",
        )

    return atom_indices


def configure_axis(
    axis,
    title: str,
    xlabel: str,
    ylabel: str,
) -> None:
    """Единое оформление графиков."""
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)
    axis.grid(
        linestyle="-",
        linewidth=0.5,
        alpha=0.7,
    )
    axis.set_xlim(left=0)


print(
    f"--- Детальный потоковый анализ Cα RMSD: "
    f"{SYSTEM_BASENAME} ---"
)


# ---------------------------------------------------------------------
# 1. Поиск Production DCD.
# ---------------------------------------------------------------------

dcd_pattern = f"{PROD_JOB}*.dcd"

dcd_files = sorted(
    WORKDIR.glob(dcd_pattern),
    key=dcd_sort_key,
)

if not dcd_files:
    raise FileNotFoundError(
        f"Траектории {dcd_pattern} не найдены "
        f"в {WORKDIR}.",
    )


valid_dcd_files = [
    path
    for path in dcd_files
    if validate_dcd(path)
]

if not valid_dcd_files:
    raise ValueError(
        "Не найдено корректных DCD-файлов "
        "для анализа.",
    )


print(
    f"Найдено валидных DCD: "
    f"{len(valid_dcd_files)}",
)

for dcd_path in valid_dcd_files:
    size_mb = dcd_path.stat().st_size / 1024**2

    print(
        f"  • {dcd_path.name} "
        f"({size_mb:.1f} MB)",
    )


# ---------------------------------------------------------------------
# 2. Загрузка только первого кадра как reference.
# ---------------------------------------------------------------------

if not SYSTEM_PRMTOP.is_file():
    raise FileNotFoundError(
        f"Не найдена топология: {SYSTEM_PRMTOP}",
    )


reference = md.load_frame(
    str(valid_dcd_files[0]),
    0,
    top=str(SYSTEM_PRMTOP),
)

topology = reference.topology

protein_ca = select_protein_ca(topology)

print(
    f"Cα белка: {len(protein_ca):,}",
)

print(
    f"ℹ️ Полная траектория в RAM не загружается."
)

print(
    f"ℹ️ Анализ выполняется chunks по "
    f"{ANALYSIS_CHUNK_SIZE} кадров."
)


# ---------------------------------------------------------------------
# 3. Потоковый расчёт RMSD.
# ---------------------------------------------------------------------

time_chunks = []
rmsd_chunks = []

global_frame_index = 0
processed_frames = 0


for dcd_path in valid_dcd_files:
    print(
        f"\n📂 Обработка: {dcd_path.name}",
    )

    for chunk in md.iterload(
        str(dcd_path),
        top=str(SYSTEM_PRMTOP),
        stride=ANALYSIS_STRIDE,
        chunk=ANALYSIS_CHUNK_SIZE,
    ):
        if chunk.n_frames == 0:
            continue

        # Выравниваем каждый небольшой chunk
        # относительно Cα reference.
        chunk.superpose(
            reference,
            frame=0,
            atom_indices=protein_ca,
        )

        coordinates = chunk.xyz[
            :,
            protein_ca,
            :,
        ]

        reference_coordinates = reference.xyz[
            0,
            protein_ca,
            :,
        ]

        rmsd = np.sqrt(
            np.mean(
                np.sum(
                    (
                        coordinates
                        - reference_coordinates
                    ) ** 2,
                    axis=2,
                ),
                axis=1,
            ),
        )

        # nm → Å.
        rmsd *= 10.0

        frame_indices = (
            global_frame_index
            + np.arange(chunk.n_frames)
        )

        time_ns = (
            frame_indices
            * PROD_DCD_PS
            * ANALYSIS_STRIDE
            / 1000.0
        )

        time_chunks.append(time_ns)
        rmsd_chunks.append(rmsd)

        global_frame_index += chunk.n_frames
        processed_frames += chunk.n_frames

        del chunk
        del coordinates
        del rmsd
        gc.collect()

    print(
        f"  ✅ Обработано кадров: "
        f"{processed_frames:,}",
    )


if processed_frames == 0:
    raise RuntimeError(
        "Не удалось обработать ни одного кадра.",
    )


# ---------------------------------------------------------------------
# 4. Объединяем только маленькие результаты RMSD.
# ---------------------------------------------------------------------

time_ns = np.concatenate(
    time_chunks,
)

rmsd_protein = np.concatenate(
    rmsd_chunks,
)


print(
    f"\n✅ Всего обработано кадров: "
    f"{len(rmsd_protein):,}",
)


# ---------------------------------------------------------------------
# 5. Статистика RMSD.
# ---------------------------------------------------------------------

if not np.all(np.isfinite(rmsd_protein)):
    raise ValueError(
        "В RMSD обнаружены NaN или Inf.",
    )


mean_rmsd = np.mean(rmsd_protein)
median_rmsd = np.median(rmsd_protein)
min_rmsd = np.min(rmsd_protein)
max_rmsd = np.max(rmsd_protein)
std_rmsd = np.std(rmsd_protein)

print(
    "\nCα RMSD:"
    f"\n  Среднее  : {mean_rmsd:.2f} Å"
    f"\n  Медиана  : {median_rmsd:.2f} Å"
    f"\n  SD       : {std_rmsd:.2f} Å"
    f"\n  Минимум  : {min_rmsd:.2f} Å"
    f"\n  Максимум : {max_rmsd:.2f} Å",
)


# ---------------------------------------------------------------------
# 6. Подготовка данных для KDE.
# ---------------------------------------------------------------------

kde_data = rmsd_protein

if len(kde_data) > MAX_KDE_POINTS:
    kde_indices = np.linspace(
        0,
        len(kde_data) - 1,
        MAX_KDE_POINTS,
        dtype=np.int64,
    )

    kde_data = kde_data[kde_indices]

    print(
        f"\nℹ️ Для KDE используется "
        f"{len(kde_data):,} точек "
        f"из {len(rmsd_protein):,}.",
    )
else:
    print(
        f"\nℹ️ Для KDE используется "
        f"{len(kde_data):,} точек.",
    )


# ---------------------------------------------------------------------
# 7. Построение графиков.
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.5),
)


# RMSD во времени.
axes[0].plot(
    time_ns,
    rmsd_protein,
    linewidth=1.2,
)

axes[0].set_title(
    "Protein Cα RMSD vs Time",
)

axes[0].set_xlabel(
    "Time (ns)",
)

axes[0].set_ylabel(
    "Cα RMSD (Å)",
)

axes[0].set_xlim(
    left=0,
)

axes[0].grid(
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)


# Распределение RMSD.
axes[1].hist(
    rmsd_protein,
    bins=30,
    density=True,
    alpha=0.5,
    edgecolor="black",
)


# KDE только при наличии разброса.
if (
    len(kde_data) > 1
    and np.std(kde_data) > 0
):
    kde_x = np.linspace(
        np.min(kde_data),
        np.max(kde_data),
        400,
    )

    kde = gaussian_kde(kde_data)

    axes[1].plot(
        kde_x,
        kde(kde_x),
        linewidth=2.0,
        label="KDE",
    )

    axes[1].legend()


axes[1].set_title(
    "Cα RMSD Distribution",
)

axes[1].set_xlabel(
    "Cα RMSD (Å)",
)

axes[1].set_ylabel(
    "Probability Density",
)

axes[1].grid(
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)


fig.suptitle(
    f"{SYSTEM_BASENAME} — Protein Cα RMSD",
    fontsize=14,
)

fig.tight_layout()


# ---------------------------------------------------------------------
# 8. Сохранение результатов.
# ---------------------------------------------------------------------

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

png_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_protein_rmsd_dist.png"
)

svg_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_protein_rmsd_dist.svg"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
)

print("\n📊 Графики сохранены:")
print(f"PNG: {png_path.resolve()}")
print(f"SVG: {svg_path.resolve()}")

plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# 9. Очистка.
# ---------------------------------------------------------------------

del time_chunks
del rmsd_chunks
del rmsd_protein
del time_ns
del reference

gc.collect()

print(
    "🧹 Временные данные анализа освобождены.",
)

In [ ]:
# @title 🧪 17. Детальный анализ RMSD лиганда и распределения ✅

import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
from scipy.stats import gaussian_kde


EXCLUDE_TAIL = False
ANALYSIS_STRIDE = 5
ANALYSIS_CHUNK_SIZE = 10
MAX_KDE_POINTS = 20_000


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет наличие данных и возможность чтения DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(str(path)) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


def select_protein_ca(
    topology: md.Topology,
) -> np.ndarray:
    """Возвращает индексы Cα выбранной области белка."""
    if EXCLUDE_TAIL:
        atom_indices = topology.select(
            "protein and name CA and residue 0 to 295",
        )
        print(
            "✂️ Анализ Cα выполняется для остатков 0–295.",
        )
    else:
        atom_indices = topology.select(
            "protein and name CA",
        )
        print(
            "Анализ Cα выполняется для всего белка.",
        )

    if len(atom_indices) == 0:
        raise ValueError(
            "Cα-атомы белка не найдены.",
        )

    return atom_indices


def select_ligand_heavy_atoms(
    topology: md.Topology,
) -> np.ndarray:
    """Выбирает тяжёлые атомы лиганда по типу системы."""
    if LIGAND_KIND == "small_molecule":
        residue_names = {
            SMALL_MOL_RESNAME.strip().upper(),
        }

    elif LIGAND_KIND == "nucleic_acid":
        residue_names = {
            "A",
            "C",
            "G",
            "U",
            "DA",
            "DC",
            "DG",
            "DT",
            "RA",
            "RC",
            "RG",
            "RU",
            "DA5",
            "DG5",
            "DT3",
            "DC3",
            "A5",
            "C3",
            "G5",
        }

        modified_resname = str(
            MODIFIED_NA_RESNAME,
        ).strip().upper()

        if modified_resname:
            residue_names.add(modified_resname)

    else:
        raise ValueError(
            f"Неизвестный LIGAND_KIND: "
            f"{LIGAND_KIND}",
        )

    atom_indices = np.array(
        [
            atom.index
            for atom in topology.atoms
            if (
                atom.residue.name.strip().upper()
                in residue_names
                and atom.element is not None
                and atom.element.symbol != "H"
            )
        ],
        dtype=int,
    )

    if len(atom_indices) == 0:
        raise ValueError(
            "Тяжёлые атомы лиганда не найдены "
            "в топологии.",
        )

    return atom_indices


print(
    f"--- Детальный потоковый анализ RMSD лиганда "
    f"({LIGAND_KIND}): {SYSTEM_BASENAME} ---",
)


# ---------------------------------------------------------------------
# 1. Поиск Production DCD.
# ---------------------------------------------------------------------

dcd_pattern = f"{PROD_JOB}*.dcd"

dcd_files = sorted(
    WORKDIR.glob(dcd_pattern),
    key=dcd_sort_key,
)

if not dcd_files:
    raise FileNotFoundError(
        f"Траектории {dcd_pattern} не найдены "
        f"в {WORKDIR}.",
    )


valid_dcd_files = [
    path
    for path in dcd_files
    if validate_dcd(path)
]

if not valid_dcd_files:
    raise ValueError(
        "Не найдено корректных DCD-файлов "
        "для анализа.",
    )


print(
    f"Найдено валидных DCD: "
    f"{len(valid_dcd_files)}",
)

for dcd_path in valid_dcd_files:
    size_mb = dcd_path.stat().st_size / 1024**2

    print(
        f"  • {dcd_path.name} "
        f"({size_mb:.1f} MB)",
    )


# ---------------------------------------------------------------------
# 2. Загружаем только первый кадр.
# ---------------------------------------------------------------------

if not SYSTEM_PRMTOP.is_file():
    raise FileNotFoundError(
        f"Не найдена топология: {SYSTEM_PRMTOP}",
    )


reference = md.load_frame(
    str(valid_dcd_files[0]),
    0,
    top=str(SYSTEM_PRMTOP),
)

topology = reference.topology


# ---------------------------------------------------------------------
# 3. Выбор атомов.
# ---------------------------------------------------------------------

protein_ca = select_protein_ca(
    topology,
)

ligand_heavy = select_ligand_heavy_atoms(
    topology,
)

print(
    f"Cα белка для выравнивания: "
    f"{len(protein_ca):,}",
)

print(
    f"Тяжёлых атомов лиганда: "
    f"{len(ligand_heavy):,}",
)

print(
    f"ℹ️ Полная траектория в RAM не загружается.",
)

print(
    f"ℹ️ Размер chunk: "
    f"{ANALYSIS_CHUNK_SIZE} кадров.",
)


# ---------------------------------------------------------------------
# 4. Потоковый расчёт RMSD.
# ---------------------------------------------------------------------

time_chunks = []
rmsd_chunks = []

global_frame_index = 0
processed_frames = 0


for dcd_path in valid_dcd_files:
    print(
        f"\n📂 Обработка: {dcd_path.name}",
    )

    for chunk in md.iterload(
        str(dcd_path),
        top=str(SYSTEM_PRMTOP),
        stride=ANALYSIS_STRIDE,
        chunk=ANALYSIS_CHUNK_SIZE,
    ):
        if chunk.n_frames == 0:
            continue

        # Сначала устраняем глобальное перемещение
        # и вращение белка по Cα.
        chunk.superpose(
            reference,
            frame=0,
            atom_indices=protein_ca,
        )

        # -------------------------------------------------------------
        # RMSD лиганда относительно первого кадра.
        # -------------------------------------------------------------

        ligand_coordinates = chunk.xyz[
            :,
            ligand_heavy,
            :,
        ]

        reference_ligand_coordinates = (
            reference.xyz[
                0,
                ligand_heavy,
                :,
            ]
        )

        rmsd_ligand = np.sqrt(
            np.mean(
                np.sum(
                    (
                        ligand_coordinates
                        - reference_ligand_coordinates
                    ) ** 2,
                    axis=2,
                ),
                axis=1,
            ),
        )

        # nm → Å.
        rmsd_ligand *= 10.0

        # -------------------------------------------------------------
        # Временная шкала.
        # -------------------------------------------------------------

        frame_indices = (
            global_frame_index
            + np.arange(chunk.n_frames)
        )

        time_ns = (
            frame_indices
            * PROD_DCD_PS
            * ANALYSIS_STRIDE
            / 1000.0
        )

        time_chunks.append(
            time_ns,
        )

        rmsd_chunks.append(
            rmsd_ligand,
        )

        global_frame_index += chunk.n_frames
        processed_frames += chunk.n_frames

        # Освобождаем chunk.
        del chunk
        del ligand_coordinates
        del rmsd_ligand
        gc.collect()

    print(
        f"  ✅ Обработано кадров: "
        f"{processed_frames:,}",
    )


if processed_frames == 0:
    raise RuntimeError(
        "Не удалось обработать ни одного кадра.",
    )


# ---------------------------------------------------------------------
# 5. Объединение маленьких массивов результатов.
# ---------------------------------------------------------------------

time_ns = np.concatenate(
    time_chunks,
)

rmsd_ligand = np.concatenate(
    rmsd_chunks,
)


if not np.all(np.isfinite(rmsd_ligand)):
    raise ValueError(
        "В RMSD лиганда обнаружены NaN или Inf.",
    )


print(
    f"\n✅ Всего обработано кадров: "
    f"{len(rmsd_ligand):,}",
)


# ---------------------------------------------------------------------
# 6. Статистика.
# ---------------------------------------------------------------------

mean_rmsd = np.mean(rmsd_ligand)
median_rmsd = np.median(rmsd_ligand)
min_rmsd = np.min(rmsd_ligand)
max_rmsd = np.max(rmsd_ligand)
std_rmsd = np.std(rmsd_ligand)

print(
    "\nLigand RMSD:"
    f"\n  Среднее  : {mean_rmsd:.2f} Å"
    f"\n  Медиана  : {median_rmsd:.2f} Å"
    f"\n  SD       : {std_rmsd:.2f} Å"
    f"\n  Минимум  : {min_rmsd:.2f} Å"
    f"\n  Максимум : {max_rmsd:.2f} Å",
)


# ---------------------------------------------------------------------
# 7. Ограничиваем KDE.
# ---------------------------------------------------------------------

kde_data = rmsd_ligand

if len(kde_data) > MAX_KDE_POINTS:
    kde_indices = np.linspace(
        0,
        len(kde_data) - 1,
        MAX_KDE_POINTS,
        dtype=np.int64,
    )

    kde_data = kde_data[kde_indices]

    print(
        f"\nℹ️ Для KDE используется "
        f"{len(kde_data):,} точек "
        f"из {len(rmsd_ligand):,}.",
    )
else:
    print(
        f"\nℹ️ Для KDE используется "
        f"{len(kde_data):,} точек.",
    )


# ---------------------------------------------------------------------
# 8. Визуализация.
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.5),
)


# RMSD во времени.
axes[0].plot(
    time_ns,
    rmsd_ligand,
    linewidth=1.2,
)

axes[0].set_title(
    f"Ligand RMSD vs Time ({LIGAND_KIND})",
)

axes[0].set_xlabel(
    "Time (ns)",
)

axes[0].set_ylabel(
    "Ligand Heavy-Atom RMSD (Å)",
)

axes[0].set_xlim(
    left=0,
)

axes[0].grid(
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)


# Распределение RMSD.
axes[1].hist(
    rmsd_ligand,
    bins=30,
    density=True,
    alpha=0.5,
    edgecolor="black",
)


# KDE.
if (
    len(kde_data) > 1
    and np.std(kde_data) > 0
):
    kde_x = np.linspace(
        np.min(kde_data),
        np.max(kde_data),
        400,
    )

    kde = gaussian_kde(
        kde_data,
    )

    axes[1].plot(
        kde_x,
        kde(kde_x),
        linewidth=2.0,
        label="KDE",
    )

    axes[1].legend()


axes[1].set_title(
    "Ligand RMSD Distribution",
)

axes[1].set_xlabel(
    "Ligand Heavy-Atom RMSD (Å)",
)

axes[1].set_ylabel(
    "Probability Density",
)

axes[1].grid(
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)


fig.suptitle(
    f"{SYSTEM_BASENAME} — Ligand RMSD",
    fontsize=14,
)

fig.tight_layout()


# ---------------------------------------------------------------------
# 9. Сохранение PNG + SVG.
# ---------------------------------------------------------------------

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

png_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_ligand_rmsd_dist.png"
)

svg_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_ligand_rmsd_dist.svg"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
)

print(
    "\n📊 Графики RMSD лиганда сохранены:",
)

print(
    f"PNG: {png_path.resolve()}",
)

print(
    f"SVG: {svg_path.resolve()}",
)

plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# 10. Очистка.
# ---------------------------------------------------------------------

del time_chunks
del rmsd_chunks
del rmsd_ligand
del time_ns
del reference

gc.collect()

print(
    "🧹 Временные данные анализа освобождены.",
)

In [ ]:
# @title 🧬 18. Анализ гибкости белка по Cα RMSF ✅

import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np


ANALYSIS_STRIDE = 5
ANALYSIS_CHUNK_SIZE = 10


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет наличие данных и возможность чтения DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(str(path)) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


print(
    f"--- Потоковый анализ Cα RMSF: "
    f"{SYSTEM_BASENAME} ---"
)


# ------------------------------------------------------------------
# 1. Поиск DCD-файлов
# ------------------------------------------------------------------

dcd_candidates = [
    WORKDIR / f"{PROD_JOB}.dcd",
    *sorted(
        WORKDIR.glob(f"{PROD_JOB}_part*.dcd"),
        key=dcd_sort_key,
    ),
]

valid_dcd_files = [
    path
    for path in dcd_candidates
    if validate_dcd(path)
]

if not valid_dcd_files:
    raise FileNotFoundError(
        "Корректные Production DCD-файлы не найдены."
    )

print("DCD-файлы для анализа:")

for path in valid_dcd_files:
    print(
        f"  • {path.name} "
        f"({path.stat().st_size / 1024**2:.1f} MB)"
    )


# ------------------------------------------------------------------
# 2. Загрузка только первого кадра
# ------------------------------------------------------------------

reference = md.load_frame(
    str(valid_dcd_files[0]),
    index=0,
    top=str(SYSTEM_PRMTOP),
)

topology = reference.topology

protein_ca = topology.select(
    "protein and name CA",
)

if len(protein_ca) == 0:
    raise ValueError(
        "Cα-атомы белка не найдены в топологии."
    )

print(
    f"\nCα-атомов белка: {len(protein_ca)}"
)


# ------------------------------------------------------------------
# 3. Подготовка потокового накопления RMSF
# ------------------------------------------------------------------

# Сумма координат каждого Cα по всем кадрам.
coordinate_sum = np.zeros(
    (len(protein_ca), 3),
    dtype=np.float64,
)

# Сумма квадратов координат каждого Cα.
coordinate_squared_sum = np.zeros(
    (len(protein_ca), 3),
    dtype=np.float64,
)

frame_count = 0


# ------------------------------------------------------------------
# 4. Потоковый проход по траектории
# ------------------------------------------------------------------

print(
    "\nЗапуск потокового расчёта RMSF..."
)

for dcd_path in valid_dcd_files:
    print(
        f"Обработка: {dcd_path.name}"
    )

    for chunk in md.iterload(
        str(dcd_path),
        top=str(SYSTEM_PRMTOP),
        stride=ANALYSIS_STRIDE,
        chunk=ANALYSIS_CHUNK_SIZE,
    ):
        # Выравнивание каждого чанка по Cα первого кадра.
        chunk.superpose(
            reference,
            frame=0,
            atom_indices=protein_ca,
        )

        coordinates = (
            chunk.xyz[:, protein_ca, :]
            .astype(np.float64, copy=False)
        )

        coordinate_sum += coordinates.sum(
            axis=0,
        )

        coordinate_squared_sum += np.square(
            coordinates,
        ).sum(
            axis=0,
        )

        frame_count += coordinates.shape[0]

        del coordinates
        del chunk


if frame_count == 0:
    raise RuntimeError(
        "Не удалось обработать ни одного кадра траектории."
    )


print(
    f"Обработано кадров: {frame_count}"
)


# ------------------------------------------------------------------
# 5. Расчёт RMSF
# ------------------------------------------------------------------

mean_coordinates = (
    coordinate_sum / frame_count
)

mean_squared_coordinates = (
    coordinate_squared_sum / frame_count
)

# Var(x) = E[x²] - E[x]²
variance = (
    mean_squared_coordinates
    - np.square(mean_coordinates)
)

# Из-за погрешности float могут появиться очень маленькие
# отрицательные значения порядка 1e-15.
variance = np.maximum(
    variance,
    0.0,
)

rmsf_nm = np.sqrt(
    variance.sum(axis=1),
)

rmsf_ang = rmsf_nm * 10.0


if not np.all(np.isfinite(rmsf_ang)):
    raise ValueError(
        "В рассчитанном RMSF обнаружены NaN или Inf."
    )


# ------------------------------------------------------------------
# 6. Реальные номера остатков
# ------------------------------------------------------------------

residue_numbers = np.array(
    [
        topology.atom(
            atom_index,
        ).residue.resSeq
        for atom_index in protein_ca
    ],
    dtype=int,
)


# ------------------------------------------------------------------
# 7. Статистика
# ------------------------------------------------------------------

mean_rmsf = np.mean(rmsf_ang)
median_rmsf = np.median(rmsf_ang)
min_rmsf = np.min(rmsf_ang)
max_rmsf = np.max(rmsf_ang)
std_rmsf = np.std(rmsf_ang)

print(
    "\nCα RMSF:"
    f"\n  Среднее  : {mean_rmsf:.2f} Å"
    f"\n  Медиана  : {median_rmsf:.2f} Å"
    f"\n  Минимум  : {min_rmsf:.2f} Å"
    f"\n  Максимум : {max_rmsf:.2f} Å"
    f"\n  Std      : {std_rmsf:.2f} Å"
)


# ------------------------------------------------------------------
# 8. Топ-5 наиболее гибких остатков
# ------------------------------------------------------------------

top_flexible = np.argsort(
    rmsf_ang,
)[-5:][::-1]

print(
    "\nТоп-5 наиболее гибких остатков:"
)

for index in top_flexible:
    residue = topology.atom(
        protein_ca[index],
    ).residue

    print(
        f"  • {residue.name}{residue.resSeq}: "
        f"{rmsf_ang[index]:.2f} Å"
    )


# ------------------------------------------------------------------
# 9. График RMSF
# ------------------------------------------------------------------

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

fig, ax = plt.subplots(
    figsize=(10, 4.5),
)

ax.plot(
    residue_numbers,
    rmsf_ang,
    linewidth=1.2,
)

ax.set_title(
    "Protein Cα Root Mean Square Fluctuation (RMSF)",
)

ax.set_xlabel(
    "Residue Number",
)

ax.set_ylabel(
    "RMSF (Å)",
)

ax.set_xlim(
    residue_numbers.min(),
    residue_numbers.max(),
)

ax.set_ylim(
    bottom=0,
)

ax.grid(
    linestyle="-",
    linewidth=0.5,
    alpha=0.7,
)

fig.tight_layout()


# ------------------------------------------------------------------
# 10. Сохранение PNG + SVG
# ------------------------------------------------------------------

png_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_protein_rmsf.png"
)

svg_path = (
    FIGURES_DIR
    / f"{SYSTEM_BASENAME}_protein_rmsf.svg"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
)

print(
    "\n📊 Графики RMSF сохранены:"
)

print(
    f"PNG: {png_path.resolve()}"
)

print(
    f"SVG: {svg_path.resolve()}"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------------
# 11. Очистка памяти
# ------------------------------------------------------------------

del reference
del topology
del coordinate_sum
del coordinate_squared_sum
del mean_coordinates
del mean_squared_coordinates
del variance
del rmsf_nm
del rmsf_ang

gc.collect()

print(
    "\n✅ Потоковый анализ Cα RMSF завершён."
)

In [ ]:
# @title ⚙️ 19. Анализ координации иона металла с белком ✅

import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
from scipy.stats import gaussian_kde


ANALYSIS_STRIDE = 5
ANALYSIS_CHUNK_SIZE = 10
MAX_KDE_POINTS = 20_000


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет наличие данных и возможность чтения DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(str(path)) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


def atom_matches_metal(
    atom,
    target: str,
    metal_elements: set[str],
) -> bool:
    """Проверяет, соответствует ли атом заданному металлу."""
    target = str(target).strip().upper()

    identifiers = {
        atom.name.strip().upper(),
        atom.residue.name.strip().upper(),
    }

    if atom.element is not None:
        identifiers.add(
            atom.element.symbol.strip().upper(),
        )

    if target == "AUTO":
        return bool(
            identifiers & metal_elements,
        )

    return target in identifiers


def find_metal_atoms(
    topology: md.Topology,
    target: str,
    metal_elements: set[str],
) -> np.ndarray:
    """Возвращает индексы атомов, соответствующих металлу."""
    return np.array(
        [
            atom.index
            for atom in topology.atoms
            if atom_matches_metal(
                atom,
                target,
                metal_elements,
            )
        ],
        dtype=int,
    )


print(
    f"--- Потоковый анализ координации металла: "
    f"{SYSTEM_BASENAME} ---"
)


# ------------------------------------------------------------------
# 1. Поиск Production DCD
# ------------------------------------------------------------------

dcd_candidates = [
    WORKDIR / f"{PROD_JOB}.dcd",
    *sorted(
        WORKDIR.glob(f"{PROD_JOB}_part*.dcd"),
        key=dcd_sort_key,
    ),
]

valid_dcd_files = [
    path
    for path in dcd_candidates
    if validate_dcd(path)
]

if not valid_dcd_files:
    raise FileNotFoundError(
        "Корректные Production DCD-файлы не найдены."
    )

print("DCD-файлы для анализа:")

for path in valid_dcd_files:
    print(
        f"  • {path.name} "
        f"({path.stat().st_size / 1024**2:.1f} MB)"
    )


# ------------------------------------------------------------------
# 2. Загрузка только reference frame
# ------------------------------------------------------------------

reference = md.load_frame(
    str(valid_dcd_files[0]),
    index=0,
    top=str(SYSTEM_PRMTOP),
)

topology = reference.topology


# ------------------------------------------------------------------
# 3. Поиск металла
# ------------------------------------------------------------------

metal_target = str(
    globals().get(
        "METAL_TO_ANALYZE",
        "auto",
    )
).strip().upper()

metal_elements = {
    str(element).strip().upper()
    for element in globals().get(
        "METAL_ELEMENTS_LIST",
        [
            "MG",
            "ZN",
            "CA",
            "FE",
            "MN",
            "CU",
            "NA",
            "K",
        ],
    )
}

metal_indices = find_metal_atoms(
    topology=topology,
    target=metal_target,
    metal_elements=metal_elements,
)

if len(metal_indices) == 0:
    print(
        "⚠️ Ионы металлов не найдены. "
        "Анализ координации пропущен."
    )

    del reference
    del topology
    gc.collect()

else:
    if len(metal_indices) > 1:
        print(
            f"⚠️ Найдено металлов: {len(metal_indices)}. "
            "Для анализа используется первый."
        )

    metal_index = int(
        metal_indices[0],
    )

    metal_atom = topology.atom(
        metal_index,
    )

    print(
        f"Металл:"
        f"\n  Индекс  : {metal_index}"
        f"\n  Атом    : {metal_atom.name}"
        f"\n  Остаток : "
        f"{metal_atom.residue.name}"
        f"{metal_atom.residue.resSeq}"
    )


    # ------------------------------------------------------------------
    # 4. Поиск карбоксилатных O Asp/Glu
    # ------------------------------------------------------------------

    carboxylate_oxygen = topology.select(
        "protein and "
        "(name OD1 or name OD2 or name OE1 or name OE2)",
    )

    if len(carboxylate_oxygen) == 0:
        print(
            "⚠️ Кислородные атомы Asp/Glu "
            "не найдены. Анализ невозможен."
        )

        del reference
        del topology
        gc.collect()

    else:
        print(
            f"Карбоксилатных O-атомов: "
            f"{len(carboxylate_oxygen)}"
        )


        # ------------------------------------------------------------------
        # 5. Пары металл–O
        # ------------------------------------------------------------------

        pairs = np.column_stack(
            (
                np.full(
                    len(carboxylate_oxygen),
                    metal_index,
                    dtype=int,
                ),
                carboxylate_oxygen,
            ),
        )


        # ------------------------------------------------------------------
        # 6. Потоковый расчёт минимального расстояния
        # ------------------------------------------------------------------

        all_distances = []
        all_times = []

        global_frame_index = 0

        print(
            "\nЗапуск потокового расчёта "
            "расстояния металл–O..."
        )

        for dcd_path in valid_dcd_files:
            print(
                f"Обработка: {dcd_path.name}"
            )

            for chunk in md.iterload(
                str(dcd_path),
                top=str(SYSTEM_PRMTOP),
                stride=ANALYSIS_STRIDE,
                chunk=ANALYSIS_CHUNK_SIZE,
            ):
                chunk_distances = (
                    md.compute_distances(
                        chunk,
                        pairs,
                    )
                    .min(axis=1)
                    * 10.0
                )

                frame_indices = (
                    global_frame_index
                    + np.arange(
                        len(chunk_distances),
                    )
                    * ANALYSIS_STRIDE
                )

                time_ns = (
                    frame_indices
                    * PROD_DCD_PS
                    / 1000.0
                )

                all_distances.append(
                    chunk_distances,
                )

                all_times.append(
                    time_ns,
                )

                global_frame_index += (
                    len(chunk_distances)
                    * ANALYSIS_STRIDE
                )

                del chunk_distances
                del frame_indices
                del time_ns
                del chunk


        if not all_distances:
            raise RuntimeError(
                "Не удалось обработать ни одного "
                "кадра Production-траектории."
            )


        distances_ang = np.concatenate(
            all_distances,
        )

        time_ns = np.concatenate(
            all_times,
        )


        # ------------------------------------------------------------------
        # 7. Проверка результатов
        # ------------------------------------------------------------------

        if not np.all(
            np.isfinite(distances_ang),
        ):
            raise ValueError(
                "В расстояниях металл–O обнаружены "
                "NaN или Inf."
            )

        if not np.all(
            np.isfinite(time_ns),
        ):
            raise ValueError(
                "Во временной шкале обнаружены "
                "NaN или Inf."
            )


        # ------------------------------------------------------------------
        # 8. Статистика
        # ------------------------------------------------------------------

        mean_distance = np.mean(
            distances_ang,
        )

        median_distance = np.median(
            distances_ang,
        )

        min_distance = np.min(
            distances_ang,
        )

        max_distance = np.max(
            distances_ang,
        )

        std_distance = np.std(
            distances_ang,
        )

        print(
            f"\nОбработано кадров: "
            f"{len(distances_ang)}"
        )

        print(
            f"\nМинимальное расстояние металл–O:"
            f"\n  Среднее  : "
            f"{mean_distance:.2f} Å"
            f"\n  Медиана  : "
            f"{median_distance:.2f} Å"
            f"\n  SD       : "
            f"{std_distance:.2f} Å"
            f"\n  Минимум  : "
            f"{min_distance:.2f} Å"
            f"\n  Максимум : "
            f"{max_distance:.2f} Å"
        )


        # ------------------------------------------------------------------
        # 9. Подготовка KDE
        # ------------------------------------------------------------------

        if len(distances_ang) > MAX_KDE_POINTS:
            kde_indices = np.linspace(
                0,
                len(distances_ang) - 1,
                MAX_KDE_POINTS,
                dtype=int,
            )

            kde_data = distances_ang[
                kde_indices
            ]

        else:
            kde_data = distances_ang


        # ------------------------------------------------------------------
        # 10. Визуализация
        # ------------------------------------------------------------------

        FIGURES_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(12, 4.5),
        )


        # Расстояние во времени.
        axes[0].plot(
            time_ns,
            distances_ang,
            linewidth=1.2,
        )

        axes[0].set_title(
            "Metal–Protein Oxygen Distance",
        )

        axes[0].set_xlabel(
            "Time (ns)",
        )

        axes[0].set_ylabel(
            "Minimum Metal–O Distance (Å)",
        )

        axes[0].set_xlim(
            left=0,
        )

        axes[0].set_ylim(
            bottom=0,
        )

        axes[0].grid(
            linestyle="-",
            linewidth=0.5,
            alpha=0.7,
        )


        # Распределение.
        axes[1].hist(
            distances_ang,
            bins=30,
            density=True,
            alpha=0.5,
            edgecolor="black",
        )

        if (
            len(kde_data) > 1
            and np.std(kde_data) > 0
        ):
            kde_x = np.linspace(
                kde_data.min(),
                kde_data.max(),
                400,
            )

            kde = gaussian_kde(
                kde_data,
            )

            axes[1].plot(
                kde_x,
                kde(kde_x),
                linewidth=2.0,
                label="KDE",
            )

            axes[1].legend()


        axes[1].set_title(
            "Metal–Protein Distance Distribution",
        )

        axes[1].set_xlabel(
            "Minimum Metal–O Distance (Å)",
        )

        axes[1].set_ylabel(
            "Probability Density",
        )

        axes[1].set_xlim(
            left=0,
        )

        axes[1].grid(
            linestyle="-",
            linewidth=0.5,
            alpha=0.7,
        )


        fig.suptitle(
            f"{SYSTEM_BASENAME} — Metal Coordination",
            fontsize=14,
        )

        fig.tight_layout()


        # ------------------------------------------------------------------
        # 11. Сохранение PNG + SVG
        # ------------------------------------------------------------------

        png_path = (
            FIGURES_DIR
            / f"{SYSTEM_BASENAME}_metal_coordination_dist.png"
        )

        svg_path = (
            FIGURES_DIR
            / f"{SYSTEM_BASENAME}_metal_coordination_dist.svg"
        )

        fig.savefig(
            png_path,
            dpi=300,
            bbox_inches="tight",
        )

        fig.savefig(
            svg_path,
            bbox_inches="tight",
        )

        print(
            "\n📊 Графики координации сохранены:"
        )

        print(
            f"PNG: {png_path.resolve()}"
        )

        print(
            f"SVG: {svg_path.resolve()}"
        )

        plt.show()
        plt.close(fig)


        # ------------------------------------------------------------------
        # 12. Очистка памяти
        # ------------------------------------------------------------------

        del reference
        del topology
        del all_distances
        del all_times
        del distances_ang
        del time_ns
        del kde_data
        del pairs

        gc.collect()

        print(
            "\n✅ Потоковый анализ координации "
            "металла завершён."
        )

In [ ]:
# @title 🔍 20. Отладка присутствия ионов металлов в пайплайне ✅

import re
from pathlib import Path

import mdtraj as md
import parmed as pmd


ANALYSIS_CHUNK_SIZE = 10


print(
    f"--- Отладка металлов: "
    f"{SYSTEM_BASENAME} ---"
)


# ------------------------------------------------------------------
# 1. Настройки металлов
# ------------------------------------------------------------------

def get_metal_elements() -> set[str]:
    """Возвращает нормализованный набор исследуемых металлов."""
    return {
        str(element).strip().upper()
        for element in globals().get(
            "METAL_ELEMENTS_LIST",
            [
                "MG",
                "ZN",
                "CA",
                "FE",
                "MN",
                "CU",
                "NA",
                "K",
            ],
        )
        if str(element).strip()
    }


def find_metal_in_pdb(
    pdb_path: Path,
    metal_elements: set[str],
) -> tuple[bool, str | None]:
    """Ищет первый атом целевого металла в PDB."""
    if not pdb_path.is_file():
        return False, None

    try:
        with pdb_path.open(
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as handle:
            for line in handle:
                if not line.startswith(
                    ("ATOM", "HETATM"),
                ):
                    continue

                residue_name = (
                    line[17:20]
                    .strip()
                    .upper()
                )

                atom_name = (
                    line[12:16]
                    .strip()
                    .upper()
                )

                element = (
                    line[76:78]
                    .strip()
                    .upper()
                    if len(line) >= 78
                    else ""
                )

                identifiers = {
                    residue_name,
                    atom_name,
                    element,
                }

                if identifiers & metal_elements:
                    return True, line[:80].rstrip()

    except OSError as exc:
        return False, f"Ошибка чтения: {exc}"

    return False, None


def find_metal_atoms_in_parmed(
    topology,
    metal_elements: set[str],
):
    """Возвращает атомы металлов из ParmEd-топологии."""
    return [
        atom
        for atom in topology.atoms
        if (
            atom.name.strip().upper()
            in metal_elements
            or atom.residue.name.strip().upper()
            in metal_elements
        )
    ]


def find_metal_atoms_in_mdtraj(
    topology: md.Topology,
    metal_elements: set[str],
) -> list[int]:
    """Возвращает индексы атомов металлов MDTraj."""
    metal_indices = []

    for atom in topology.atoms:
        identifiers = {
            atom.name.strip().upper(),
            atom.residue.name.strip().upper(),
        }

        if atom.element is not None:
            identifiers.add(
                atom.element.symbol.strip().upper(),
            )

        if identifiers & metal_elements:
            metal_indices.append(atom.index)

    return metal_indices


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


metal_elements = get_metal_elements()

print(
    "Исследуемые металлы: "
    + ", ".join(sorted(metal_elements))
)


# ------------------------------------------------------------------
# 2. Проверка PDB-файлов
# ------------------------------------------------------------------

pdb_files = {
    "protein_input.pdb": globals().get("PROT_IN"),
    "protein_amber.pdb": globals().get(
        "PROT_AMBER_PDB",
    ),
    "prepared_complex.pdb": globals().get(
        "PREPARED_COMPLEX_PDB",
    ),
    "tleap_system.pdb": globals().get(
        "SYSTEM_PDB",
    ),
}


print(
    "\n---- 1. Проверка PDB-файлов ----"
)

for name, path in pdb_files.items():
    if path is None:
        print(
            f"{name:22s} | ⚠️ Путь не задан"
        )
        continue

    pdb_path = Path(path)

    if not pdb_path.is_file():
        print(
            f"{name:22s} | ❌ Файл отсутствует"
        )
        continue

    found, line = find_metal_in_pdb(
        pdb_path=pdb_path,
        metal_elements=metal_elements,
    )

    status = (
        "✅ НАЙДЕН"
        if found
        else "❌ Отсутствует"
    )

    print(
        f"{name:22s} | "
        f"{pdb_path.name:30s} | "
        f"{status}"
    )

    if line:
        print(
            f"   Пример: {line}"
        )


# ------------------------------------------------------------------
# 3. Проверка AMBER prmtop
# ------------------------------------------------------------------

print(
    "\n---- 2. Проверка AMBER prmtop ----"
)

prmtop_path = Path(
    SYSTEM_PRMTOP,
)

parm = None
parm_metal_indices = []

if (
    prmtop_path.is_file()
    and prmtop_path.stat().st_size > 2000
):
    try:
        parm = pmd.load_file(
            str(prmtop_path),
        )

        parm_metal_atoms = (
            find_metal_atoms_in_parmed(
                parm,
                metal_elements,
            )
        )

        parm_metal_indices = [
            atom.idx
            for atom in parm_metal_atoms
        ]

        print(
            f"Атомов в prmtop: "
            f"{len(parm.atoms)}"
        )

        print(
            f"Найдено металлов в prmtop: "
            f"{len(parm_metal_atoms)}"
        )

        for atom in parm_metal_atoms[:5]:
            print(
                f"  • {atom.idx}: "
                f"{atom.name} "
                f"({atom.residue.name}"
                f"{atom.residue.number})"
            )

    except Exception as exc:
        print(
            f"❌ Ошибка чтения prmtop: {exc}"
        )

else:
    print(
        "❌ Файл prmtop отсутствует "
        "или имеет подозрительно маленький размер."
    )


# ------------------------------------------------------------------
# 4. Проверка DCD-траекторий
# ------------------------------------------------------------------

print(
    "\n---- 3. Проверка DCD-траекторий ----"
)

dcd_files = sorted(
    WORKDIR.glob(
        f"{PROD_JOB}*.dcd",
    ),
    key=dcd_sort_key,
)

if not dcd_files:
    print(
        "❌ DCD-файлы Production MD "
        "не найдены."
    )

elif parm is None:
    print(
        "❌ prmtop отсутствует или "
        "не удалось его загрузить — "
        "невозможно проверить DCD."
    )

else:
    print(
        f"Найдено DCD-файлов: "
        f"{len(dcd_files)}"
    )

    expected_particles = len(
        parm.atoms,
    )

    for dcd_path in dcd_files:
        print(
            f"\n  • {dcd_path.name}"
        )

        try:
            with md.formats.DCDTrajectoryFile(
                str(dcd_path),
            ) as dcd:
                n_frames = len(dcd)

                if n_frames == 0:
                    print(
                        "    ❌ DCD не содержит кадров."
                    )
                    continue

                first_chunk = dcd.read(
                    n_frames=1,
                )

                coordinates = first_chunk[0]

                n_particles = (
                    coordinates.shape[1]
                )

                if n_particles != expected_particles:
                    print(
                        "    ❌ Несовпадение числа частиц:"
                        f" DCD={n_particles},"
                        f" prmtop={expected_particles}"
                    )
                    continue

            # MDTraj получает топологию только из prmtop.
            topology = md.load_frame(
                str(dcd_path),
                index=0,
                top=str(prmtop_path),
            ).topology

            dcd_metal_indices = (
                find_metal_atoms_in_mdtraj(
                    topology,
                    metal_elements,
                )
            )

            print(
                f"    Кадров: {n_frames}"
            )

            print(
                f"    Частиц: {n_particles}"
            )

            print(
                f"    Металлов: "
                f"{len(dcd_metal_indices)}"
            )

            if (
                dcd_metal_indices
                == parm_metal_indices
            ):
                print(
                    "    ✅ Индексы металлов "
                    "совпадают с prmtop."
                )
            else:
                print(
                    "    ⚠️ Индексы металлов "
                    "отличаются от prmtop."
                )

        except Exception as exc:
            print(
                f"    ❌ Ошибка чтения: {exc}"
            )


# ------------------------------------------------------------------
# 5. Итог
# ------------------------------------------------------------------

print(
    "\n---- 4. Итоговая проверка ----"
)

if parm is not None:
    print(
        f"Металлов в AMBER topology: "
        f"{len(parm_metal_indices)}"
    )

if not dcd_files:
    print(
        "⚠️ Production DCD пока отсутствуют."
    )
elif parm is not None:
    print(
        "✅ Проверка присутствия металлов "
        "в пайплайне завершена."
    )

In [ ]:
# @title 🧪 21. Подготовка подтраектории комплекса для MM/PBSA ✅

import gc
import re
from pathlib import Path

import mdtraj as md


ANALYSIS_STRIDE = 1
ANALYSIS_CHUNK_SIZE = 10


def dcd_sort_key(
    path: Path,
) -> tuple[int, str]:
    """Сортирует основной DCD перед последующими partN."""
    match = re.search(
        r"_part(\d+)\.dcd$",
        path.name,
    )

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет наличие данных и возможность чтения DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(
            str(path),
        ) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


def select_solute_atoms(
    topology: md.Topology,
) -> list[int]:
    """
    Выбирает белок и малую молекулу LIG,
    исключая воду, ионы и прочие компоненты.
    """
    protein_atoms = topology.select(
        "protein",
    )

    ligand_atoms = topology.select(
        f"resname {SMALL_MOL_RESNAME}",
    )

    atom_indices = sorted(
        set(
            protein_atoms.tolist()
        )
        | set(
            ligand_atoms.tolist()
        )
    )

    if not atom_indices:
        raise ValueError(
            "Не удалось найти белок или лиганд "
            "в топологии."
        )

    if len(ligand_atoms) == 0:
        raise ValueError(
            f"Лиганд {SMALL_MOL_RESNAME} "
            "не найден в топологии."
        )

    return atom_indices


print(
    f"--- Потоковая подготовка подтраектории "
    f"комплекса без растворителя: "
    f"{SYSTEM_BASENAME} ---"
)


# ------------------------------------------------------------------
# 1. Поиск Production DCD
# ------------------------------------------------------------------

dcd_candidates = [
    WORKDIR / f"{PROD_JOB}.dcd",
    *sorted(
        WORKDIR.glob(
            f"{PROD_JOB}_part*.dcd",
        ),
        key=dcd_sort_key,
    ),
]

valid_dcd_files = [
    path
    for path in dcd_candidates
    if validate_dcd(path)
]

if not valid_dcd_files:
    raise FileNotFoundError(
        "Корректные Production DCD-файлы "
        "не найдены."
    )

print(
    "\nDCD-файлы для обработки:"
)

for path in valid_dcd_files:
    print(
        f"  • {path.name} "
        f"({path.stat().st_size / 1024**2:.1f} MB)"
    )


# ------------------------------------------------------------------
# 2. Reference topology
# ------------------------------------------------------------------

reference = md.load_frame(
    str(valid_dcd_files[0]),
    index=0,
    top=str(SYSTEM_PRMTOP),
)

topology = reference.topology

print(
    f"\nПолная система: "
    f"{topology.n_atoms} атомов"
)


# ------------------------------------------------------------------
# 3. Выбор solute
# ------------------------------------------------------------------

atom_indices = select_solute_atoms(
    topology,
)

print(
    f"Белок + лиганд: "
    f"{len(atom_indices)} атомов"
)

protein_indices = topology.select(
    "protein",
)

ligand_indices = topology.select(
    f"resname {SMALL_MOL_RESNAME}",
)

print(
    f"  • Белок : {len(protein_indices)} атомов"
)

print(
    f"  • Лиганд: {len(ligand_indices)} атомов"
)


# ------------------------------------------------------------------
# 4. Подготовка выходных файлов
# ------------------------------------------------------------------

solute_dcd = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_solute_complex.dcd"
)

solute_pdb = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_solute_complex.pdb"
)

if solute_dcd.is_file():
    solute_dcd.unlink()

if solute_pdb.is_file():
    solute_pdb.unlink()


# ------------------------------------------------------------------
# 5. Создание solute-топологии
# ------------------------------------------------------------------

solute_reference = reference.atom_slice(
    atom_indices,
)

solute_reference[0].save_pdb(
    str(solute_pdb),
)

solute_topology = solute_reference.topology


# ------------------------------------------------------------------
# 6. Потоковая запись DCD
# ------------------------------------------------------------------

print(
    "\nЗапуск потоковой записи "
    "solute DCD..."
)

total_frames = 0

with md.formats.DCDTrajectoryFile(
    str(solute_dcd),
    mode="w",
) as dcd_writer:

    for dcd_path in valid_dcd_files:
        print(
            f"Обработка: {dcd_path.name}"
        )

        for chunk in md.iterload(
            str(dcd_path),
            top=str(SYSTEM_PRMTOP),
            stride=ANALYSIS_STRIDE,
            chunk=ANALYSIS_CHUNK_SIZE,
        ):
            solute_chunk = chunk.atom_slice(
                atom_indices,
            )

            dcd_writer.write(
                solute_chunk.xyz,
                cell_lengths=solute_chunk.unitcell_lengths,
                cell_angles=solute_chunk.unitcell_angles,
            )

            total_frames += (
                solute_chunk.n_frames
            )

            del solute_chunk
            del chunk


# ------------------------------------------------------------------
# 7. Проверка результата
# ------------------------------------------------------------------

if not solute_dcd.is_file():
    raise RuntimeError(
        "DCD подтраектории не был создан."
    )

if solute_dcd.stat().st_size == 0:
    raise RuntimeError(
        "DCD подтраектории пуст."
    )

if not solute_pdb.is_file():
    raise RuntimeError(
        "PDB подтраектории не был создан."
    )


dcd_size_mb = (
    solute_dcd.stat().st_size
    / (1024**2)
)


print(
    "\n✅ Подтраектория комплекса сформирована:"
)

print(
    f"  • Атомов: "
    f"{len(atom_indices)}"
)

print(
    f"  • Кадров: "
    f"{total_frames}"
)

print(
    f"  • Белок: "
    f"{len(protein_indices)} атомов"
)

print(
    f"  • Лиганд: "
    f"{len(ligand_indices)} атомов"
)

print(
    f"  • PDB первого кадра: "
    f"{solute_pdb.resolve()}"
)

print(
    f"  • DCD: "
    f"{solute_dcd.resolve()} "
    f"({dcd_size_mb:.2f} MB)"
)


# ------------------------------------------------------------------
# 8. Финальная проверка DCD
# ------------------------------------------------------------------

with md.formats.DCDTrajectoryFile(
    str(solute_dcd),
) as dcd_check:
    written_frames = len(dcd_check)

if written_frames != total_frames:
    raise RuntimeError(
        "Количество записанных кадров "
        "не совпадает с ожидаемым: "
        f"{written_frames} != {total_frames}"
    )

print(
    "✅ Число кадров DCD проверено."
)


# ------------------------------------------------------------------
# 9. Очистка памяти
# ------------------------------------------------------------------

del reference
del topology
del solute_reference
del solute_topology
del protein_indices
del ligand_indices
del atom_indices

gc.collect()

print(
    "\n✅ Потоковая подготовка "
    "MM/PBSA подтраектории завершена."
)

In [ ]:
# @title 🧪 22. Безопасная подтраектория комплекса для MM/PBSA ✅

import gc
import re
from pathlib import Path

import mdtraj as md
import numpy as np


ANALYSIS_STRIDE = 1
ANALYSIS_CHUNK_SIZE = 10

EXCLUDED_RESNAMES = {
    "WAT",
    "HOH",
    "TIP3",
    "TIP3P",
    "SOL",
    "NA",
    "NA+",
    "CL",
    "CL-",
    "SOD",
    "CLA",
    "K",
    "K+",
    "POT",
    "MG",
    "CA",
}

SOLUTE_SELECTION = (
    "protein or resname LIG"
)


def dcd_sort_key(path: Path) -> tuple[int, str]:
    """Сортирует DCD по номеру части Production-траектории."""
    match = re.search(r"_part(\d+)\.dcd$", path.name)

    if match:
        return int(match.group(1)), path.name

    return 0, path.name


def validate_dcd(
    path: Path,
    min_size_mb: float = 0.1,
) -> bool:
    """Проверяет существование и читаемость DCD."""
    if not path.is_file():
        return False

    if path.stat().st_size < min_size_mb * 1024**2:
        return False

    try:
        with md.formats.DCDTrajectoryFile(str(path)) as dcd:
            return len(dcd) > 0
    except Exception:
        return False


print(
    f"--- Формирование подтраектории комплекса "
    f"(MM/PBSA): {SYSTEM_BASENAME} ---"
)


# Поиск Production DCD.
dcd_files = sorted(
    [
        path
        for path in WORKDIR.glob(
            f"{JOB_PREFIX}_prod*.dcd"
        )
        if validate_dcd(path)
    ],
    key=dcd_sort_key,
)

if not dcd_files:
    raise FileNotFoundError(
        "Production DCD-файлы не найдены. "
        "Сначала необходимо завершить Production MD."
    )

print(
    f"Найдено Production DCD: {len(dcd_files)}"
)

for path in dcd_files:
    print(
        f"  • {path.name}: "
        f"{path.stat().st_size / 1024**2:.2f} МБ"
    )


# Проверяем топологию и определяем атомы комплекса.
reference = md.load_frame(
    str(dcd_files[0]),
    index=0,
    top=str(SYSTEM_PRMTOP),
)

if reference.n_atoms != reference.topology.n_atoms:
    raise ValueError(
        "Количество атомов траектории не совпадает "
        "с количеством атомов topology."
    )


atom_indices = reference.topology.select(
    SOLUTE_SELECTION
)

if len(atom_indices) == 0:
    raise ValueError(
        "Маска комплекса не содержит атомов. "
        "Проверьте topology и имя LIG."
    )


ligand_indices = reference.topology.select(
    "resname LIG"
)

if len(ligand_indices) == 0:
    raise ValueError(
        "В topology не найден residue LIG."
    )


protein_indices = reference.topology.select(
    "protein"
)

if len(protein_indices) == 0:
    raise ValueError(
        "В topology не найдены атомы белка."
    )


solute_dcd = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_solute_complex.dcd"
)

solute_pdb = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_solute_complex.pdb"
)


# Удаляем старый результат, чтобы не допустить
# случайного добавления кадров к предыдущему файлу.
if solute_dcd.is_file():
    solute_dcd.unlink()

if solute_pdb.is_file():
    solute_pdb.unlink()


# Сохраняем PDB первого кадра комплекса.
reference_solute = reference.atom_slice(
    atom_indices
)

reference_solute.save_pdb(
    str(solute_pdb)
)


# Потоковая запись подтраектории.
total_frames = 0
total_input_frames = 0

with md.formats.DCDTrajectoryFile(
    str(solute_dcd),
    mode="w",
) as dcd_writer:

    for dcd_path in dcd_files:
        print(
            f"Обработка: {dcd_path.name}"
        )

        with md.formats.DCDTrajectoryFile(
            str(dcd_path)
        ) as dcd_reader:

            total_input_frames += len(dcd_reader)

        for chunk in md.iterload(
            str(dcd_path),
            top=str(SYSTEM_PRMTOP),
            stride=ANALYSIS_STRIDE,
            chunk=ANALYSIS_CHUNK_SIZE,
        ):
            if chunk.n_atoms != reference.n_atoms:
                raise ValueError(
                    f"Некорректное число атомов в "
                    f"{dcd_path.name}: "
                    f"{chunk.n_atoms} вместо "
                    f"{reference.n_atoms}."
                )

            chunk_solute = chunk.atom_slice(
                atom_indices
            )

            if (
                chunk_solute.unitcell_lengths is None
                or chunk_solute.unitcell_angles is None
            ):
                raise ValueError(
                    f"В {dcd_path.name} отсутствуют "
                    "параметры периодической коробки."
                )

            dcd_writer.write(
                chunk_solute.xyz,
                cell_lengths=chunk_solute.unitcell_lengths,
                cell_angles=chunk_solute.unitcell_angles,
            )

            total_frames += chunk_solute.n_frames

            del chunk
            del chunk_solute

            gc.collect()


if not solute_dcd.is_file():
    raise RuntimeError(
        "DCD подтраектории не был создан."
    )

if total_frames == 0:
    raise RuntimeError(
        "Подтраектория создана, но не содержит кадров."
    )


# Финальная проверка результата.
with md.formats.DCDTrajectoryFile(
    str(solute_dcd)
) as dcd_check:
    saved_frames = len(dcd_check)


if saved_frames != total_frames:
    raise RuntimeError(
        "Количество записанных кадров не совпадает "
        f"с ожидаемым: {saved_frames} != {total_frames}."
    )


dcd_size_mb = (
    solute_dcd.stat().st_size
    / (1024**2)
)


print()
print("✅ Подтраектория успешно сформирована:")
print(
    f"  • Атомов в исходной траектории: "
    f"{reference.n_atoms}"
)
print(
    f"  • Атомов белка: "
    f"{len(protein_indices)}"
)
print(
    f"  • Атомов лиганда LIG: "
    f"{len(ligand_indices)}"
)
print(
    f"  • Атомов комплекса: "
    f"{len(atom_indices)}"
)
print(
    f"  • Кадров Production: "
    f"{total_input_frames}"
)
print(
    f"  • Кадров в подтраектории: "
    f"{saved_frames}"
)
print(
    f"  • PDB первого кадра: "
    f"{solute_pdb.name}"
)
print(
    f"  • DCD: "
    f"{solute_dcd.name} "
    f"({dcd_size_mb:.2f} МБ)"
)
print(
    f"  • Рабочая директория: "
    f"{solute_dcd.parent.resolve()}"
)


del reference
del reference_solute
gc.collect()

In [ ]:
# @title ⚡ 23. Расчёт свободной энергии связывания MM/GBSA через MMPBSA.py ✅

import gc
import os
import shutil
import subprocess
import textwrap
from pathlib import Path

import mdtraj as md
import numpy as np
import parmed as pmd


print(
    f"--- Расчёт свободной энергии связывания MM/GBSA: "
    f"{SYSTEM_BASENAME} ---"
)


# ---------------------------------------------------------------------------
# Конфигурация
# ---------------------------------------------------------------------------

AMBER_HOME = str(
    globals().get(
        "AMBERHOME",
        "/usr/local",
    )
)

os.environ["AMBERHOME"] = AMBER_HOME

FULL_PRMTOP = Path(SYSTEM_PRMTOP)

SOLUTE_DCD = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_solute_complex.dcd"
)

if not SOLUTE_DCD.is_file():
    fallback_dcd = WORKDIR / "prod_solute_complex.dcd"

    if fallback_dcd.is_file():
        SOLUTE_DCD = fallback_dcd

SOLUTE_PRMTOP = WORKDIR / "complex_solute.prmtop"
RECEPTOR_PRMTOP = WORKDIR / "receptor.prmtop"
LIGAND_PRMTOP = WORKDIR / "ligand.prmtop"

COMPLEX_IFBOX0_PRMTOP = (
    WORKDIR / "complex_IFBOX0.prmtop"
)
RECEPTOR_IFBOX0_PRMTOP = (
    WORKDIR / "receptor_IFBOX0.prmtop"
)
LIGAND_IFBOX0_PRMTOP = (
    WORKDIR / "ligand_IFBOX0.prmtop"
)

MMPBSA_INPUT = WORKDIR / "mmpbsa.in"
MMPBSA_LOG = WORKDIR / "MMPBSA.log"
MMPBSA_RESULTS = (
    WORKDIR / "FINAL_RESULTS_MMPBSA.dat"
)

MMPBSA_EXECUTABLE = shutil.which(
    "MMPBSA.py"
)


# ---------------------------------------------------------------------------
# Проверка входных файлов
# ---------------------------------------------------------------------------

if (
    not FULL_PRMTOP.is_file()
    or FULL_PRMTOP.stat().st_size <= 2000
):
    raise FileNotFoundError(
        f"Корректная топология не найдена: "
        f"{FULL_PRMTOP}"
    )

if (
    not SOLUTE_DCD.is_file()
    or SOLUTE_DCD.stat().st_size <= 1000
):
    raise FileNotFoundError(
        f"Корректная подтраектория не найдена: "
        f"{SOLUTE_DCD}"
    )

if MMPBSA_EXECUTABLE is None:
    raise FileNotFoundError(
        "Исполняемый файл MMPBSA.py не найден в PATH."
    )


print(f"Топология : {FULL_PRMTOP.name}")
print(f"Траектория: {SOLUTE_DCD.name}")
print(f"MMPBSA.py : {MMPBSA_EXECUTABLE}")


# ---------------------------------------------------------------------------
# Очистка результатов предыдущего расчёта
# ---------------------------------------------------------------------------

OLD_RESULT_FILES = (
    SOLUTE_PRMTOP,
    RECEPTOR_PRMTOP,
    LIGAND_PRMTOP,
    COMPLEX_IFBOX0_PRMTOP,
    RECEPTOR_IFBOX0_PRMTOP,
    LIGAND_IFBOX0_PRMTOP,
    MMPBSA_RESULTS,
    MMPBSA_LOG,
    MMPBSA_INPUT,
    WORKDIR / "_parmed_make_solute.in",
)

for file_path in OLD_RESULT_FILES:
    if file_path.is_file():
        file_path.unlink()


# ---------------------------------------------------------------------------
# Формирование topology комплекса без растворителя
# ---------------------------------------------------------------------------

EXCLUDED_RESNAMES = {
    "WAT",
    "HOH",
    "TIP3",
    "TIP3P",
    "SOL",
    "NA",
    "NA+",
    "CL",
    "CL-",
    "SOD",
    "CLA",
    "K",
    "K+",
    "POT",
    "MG",
    "CA",
}


parm_full = pmd.load_file(
    str(FULL_PRMTOP)
)

parm_solute = parm_full.copy()

residues_to_remove = [
    residue
    for residue in parm_solute.residues
    if residue.name.strip().upper()
    in EXCLUDED_RESNAMES
]

for residue in reversed(residues_to_remove):
    parm_solute.strip(
        f":{residue.number}"
    )

parm_solute.save(
    str(SOLUTE_PRMTOP),
    overwrite=True,
)

if not SOLUTE_PRMTOP.is_file():
    raise RuntimeError(
        "Не удалось создать solute prmtop."
    )

print(
    f"✅ Solute topology: "
    f"{SOLUTE_PRMTOP.name}"
)


# ---------------------------------------------------------------------------
# Поиск остатков лиганда
# ---------------------------------------------------------------------------

parm_solute_check = pmd.load_file(
    str(SOLUTE_PRMTOP)
)

residue_names = [
    residue.name.strip().upper()
    for residue in parm_solute_check.residues
]

residue_count = len(residue_names)


def find_ligand_residues(
    residue_names: list[str],
) -> list[int]:
    """Возвращает номера остатков лиганда в prmtop."""
    if LIGAND_KIND == "small_molecule":
        target_name = (
            SMALL_MOL_RESNAME
            .strip()
            .upper()
        )

        return [
            index + 1
            for index, residue_name in enumerate(
                residue_names
            )
            if residue_name == target_name
        ]

    if LIGAND_KIND == "nucleic_acid":
        allowed_names = {
            "A",
            "C",
            "G",
            "U",
            "DA",
            "DC",
            "DG",
            "DT",
            "RA",
            "RC",
            "RG",
            "RU",
            "DA5",
            "DG5",
            "DT3",
            "DC3",
            "A5",
            "C3",
            "G5",
        }

        modified_name = str(
            globals().get(
                "MODIFIED_NA_RESNAME",
                "",
            )
        ).strip().upper()

        if modified_name:
            allowed_names.add(
                modified_name
            )

        return [
            index + 1
            for index, residue_name in enumerate(
                residue_names
            )
            if residue_name in allowed_names
        ]

    raise ValueError(
        f"Неизвестный LIGAND_KIND: "
        f"{LIGAND_KIND}"
    )


ligand_residues = find_ligand_residues(
    residue_names
)

if not ligand_residues:
    raise ValueError(
        "Остатки лиганда не найдены "
        "в solute topology."
    )


ligand_min = min(ligand_residues)
ligand_max = max(ligand_residues)

expected_residues = list(
    range(
        ligand_min,
        ligand_max + 1,
    )
)

if ligand_residues != expected_residues:
    raise ValueError(
        "Остатки лиганда не образуют "
        "непрерывный блок. "
        f"Найдены: {ligand_residues}"
    )


print(
    f"Лиганд: остатки "
    f"{ligand_min}–{ligand_max} "
    f"из {residue_count}"
)


# ---------------------------------------------------------------------------
# Формирование receptor / ligand topology
# ---------------------------------------------------------------------------

receptor = parm_solute_check.copy()

receptor.strip(
    f":{ligand_min}-{ligand_max}"
)

receptor.save(
    str(RECEPTOR_PRMTOP),
    overwrite=True,
)


ligand = parm_solute_check.copy()

if ligand_max < residue_count:
    ligand.strip(
        f":{ligand_max + 1}-{residue_count}"
    )

if ligand_min > 1:
    ligand.strip(
        f":1-{ligand_min - 1}"
    )

ligand.save(
    str(LIGAND_PRMTOP),
    overwrite=True,
)


# ---------------------------------------------------------------------------
# Проверка согласованности количества атомов
# ---------------------------------------------------------------------------

def get_atom_count(
    topology_path: Path,
) -> int:
    """Возвращает количество атомов в topology."""
    structure = pmd.load_file(
        str(topology_path)
    )

    return structure.ptr("natom")


complex_atom_count = get_atom_count(
    SOLUTE_PRMTOP
)

receptor_atom_count = get_atom_count(
    RECEPTOR_PRMTOP
)

ligand_atom_count = get_atom_count(
    LIGAND_PRMTOP
)

if complex_atom_count != (
    receptor_atom_count
    + ligand_atom_count
):
    raise ValueError(
        "Несогласованное число атомов:\n"
        f"  Complex = {complex_atom_count}\n"
        f"  Receptor = {receptor_atom_count}\n"
        f"  Ligand = {ligand_atom_count}\n"
        f"  Receptor + Ligand = "
        f"{receptor_atom_count + ligand_atom_count}"
    )


print("✅ Число атомов согласовано:")
print(f"  Complex : {complex_atom_count}")
print(f"  Receptor: {receptor_atom_count}")
print(f"  Ligand  : {ligand_atom_count}")


# ---------------------------------------------------------------------------
# Подготовка IFBOX=0 topology
# ---------------------------------------------------------------------------

def save_ifbox_zero(
    source_path: Path,
    output_path: Path,
) -> None:
    """Создаёт topology с отключённым периодическим боксом."""
    structure = pmd.load_file(
        str(source_path)
    )

    pointers = structure.parm_data.get(
        "POINTERS"
    )

    if pointers is None or len(pointers) <= 27:
        raise ValueError(
            f"Не найден POINTERS в "
            f"{source_path.name}."
        )

    pointers[27] = 0

    structure.save(
        str(output_path),
        overwrite=True,
    )


save_ifbox_zero(
    SOLUTE_PRMTOP,
    COMPLEX_IFBOX0_PRMTOP,
)

save_ifbox_zero(
    RECEPTOR_PRMTOP,
    RECEPTOR_IFBOX0_PRMTOP,
)

save_ifbox_zero(
    LIGAND_PRMTOP,
    LIGAND_IFBOX0_PRMTOP,
)

print("✅ Созданы topology с IFBOX=0.")


# ---------------------------------------------------------------------------
# Проверка соответствия topology и DCD
# ---------------------------------------------------------------------------

with md.formats.DCDTrajectoryFile(
    str(SOLUTE_DCD)
) as dcd_reader:
    total_frames = len(dcd_reader)

    if total_frames == 0:
        raise ValueError(
            "Подтраектория DCD не содержит кадров."
        )

    first_frame = dcd_reader.read(
        n_frames=1
    )

dcd_atom_count = first_frame[0].shape[1]

if dcd_atom_count != complex_atom_count:
    raise ValueError(
        "Число атомов DCD не соответствует "
        "solute topology:\n"
        f"  DCD      : {dcd_atom_count}\n"
        f"  topology : {complex_atom_count}"
    )

if total_frames < 2:
    raise ValueError(
        "Для MM/GBSA требуется минимум "
        "2 кадра траектории."
    )

print(
    "✅ DCD ↔ topology согласованы:"
)
print(
    f"  Атомов: {dcd_atom_count}"
)
print(
    f"  Кадров: {total_frames}"
)


# ---------------------------------------------------------------------------
# Формирование набора кадров
# ---------------------------------------------------------------------------

start_frame = max(
    1,
    int(total_frames * 0.20),
)

end_frame = total_frames

available_frames = (
    end_frame - start_frame + 1
)

target_frame_count = min(
    100,
    available_frames,
)

interval = max(
    1,
    int(
        np.ceil(
            available_frames
            / target_frame_count
        )
    ),
)

selected_frame_count = (
    (end_frame - start_frame)
    // interval
    + 1
)


# ---------------------------------------------------------------------------
# Конфигурация MMPBSA
# ---------------------------------------------------------------------------

mmpbsa_input = textwrap.dedent(
    f"""\
    &general
      startframe={start_frame},
      endframe={end_frame},
      interval={interval},
      verbose=1,
      keep_files=0,
      use_sander=1,
    /
    &gb
      igb=5,
      saltcon=0.150,
    /
    """
)

MMPBSA_INPUT.write_text(
    mmpbsa_input,
    encoding="utf-8",
)


print()
print("Параметры MM/GBSA:")
print(
    f"  Всего кадров DCD : {total_frames}"
)
print(
    f"  Первый кадр      : {start_frame}"
)
print(
    f"  Последний кадр   : {end_frame}"
)
print(
    f"  Интервал         : {interval}"
)
print(
    f"  Кадров к расчёту : {selected_frame_count}"
)


# ---------------------------------------------------------------------------
# Запуск MMPBSA.py
# ---------------------------------------------------------------------------

mmpbsa_command = [
    MMPBSA_EXECUTABLE,
    "-O",
    "-i",
    MMPBSA_INPUT.name,
    "-cp",
    COMPLEX_IFBOX0_PRMTOP.name,
    "-rp",
    RECEPTOR_IFBOX0_PRMTOP.name,
    "-lp",
    LIGAND_IFBOX0_PRMTOP.name,
    "-sp",
    COMPLEX_IFBOX0_PRMTOP.name,
    "-y",
    SOLUTE_DCD.name,
    "-o",
    MMPBSA_RESULTS.name,
]

print()
print("Запуск MMPBSA.py...")
print(
    " ".join(mmpbsa_command)
)


result = subprocess.run(
    mmpbsa_command,
    cwd=WORKDIR,
    capture_output=True,
    text=True,
    check=False,
)


MMPBSA_LOG.write_text(
    result.stdout
    + "\n\n=== STDERR ===\n"
    + result.stderr,
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# Проверка результата
# ---------------------------------------------------------------------------

if (
    result.returncode != 0
    or not MMPBSA_RESULTS.is_file()
    or MMPBSA_RESULTS.stat().st_size == 0
):
    print(
        f"❌ MMPBSA.py завершился с ошибкой "
        f"(код {result.returncode})."
    )

    print(
        f"Подробный лог: "
        f"{MMPBSA_LOG.resolve()}"
    )

    if result.stderr.strip():
        print()
        print("Последние сообщения STDERR:")
        print(result.stderr[-5000:])

    raise RuntimeError(
        "Расчёт MM/GBSA не завершён успешно."
    )


print()
print("✅ Расчёт MM/GBSA успешно завершён.")
print(
    f"📊 Результаты: "
    f"{MMPBSA_RESULTS.resolve()}"
)
print(
    f"📝 Лог: "
    f"{MMPBSA_LOG.resolve()}"
)


# ---------------------------------------------------------------------------
# Вывод итоговой секции энергий
# ---------------------------------------------------------------------------

result_lines = MMPBSA_RESULTS.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

delta_started = False

print()
print("--- Итоговые компоненты энергии ---")

for line in result_lines:
    if (
        "Differences (Complex - Receptor - Ligand)"
        in line
        or line.strip().upper() == "DELTA"
    ):
        delta_started = True

    if delta_started and line.strip():
        print(line)


# Освобождаем крупные topology-объекты.
del parm_full
del parm_solute
del parm_solute_check
del receptor
del ligand
del first_frame

gc.collect()

In [ ]:
# @title ⚡ 24. Оптимизированный расчёт MM/GBSA на локальном SSD ✅

import gc
import shutil
import subprocess
import textwrap
from pathlib import Path

import mdtraj as md


# ---------------------------------------------------------------------------
# Конфигурация локального рабочего пространства
# ---------------------------------------------------------------------------

DRIVE_WORKDIR = WORKDIR.resolve()

LOCAL_WORKDIR = Path(
    "/tmp/mmpbsa_local_scratch"
)

LOCAL_WORKDIR.mkdir(
    parents=True,
    exist_ok=True,
)

MMPBSA_EXECUTABLE = shutil.which(
    "MMPBSA.py"
)

if MMPBSA_EXECUTABLE is None:
    raise FileNotFoundError(
        "MMPBSA.py не найден в PATH. "
        "Проверьте установку AmberTools."
    )


# ---------------------------------------------------------------------------
# Входные и выходные файлы
# ---------------------------------------------------------------------------

DRIVE_SOLUTE_PRMTOP = (
    DRIVE_WORKDIR
    / "complex_solute.prmtop"
)

DRIVE_RECEPTOR_PRMTOP = (
    DRIVE_WORKDIR
    / "receptor.prmtop"
)

DRIVE_LIGAND_PRMTOP = (
    DRIVE_WORKDIR
    / "ligand.prmtop"
)

DRIVE_SOLUTE_DCD = (
    DRIVE_WORKDIR
    / f"{SYSTEM_BASENAME}_solute_complex.dcd"
)

LOCAL_SOLUTE_PRMTOP = (
    LOCAL_WORKDIR
    / "complex_solute.prmtop"
)

LOCAL_RECEPTOR_PRMTOP = (
    LOCAL_WORKDIR
    / "receptor.prmtop"
)

LOCAL_LIGAND_PRMTOP = (
    LOCAL_WORKDIR
    / "ligand.prmtop"
)

LOCAL_SOLUTE_DCD = (
    LOCAL_WORKDIR
    / DRIVE_SOLUTE_DCD.name
)

LOCAL_MMPBSA_INPUT = (
    LOCAL_WORKDIR
    / "mmpbsa.in"
)

LOCAL_RESULTS = (
    LOCAL_WORKDIR
    / "FINAL_RESULTS_MMPBSA.dat"
)

LOCAL_LOG = (
    LOCAL_WORKDIR
    / "MMPBSA.log"
)

DRIVE_RESULTS = (
    DRIVE_WORKDIR
    / "FINAL_RESULTS_MMPBSA.dat"
)

DRIVE_LOG = (
    DRIVE_WORKDIR
    / "MMPBSA.log"
)

DRIVE_ERROR_LOG = (
    DRIVE_WORKDIR
    / "MMPBSA_ERROR.log"
)


print(
    f"--- Оптимизированный MM/GBSA: "
    f"{SYSTEM_BASENAME} ---"
)


# ---------------------------------------------------------------------------
# Проверка исходных файлов
# ---------------------------------------------------------------------------

SOURCE_FILES = {
    "complex topology": DRIVE_SOLUTE_PRMTOP,
    "receptor topology": DRIVE_RECEPTOR_PRMTOP,
    "ligand topology": DRIVE_LIGAND_PRMTOP,
    "solute trajectory": DRIVE_SOLUTE_DCD,
}

for description, source_path in SOURCE_FILES.items():
    if (
        not source_path.is_file()
        or source_path.stat().st_size == 0
    ):
        raise FileNotFoundError(
            f"Не найден корректный файл "
            f"{description}: {source_path}"
        )


# ---------------------------------------------------------------------------
# Копирование тяжёлых файлов на локальный SSD
# ---------------------------------------------------------------------------

print(
    "\n⚡ Копирование topology и trajectory "
    "на локальный SSD..."
)

for source_path, local_path in (
    (DRIVE_SOLUTE_PRMTOP, LOCAL_SOLUTE_PRMTOP),
    (DRIVE_RECEPTOR_PRMTOP, LOCAL_RECEPTOR_PRMTOP),
    (DRIVE_LIGAND_PRMTOP, LOCAL_LIGAND_PRMTOP),
    (DRIVE_SOLUTE_DCD, LOCAL_SOLUTE_DCD),
):
    shutil.copy2(
        source_path,
        local_path,
    )

print(
    f"  • Complex topology: "
    f"{LOCAL_SOLUTE_PRMTOP.stat().st_size / 1024**2:.1f} МБ"
)

print(
    f"  • Receptor topology: "
    f"{LOCAL_RECEPTOR_PRMTOP.stat().st_size / 1024**2:.1f} МБ"
)

print(
    f"  • Ligand topology: "
    f"{LOCAL_LIGAND_PRMTOP.stat().st_size / 1024**2:.1f} МБ"
)

print(
    f"  • DCD: "
    f"{LOCAL_SOLUTE_DCD.stat().st_size / 1024**2:.1f} МБ"
)


# ---------------------------------------------------------------------------
# Проверка количества кадров и атомов без загрузки всей DCD
# ---------------------------------------------------------------------------

print(
    "\n🔎 Проверка траектории..."
)

with md.formats.DCDTrajectoryFile(
    str(LOCAL_SOLUTE_DCD)
) as dcd_file:
    total_frames = len(dcd_file)

    if total_frames == 0:
        raise ValueError(
            "DCD не содержит кадров."
        )

    first_frame = dcd_file.read(
        n_frames=1
    )

dcd_atom_count = first_frame[0].shape[1]

print(
    f"  • Кадров: {total_frames}"
)
print(
    f"  • Атомов в DCD: {dcd_atom_count}"
)


# ---------------------------------------------------------------------------
# Проверка topology ↔ DCD
# ---------------------------------------------------------------------------

with md.formats.DCDTrajectoryFile(
    str(LOCAL_SOLUTE_DCD)
) as dcd_file:
    dcd_atom_count = dcd_file.read(
        n_frames=1
    )[0].shape[1]

import parmed as pmd

complex_parm = pmd.load_file(
    str(LOCAL_SOLUTE_PRMTOP)
)

complex_atom_count = complex_parm.ptr(
    "natom"
)

if dcd_atom_count != complex_atom_count:
    raise ValueError(
        "Несовпадение числа атомов:\n"
        f"  DCD      : {dcd_atom_count}\n"
        f"  topology : {complex_atom_count}"
    )

print(
    f"✅ Topology ↔ DCD: "
    f"{complex_atom_count} атомов"
)


# ---------------------------------------------------------------------------
# Проверка receptor / ligand topology
# ---------------------------------------------------------------------------

receptor_parm = pmd.load_file(
    str(LOCAL_RECEPTOR_PRMTOP)
)

ligand_parm = pmd.load_file(
    str(LOCAL_LIGAND_PRMTOP)
)

receptor_atom_count = receptor_parm.ptr(
    "natom"
)

ligand_atom_count = ligand_parm.ptr(
    "natom"
)

if complex_atom_count != (
    receptor_atom_count
    + ligand_atom_count
):
    raise ValueError(
        "Несогласованные topology:\n"
        f"  Complex  : {complex_atom_count}\n"
        f"  Receptor : {receptor_atom_count}\n"
        f"  Ligand   : {ligand_atom_count}\n"
        f"  R + L    : "
        f"{receptor_atom_count + ligand_atom_count}"
    )

print(
    "✅ Topology согласованы:"
)

print(
    f"  • Complex : {complex_atom_count}"
)

print(
    f"  • Receptor: {receptor_atom_count}"
)

print(
    f"  • Ligand  : {ligand_atom_count}"
)


# ---------------------------------------------------------------------------
# Автоматический выбор интервала кадров
# ---------------------------------------------------------------------------

if total_frames < 2:
    raise ValueError(
        "Для MM/GBSA требуется минимум "
        "2 кадра."
    )

TARGET_FRAMES = 200

frame_interval = max(
    1,
    total_frames // TARGET_FRAMES,
)

selected_frame_count = (
    (total_frames - 1)
    // frame_interval
    + 1
)

print()
print(
    "🎯 Выбор кадров для MM/GBSA:"
)

print(
    f"  • Всего кадров : {total_frames}"
)

print(
    f"  • Интервал     : {frame_interval}"
)

print(
    f"  • К расчёту    : "
    f"{selected_frame_count}"
)


# ---------------------------------------------------------------------------
# Конфигурация MMPBSA
# ---------------------------------------------------------------------------

mmpbsa_input = textwrap.dedent(
    f"""\
    &general
      startframe=1,
      endframe={total_frames},
      interval={frame_interval},
      verbose=1,
      keep_files=0,
      use_sander=1,
    /
    &gb
      igb=5,
      saltcon=0.150,
    /
    """
)

LOCAL_MMPBSA_INPUT.write_text(
    mmpbsa_input,
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# Запуск MMPBSA.py с потоковым логированием
# ---------------------------------------------------------------------------

mmpbsa_command = [
    MMPBSA_EXECUTABLE,
    "-O",
    "-i",
    LOCAL_MMPBSA_INPUT.name,
    "-cp",
    LOCAL_SOLUTE_PRMTOP.name,
    "-rp",
    LOCAL_RECEPTOR_PRMTOP.name,
    "-lp",
    LOCAL_LIGAND_PRMTOP.name,
    "-y",
    LOCAL_SOLUTE_DCD.name,
    "-o",
    LOCAL_RESULTS.name,
]

print()
print(
    "🚀 Запуск MMPBSA.py "
    "на локальном SSD..."
)

print(
    f"Рабочая директория: "
    f"{LOCAL_WORKDIR}"
)

print("=" * 70)


return_code = None
log_lines = []

try:
    process = subprocess.Popen(
        mmpbsa_command,
        cwd=LOCAL_WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:
        raise RuntimeError(
            "Не удалось получить stdout MMPBSA.py."
        )

    for line in process.stdout:
        log_lines.append(line)

        message = line.strip()

        if message:
            print(
                f"[MMPBSA] {message}",
                flush=True,
            )

    return_code = process.wait()

finally:
    LOCAL_LOG.write_text(
        "".join(log_lines),
        encoding="utf-8",
    )


print("=" * 70)

print(
    f"Код возврата MMPBSA.py: "
    f"{return_code}"
)


# ---------------------------------------------------------------------------
# Проверка и сохранение результатов
# ---------------------------------------------------------------------------

if (
    return_code == 0
    and LOCAL_RESULTS.is_file()
    and LOCAL_RESULTS.stat().st_size > 0
):
    shutil.copy2(
        LOCAL_RESULTS,
        DRIVE_RESULTS,
    )

    shutil.copy2(
        LOCAL_LOG,
        DRIVE_LOG,
    )

    print()
    print(
        "🟢 MM/GBSA успешно завершён."
    )

    print(
        f"📊 Результаты: "
        f"{DRIVE_RESULTS.resolve()}"
    )

    print(
        f"📝 Лог: "
        f"{DRIVE_LOG.resolve()}"
    )

else:
    shutil.copy2(
        LOCAL_LOG,
        DRIVE_ERROR_LOG,
    )

    print()
    print(
        "🚨 MMPBSA.py завершился с ошибкой."
    )

    print(
        f"Код возврата: {return_code}"
    )

    print(
        f"Аварийный лог: "
        f"{DRIVE_ERROR_LOG.resolve()}"
    )

    raise RuntimeError(
        "Расчёт MM/GBSA не завершён успешно. "
        f"Проверьте {DRIVE_ERROR_LOG.name}."
    )


# ---------------------------------------------------------------------------
# Вывод итоговых энергий
# ---------------------------------------------------------------------------

result_lines = DRIVE_RESULTS.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

delta_started = False

print()
print(
    "--- Итоговые компоненты энергии ---"
)

for line in result_lines:
    if (
        "Differences (Complex - Receptor - Ligand)"
        in line
        or line.strip().upper() == "DELTA"
    ):
        delta_started = True

    if delta_started and line.strip():
        print(line)


# ---------------------------------------------------------------------------
# Очистка памяти и локального scratch
# ---------------------------------------------------------------------------

del complex_parm
del receptor_parm
del ligand_parm
del first_frame

gc.collect()

shutil.rmtree(
    LOCAL_WORKDIR,
    ignore_errors=True,
)

print()
print(
    "🧹 Локальный scratch очищен."
)

In [ ]:
# @title 📊 25. Обработка и сводка результатов MM/GBSA ✅

import re
from pathlib import Path

import pandas as pd


print(
    f"--- Обработка результатов MM/GBSA: "
    f"{SYSTEM_BASENAME} ---"
)


# ---------------------------------------------------------------------------
# Поиск итогового файла
# ---------------------------------------------------------------------------

RESULT_CANDIDATES = (
    WORKDIR / f"{SYSTEM_BASENAME}_MMPBSA_RESULTS.dat",
    WORKDIR / "FINAL_RESULTS_MMPBSA.dat",
)

RESULT_FILE = next(
    (
        path
        for path in RESULT_CANDIDATES
        if path.is_file() and path.stat().st_size > 0
    ),
    None,
)

if RESULT_FILE is None:
    raise FileNotFoundError(
        "Файл результатов MM/GBSA не найден. "
        "Проверьте FINAL_RESULTS_MMPBSA.dat "
        "в рабочей директории."
    )

print(f"📄 Файл результатов: {RESULT_FILE.resolve()}")


# ---------------------------------------------------------------------------
# Названия энергетических компонентов
# ---------------------------------------------------------------------------

ENERGY_LABELS = {
    "VDWAALS": "ΔE_vdw (van der Waals)",
    "EEL": "ΔE_elec (Electrostatic)",
    "EGB": "ΔG_GB (Polar Solvation)",
    "ESURF": "ΔG_np (Non-polar Solvation)",
    "GGAS": "ΔG_gas (Gas-phase Energy)",
    "GSOLV": "ΔG_solv (Solvation Free Energy)",
    "TOTAL": "ΔG_bind (Total Binding Free Energy)",
}

ENERGY_TERMS = "|".join(
    re.escape(term)
    for term in ENERGY_LABELS
)

NUMBER_PATTERN = (
    r"[-+]?"
    r"(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[EeDd][-+]?\d+)?"
)

ENERGY_ROW_PATTERN = re.compile(
    rf"^\s*"
    rf"(?P<term>{ENERGY_TERMS})"
    rf"\s+"
    rf"(?P<mean>{NUMBER_PATTERN})"
    rf"\s+"
    rf"(?P<sd>{NUMBER_PATTERN})"
    rf"(?:\s+(?P<sem>{NUMBER_PATTERN}))?"
    rf"(?:\s|$)",
)


# ---------------------------------------------------------------------------
# Вспомогательные функции
# ---------------------------------------------------------------------------

def parse_number(value: str) -> float:
    """Преобразует обычную или Fortran-запись числа в float."""
    return float(
        value.replace("D", "E").replace("d", "E")
    )


def parse_energy_rows(
    lines: list[str],
) -> list[list[float | str]]:
    """Извлекает энергетические строки из отчёта MMPBSA."""
    rows = []

    for line in lines:
        match = ENERGY_ROW_PATTERN.match(line)

        if match is None:
            continue

        term = match.group("term")

        mean_value = parse_number(
            match.group("mean")
        )
        sd_value = parse_number(
            match.group("sd")
        )

        sem_raw = match.group("sem")

        if sem_raw is not None:
            sem_value = parse_number(sem_raw)
        else:
            # Если MMPBSA не вывел SEM, не подменяем его
            # искусственной оценкой с неизвестным числом кадров.
            sem_value = float("nan")

        rows.append(
            [
                ENERGY_LABELS[term],
                mean_value,
                sd_value,
                sem_value,
            ]
        )

    return rows


# ---------------------------------------------------------------------------
# Чтение и парсинг отчёта
# ---------------------------------------------------------------------------

result_lines = RESULT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

rows = parse_energy_rows(result_lines)

if not rows:
    raise ValueError(
        "Не удалось извлечь энергетические компоненты "
        "из файла MM/GBSA.\n"
        f"Проверьте формат файла: {RESULT_FILE}"
    )


# ---------------------------------------------------------------------------
# Формирование DataFrame
# ---------------------------------------------------------------------------

summary = (
    pd.DataFrame(
        rows,
        columns=[
            "Term",
            "Mean (kcal/mol)",
            "SD",
            "SEM",
        ],
    )
    .drop_duplicates(
        subset="Term",
        keep="last",
    )
    .set_index("Term")
)

print("\n📊 Сводная таблица MM/GBSA:")
display(summary)


# ---------------------------------------------------------------------------
# Проверка числовых значений
# ---------------------------------------------------------------------------

numeric_columns = [
    "Mean (kcal/mol)",
    "SD",
    "SEM",
]

if not summary[numeric_columns].map(
    lambda column: column.notna().all()
).all():
    print(
        "⚠️ Для некоторых компонентов SEM отсутствует "
        "в исходном отчёте MMPBSA."
    )

if not summary[
    ["Mean (kcal/mol)", "SD"]
].map(
    lambda column: column.map(pd.notna).all()
).all():
    raise ValueError(
        "В энергетических результатах обнаружены "
        "пропущенные значения Mean или SD."
    )


# ---------------------------------------------------------------------------
# Сохранение CSV и LaTeX
# ---------------------------------------------------------------------------

summary_csv = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_MMGBSA_summary.csv"
)

summary_tex = (
    WORKDIR
    / f"{SYSTEM_BASENAME}_MMGBSA_summary.tex"
)

summary.to_csv(
    summary_csv,
    float_format="%.6f",
)

summary_tex.write_text(
    summary.to_latex(
        escape=False,
        float_format=lambda value: f"{value:.3f}",
    ),
    encoding="utf-8",
)

print("\n💾 Результаты сохранены:")
print(f"  • CSV: {summary_csv.resolve()}")
print(f"  • LaTeX: {summary_tex.resolve()}")


# ---------------------------------------------------------------------------
# Итоговая свободная энергия связывания
# ---------------------------------------------------------------------------

BINDING_ENERGY_KEY = (
    "ΔG_bind (Total Binding Free Energy)"
)

if BINDING_ENERGY_KEY not in summary.index:
    raise ValueError(
        "В результатах отсутствует компонент TOTAL "
        "для расчёта итоговой свободной энергии связывания."
    )

binding_energy = summary.loc[
    BINDING_ENERGY_KEY,
    "Mean (kcal/mol)",
]

binding_sd = summary.loc[
    BINDING_ENERGY_KEY,
    "SD",
]

binding_sem = summary.loc[
    BINDING_ENERGY_KEY,
    "SEM",
]

print("\n🎯 Итоговая свободная энергия связывания:")
print(
    f"  ΔG_bind = "
    f"{binding_energy:.2f} ± {binding_sd:.2f} kcal/mol"
)

if pd.notna(binding_sem):
    print(f"  SEM = {binding_sem:.2f} kcal/mol")
else:
    print("  SEM = не указан в исходном отчёте MMPBSA.")


print("\n" + "=" * 72)
print("✅ Обработка MM/GBSA завершена")
print("=" * 72)

In [ ]:
# @title 📊 26. Сводная визуализация и финализация MD-анализа ✅

import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde


SYSTEM_NAME = str(
    globals().get("SYSTEM_BASENAME", "SYSTEM")
)
WORK_DIR = Path(
    globals().get("WORKDIR", Path.cwd())
)
FIGURES_DIR = WORK_DIR / "analysis_figures"
FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROD_DCD_INTERVAL_PS = float(
    globals().get("PROD_DCD_PS", 5.0)
)

ANALYSIS_STRIDE = 5
ANALYSIS_CHUNK_SIZE = 10
MAX_KDE_POINTS = 20_000

print(
    f"--- Сводная визуализация и финализация анализа: "
    f"{SYSTEM_NAME} ---"
)


# ---------------------------------------------------------------------------
# Вспомогательные функции
# ---------------------------------------------------------------------------

def save_figure(
    figure: plt.Figure,
    stem: str,
) -> tuple[Path, Path]:
    """Сохраняет график в PNG 300 DPI и SVG."""
    filename = f"{SYSTEM_NAME}_{stem}"
    png_path = FIGURES_DIR / f"{filename}.png"
    svg_path = FIGURES_DIR / f"{filename}.svg"

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        svg_path,
        bbox_inches="tight",
    )

    print(f"📊 Сохранен график: {png_path.name}")

    return png_path, svg_path


def configure_axis(
    axis: plt.Axes,
    *,
    title: str,
    xlabel: str | None = None,
    ylabel: str | None = None,
) -> None:
    """Единообразно настраивает оси графика."""
    axis.set_title(
        title,
        fontsize=12,
        fontweight="bold",
    )

    if xlabel is not None:
        axis.set_xlabel(xlabel)

    if ylabel is not None:
        axis.set_ylabel(ylabel)

    axis.grid(
        linestyle="-",
        linewidth=0.5,
        alpha=0.7,
    )


def dcd_sort_key(path: Path) -> tuple[int, str]:
    """Сортирует DCD-файлы по номеру части."""
    match = re.search(
        r"(\d+)(?=\.dcd$)",
        path.name,
    )

    if match is None:
        return (0, path.name)

    return (
        int(match.group(1)),
        path.name,
    )


def validate_dcd(
    path: Path,
    expected_atoms: int,
) -> int:
    """Проверяет DCD и возвращает число кадров."""
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"DCD-файл отсутствует или пуст: {path}"
        )

    with md.formats.DCDTrajectoryFile(str(path)) as dcd:
        coordinates, *_ = dcd.read()

    if coordinates.ndim != 3:
        raise ValueError(
            f"Некорректная размерность DCD: {path}"
        )

    if coordinates.shape[1] != expected_atoms:
        raise ValueError(
            f"Число атомов в {path.name}: "
            f"{coordinates.shape[1]}, "
            f"ожидалось {expected_atoms}."
        )

    return coordinates.shape[0]


def parse_number(value: str) -> float:
    """Преобразует обычную или Fortran-запись числа."""
    return float(
        value.replace("D", "E").replace("d", "E")
    )


def parse_mmpbsa_results(
    path: Path | None,
) -> dict[str, tuple[float, float]] | None:
    """Извлекает средние значения и SD из отчёта MMPBSA."""
    if path is None or not path.is_file():
        return None

    energy_names = {
        "VDWAALS": "ΔE_vdw",
        "EEL": "ΔE_elec",
        "EGB": "ΔG_GB",
        "ESURF": "ΔG_np",
        "GGAS": "ΔG_gas",
        "GSOLV": "ΔG_solv",
        "TOTAL": "ΔG_bind",
    }

    number_pattern = (
        r"[-+]?"
        r"(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[EeDd][-+]?\d+)?"
    )

    pattern = re.compile(
        rf"^\s*\*?\s*"
        rf"(?:DELTA\s+)?"
        rf"(?P<term>[A-Z0-9_]+)"
        rf"\s+"
        rf"(?P<mean>{number_pattern})"
        rf"\s+"
        rf"(?P<error>{number_pattern})"
    )

    results = {}

    for line in path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines():
        match = pattern.match(line)

        if match is None:
            continue

        raw_term = match.group("term")

        if raw_term not in energy_names:
            continue

        results[energy_names[raw_term]] = (
            parse_number(match.group("mean")),
            parse_number(match.group("error")),
        )

    return results or None


def find_mmpbsa_results() -> Path | None:
    """Находит итоговый файл MM/GBSA."""
    candidates = (
        WORK_DIR / f"{SYSTEM_NAME}_MMPBSA_RESULTS.dat",
        WORK_DIR / "FINAL_RESULTS_MMPBSA.dat",
    )

    for path in candidates:
        if path.is_file() and path.stat().st_size > 0:
            return path

    return None


def select_ligand_heavy_atoms(
    topology: md.Topology,
) -> np.ndarray:
    """Возвращает индексы тяжёлых атомов лиганда."""
    ligand_kind = str(
        globals().get(
            "LIGAND_KIND",
            "small_molecule",
        )
    ).strip().lower()

    small_molecule_resname = str(
        globals().get(
            "SMALL_MOL_RESNAME",
            "LIG",
        )
    ).strip().upper()

    modified_na_resname = str(
        globals().get(
            "MODIFIED_NA_RESNAME",
            "6MA",
        )
    ).strip().upper()

    if ligand_kind == "nucleic_acid":
        ligand_resnames = {
            "A",
            "C",
            "G",
            "U",
            "DA",
            "DC",
            "DG",
            "DT",
            "RA",
            "RC",
            "RG",
            "RU",
            "DA5",
            "DG5",
            "DT3",
            "DC3",
            "A5",
            "C3",
            "G5",
            modified_na_resname,
        }
    elif ligand_kind == "small_molecule":
        ligand_resnames = {
            small_molecule_resname,
        }
    else:
        return np.array([], dtype=int)

    return np.array(
        [
            atom.index
            for atom in topology.atoms
            if (
                atom.residue.name.strip().upper()
                in ligand_resnames
                and atom.element is not None
                and atom.element.symbol.upper() != "H"
            )
        ],
        dtype=int,
    )


def calculate_com(
    xyz: np.ndarray,
    atom_indices: np.ndarray,
    masses: np.ndarray,
) -> np.ndarray:
    """Вычисляет центр масс для каждого кадра."""
    selected_xyz = xyz[:, atom_indices, :]

    return np.sum(
        selected_xyz * masses[None, :, None],
        axis=1,
    ) / np.sum(masses)


def calculate_rmsd(
    coordinates: np.ndarray,
    reference: np.ndarray,
    atom_indices: np.ndarray,
) -> np.ndarray:
    """Вычисляет RMSD выбранных атомов относительно reference."""
    differences = (
        coordinates[:, atom_indices, :]
        - reference[atom_indices][None, :, :]
    )

    return np.sqrt(
        np.mean(
            np.sum(differences**2, axis=2),
            axis=1,
        )
    )


# ---------------------------------------------------------------------------
# Поиск Production DCD
# ---------------------------------------------------------------------------

system_prmtop = globals().get("SYSTEM_PRMTOP")

if system_prmtop is None:
    raise FileNotFoundError(
        "SYSTEM_PRMTOP не найден."
    )

system_prmtop = Path(system_prmtop)

if not system_prmtop.is_file():
    raise FileNotFoundError(
        f"PRMTOP не найден: {system_prmtop}"
    )

dcd_files = sorted(
    WORK_DIR.glob(
        f"{globals().get('PROD_JOB', 'prod')}*.dcd"
    ),
    key=dcd_sort_key,
)

if not dcd_files:
    raise FileNotFoundError(
        "DCD-файлы продукционной MD не найдены."
    )


# ---------------------------------------------------------------------------
# Инициализация topology и reference frame
# ---------------------------------------------------------------------------

topology = md.load_frame(
    str(dcd_files[0]),
    0,
    top=str(system_prmtop),
).topology

n_atoms = topology.n_atoms

frame_counts = [
    validate_dcd(
        path,
        expected_atoms=n_atoms,
    )
    for path in dcd_files
]

total_frames = sum(frame_counts)

if total_frames == 0:
    raise ValueError(
        "Production-траектория не содержит кадров."
    )

print(
    f"📦 DCD-файлов: {len(dcd_files)}"
)
print(
    f"📊 Всего кадров: {total_frames}"
)
print(
    f"🔬 Атомов: {n_atoms}"
)


reference = md.load_frame(
    str(dcd_files[0]),
    0,
    top=str(system_prmtop),
)

protein_ca = topology.select(
    "protein and name CA"
)

protein_atoms = topology.select(
    "protein"
)

if len(protein_ca) == 0:
    raise RuntimeError(
        "В топологии не найдены Cα-атомы белка."
    )

if len(protein_atoms) == 0:
    raise RuntimeError(
        "В топологии не найдены атомы белка."
    )

ligand_heavy = select_ligand_heavy_atoms(
    topology
)

if len(ligand_heavy) == 0:
    print(
        "⚠️ Тяжёлые атомы лиганда не найдены."
    )

protein_masses = np.array(
    [
        atom.element.mass
        for atom in topology.atoms
        if atom.index in set(protein_atoms)
    ],
    dtype=float,
)

ligand_masses = np.array(
    [
        atom.element.mass
        for atom in topology.atoms
        if atom.index in set(ligand_heavy)
    ],
    dtype=float,
)

protein_mass_indices = np.array(
    [
        atom.index
        for atom in topology.atoms
        if atom.index in set(protein_atoms)
    ],
    dtype=int,
)

ligand_mass_indices = np.array(
    [
        atom.index
        for atom in topology.atoms
        if atom.index in set(ligand_heavy)
    ],
    dtype=int,
)


# ---------------------------------------------------------------------------
# Потоковый расчёт основных MD-метрик
# ---------------------------------------------------------------------------

time_values = []
rmsd_protein_values = []
rmsd_ligand_values = []
ligand_protein_com_values = []

coordinate_sum = np.zeros(
    (len(protein_ca), 3),
    dtype=np.float64,
)
coordinate_squared_sum = np.zeros(
    (len(protein_ca), 3),
    dtype=np.float64,
)

processed_frames = 0

reference_coordinates = reference.xyz[0]

for dcd_path in dcd_files:
    for chunk in md.iterload(
        str(dcd_path),
        top=str(system_prmtop),
        chunk=ANALYSIS_CHUNK_SIZE,
        stride=ANALYSIS_STRIDE,
    ):
        chunk.superpose(
            reference,
            0,
            atom_indices=protein_ca,
        )

        chunk_coordinates = chunk.xyz

        chunk_size = chunk.n_frames

        frame_indices = (
            processed_frames
            + np.arange(chunk_size)
            * ANALYSIS_STRIDE
        )

        time_values.extend(
            frame_indices
            * PROD_DCD_INTERVAL_PS
            / 1000.0
        )

        protein_rmsd = calculate_rmsd(
            chunk_coordinates,
            reference_coordinates,
            protein_ca,
        )

        rmsd_protein_values.extend(
            protein_rmsd * 10.0
        )

        aligned_ca = chunk_coordinates[
            :,
            protein_ca,
            :,
        ]

        coordinate_sum += np.sum(
            aligned_ca,
            axis=0,
        )

        coordinate_squared_sum += np.sum(
            aligned_ca**2,
            axis=0,
        )

        if len(ligand_heavy) > 0:
            ligand_rmsd = calculate_rmsd(
                chunk_coordinates,
                reference_coordinates,
                ligand_heavy,
            )

            rmsd_ligand_values.extend(
                ligand_rmsd * 10.0
            )

            ligand_com = calculate_com(
                chunk_coordinates,
                ligand_mass_indices,
                ligand_masses,
            )

            protein_com = calculate_com(
                chunk_coordinates,
                protein_mass_indices,
                protein_masses,
            )

            com_distance = np.linalg.norm(
                ligand_com - protein_com,
                axis=1,
            )

            ligand_protein_com_values.extend(
                com_distance * 10.0
            )

        processed_frames += (
            chunk_size * ANALYSIS_STRIDE
        )

        del chunk
        gc.collect()


time_ns = np.asarray(
    time_values,
    dtype=float,
)

rmsd_protein = np.asarray(
    rmsd_protein_values,
    dtype=float,
)

rmsd_ligand = (
    np.asarray(
        rmsd_ligand_values,
        dtype=float,
    )
    if ligand_heavy.size > 0
    else None
)

ligand_protein_com_distance = (
    np.asarray(
        ligand_protein_com_values,
        dtype=float,
    )
    if ligand_heavy.size > 0
    else None
)

n_analyzed_frames = len(rmsd_protein)

if n_analyzed_frames == 0:
    raise RuntimeError(
        "После потокового анализа не получено ни одного кадра."
    )


# ---------------------------------------------------------------------------
# Protein Cα RMSF
# ---------------------------------------------------------------------------

mean_coordinates = (
    coordinate_sum
    / n_analyzed_frames
)

mean_squared_coordinates = (
    coordinate_squared_sum
    / n_analyzed_frames
)

variance = (
    mean_squared_coordinates
    - mean_coordinates**2
)

variance = np.maximum(
    variance,
    0.0,
)

rmsf_protein = np.sqrt(
    np.sum(
        variance,
        axis=1,
    )
) * 10.0

residue_numbers = np.array(
    [
        atom.residue.resSeq
        for atom in topology.atoms
        if atom.index in set(protein_ca)
    ],
    dtype=int,
)


# ---------------------------------------------------------------------------
# Проверка конечности результатов
# ---------------------------------------------------------------------------

metric_arrays = {
    "Protein RMSD": rmsd_protein,
    "Protein RMSF": rmsf_protein,
}

if rmsd_ligand is not None:
    metric_arrays["Ligand RMSD"] = rmsd_ligand

if ligand_protein_com_distance is not None:
    metric_arrays["Ligand–Protein COM"] = (
        ligand_protein_com_distance
    )

for metric_name, values in metric_arrays.items():
    if not np.isfinite(values).all():
        raise ValueError(
            f"В метрике {metric_name} обнаружены NaN/Inf."
        )


print(
    f"✅ Проанализировано кадров: "
    f"{n_analyzed_frames}"
)


# ---------------------------------------------------------------------------
# MM/GBSA
# ---------------------------------------------------------------------------

mmpbsa_file = find_mmpbsa_results()
mmpbsa_results = parse_mmpbsa_results(
    mmpbsa_file
)

if mmpbsa_file is not None:
    print(
        f"📄 MM/GBSA: {mmpbsa_file.name}"
    )


# ---------------------------------------------------------------------------
# Сводный график 2 × 2
# ---------------------------------------------------------------------------

figure, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    constrained_layout=True,
)

axes = axes.ravel()


# Protein RMSD
axes[0].plot(
    time_ns,
    rmsd_protein,
    linewidth=1.5,
)

configure_axis(
    axes[0],
    title="Protein Cα RMSD",
    xlabel="Time (ns)",
    ylabel="RMSD (Å)",
)

axes[0].set_xlim(left=0)


# Ligand RMSD
if rmsd_ligand is not None:
    axes[1].plot(
        time_ns,
        rmsd_ligand,
        linewidth=1.5,
    )

    configure_axis(
        axes[1],
        title="Ligand Heavy-Atom RMSD",
        xlabel="Time (ns)",
        ylabel="RMSD (Å)",
    )

    axes[1].set_xlim(left=0)

else:
    axes[1].text(
        0.5,
        0.5,
        "Ligand heavy atoms not found",
        ha="center",
        va="center",
    )
    axes[1].set_axis_off()


# Protein RMSF
axes[2].plot(
    residue_numbers,
    rmsf_protein,
    linewidth=1.5,
)

configure_axis(
    axes[2],
    title="Protein Cα RMSF",
    xlabel="Residue Index",
    ylabel="RMSF (Å)",
)

axes[2].set_xlim(
    left=min(1, int(residue_numbers.min()))
)


# MM/GBSA
if (
    mmpbsa_results
    and "ΔG_bind" in mmpbsa_results
):
    energy_order = (
        "ΔE_vdw",
        "ΔE_elec",
        "ΔG_GB",
        "ΔG_np",
        "ΔG_gas",
        "ΔG_solv",
        "ΔG_bind",
    )

    energy_terms = [
        term
        for term in energy_order
        if term in mmpbsa_results
    ]

    energy_values = [
        mmpbsa_results[term][0]
        for term in energy_terms
    ]

    energy_errors = [
        mmpbsa_results[term][1]
        for term in energy_terms
    ]

    axes[3].bar(
        energy_terms,
        energy_values,
        yerr=energy_errors,
        capsize=4,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.8,
    )

    axes[3].axhline(
        0,
        linewidth=0.8,
        linestyle="--",
    )

    configure_axis(
        axes[3],
        title="MM/GBSA Energy Components",
        ylabel="Energy (kcal/mol)",
    )

    axes[3].tick_params(
        axis="x",
        rotation=35,
    )

else:
    if ligand_protein_com_distance is not None:
        axes[3].plot(
            time_ns,
            ligand_protein_com_distance,
            linewidth=1.5,
        )

        configure_axis(
            axes[3],
            title="Ligand–Protein COM Distance",
            xlabel="Time (ns)",
            ylabel="Distance (Å)",
        )

        axes[3].set_xlim(left=0)

    else:
        axes[3].text(
            0.5,
            0.5,
            "No MM/GBSA / Ligand COM Metric",
            ha="center",
            va="center",
        )
        axes[3].set_axis_off()


save_figure(
    figure,
    "md_2x2_summary",
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------------------------
# Protein Cα RMSD
# ---------------------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(7, 4)
)

axis.plot(
    time_ns,
    rmsd_protein,
    linewidth=1.5,
)

configure_axis(
    axis,
    title=f"Protein Cα RMSD ({SYSTEM_NAME})",
    xlabel="Time (ns)",
    ylabel="RMSD (Å)",
)

axis.set_xlim(left=0)

save_figure(
    figure,
    "protein_ca_rmsd",
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------------------------
# Protein Cα RMSD distribution
# ---------------------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(6, 4)
)

axis.hist(
    rmsd_protein,
    bins=30,
    density=True,
    alpha=0.5,
    edgecolor="black",
)

kde_values = rmsd_protein

if len(kde_values) > MAX_KDE_POINTS:
    kde_indices = np.linspace(
        0,
        len(kde_values) - 1,
        MAX_KDE_POINTS,
        dtype=int,
    )
    kde_values = kde_values[kde_indices]

if (
    len(kde_values) > 1
    and np.std(kde_values) > 0
):
    x_grid = np.linspace(
        np.min(kde_values),
        np.max(kde_values),
        400,
    )

    kde = gaussian_kde(kde_values)

    axis.plot(
        x_grid,
        kde(x_grid),
        linewidth=2,
        label="KDE",
    )

    axis.legend()

configure_axis(
    axis,
    title=(
        f"Protein Cα RMSD Distribution "
        f"({SYSTEM_NAME})"
    ),
    xlabel="RMSD (Å)",
    ylabel="Probability Density",
)

save_figure(
    figure,
    "protein_ca_rmsd_distribution",
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------------------------
# Ligand RMSD
# ---------------------------------------------------------------------------

if rmsd_ligand is not None:
    figure, axis = plt.subplots(
        figsize=(7, 4)
    )

    axis.plot(
        time_ns,
        rmsd_ligand,
        linewidth=1.5,
    )

    configure_axis(
        axis,
        title=(
            f"Ligand Heavy-Atom RMSD "
            f"({SYSTEM_NAME})"
        ),
        xlabel="Time (ns)",
        ylabel="RMSD (Å)",
    )

    axis.set_xlim(left=0)

    save_figure(
        figure,
        "ligand_rmsd",
    )

    plt.show()
    plt.close(figure)


# ---------------------------------------------------------------------------
# Protein RMSF
# ---------------------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(8, 4)
)

axis.plot(
    residue_numbers,
    rmsf_protein,
    linewidth=1.5,
)

configure_axis(
    axis,
    title=f"Protein Cα RMSF ({SYSTEM_NAME})",
    xlabel="Residue Index",
    ylabel="RMSF (Å)",
)

axis.set_xlim(
    left=min(1, int(residue_numbers.min()))
)

save_figure(
    figure,
    "protein_ca_rmsf",
)

plt.show()
plt.close(figure)


# ---------------------------------------------------------------------------
# Jaccard heatmaps и docking grid
# ---------------------------------------------------------------------------

summary_grid_rows = []

consensus_root = (
    WORK_DIR / "consensus_maps"
)

if not consensus_root.is_dir():
    print(
        "ℹ️ consensus_maps не найдена — "
        "дополнительные docking-графики пропущены."
    )

else:
    tables_dir = consensus_root / "tables"

    if tables_dir.is_dir():
        jaccard_files = sorted(
            tables_dir.glob(
                "*_jaccard_matrix.tsv"
            )
        )

        for jaccard_path in jaccard_files:
            jaccard_df = pd.read_csv(
                jaccard_path,
                sep="\t",
            )

            required_columns = {
                "pocket_1",
                "pocket_2",
                "jaccard",
            }

            if (
                jaccard_df.empty
                or not required_columns.issubset(
                    jaccard_df.columns
                )
            ):
                continue

            matrix = (
                jaccard_df
                .pivot(
                    index="pocket_1",
                    columns="pocket_2",
                    values="jaccard",
                )
                .fillna(0.0)
            )

            figure, axis = plt.subplots(
                figsize=(
                    max(
                        6,
                        0.45 * matrix.shape[1],
                    ),
                    max(
                        5,
                        0.45 * matrix.shape[0],
                    ),
                )
            )

            image = axis.imshow(
                matrix.values,
                vmin=0.0,
                vmax=1.0,
            )

            axis.set_xticks(
                np.arange(
                    matrix.shape[1]
                )
            )

            axis.set_xticklabels(
                matrix.columns,
                rotation=90,
                fontsize=8,
            )

            axis.set_yticks(
                np.arange(
                    matrix.shape[0]
                )
            )

            axis.set_yticklabels(
                matrix.index,
                fontsize=8,
            )

            stem_name = (
                jaccard_path.stem
                .replace(
                    "_jaccard_matrix",
                    "",
                )
            )

            axis.set_title(
                f"Jaccard Heatmap — {stem_name}",
                fontsize=11,
                fontweight="bold",
            )

            colorbar = figure.colorbar(
                image,
                ax=axis,
                fraction=0.046,
                pad=0.04,
            )

            colorbar.set_label(
                "Jaccard Index"
            )

            figure.tight_layout()

            save_figure(
                figure,
                f"jaccard_{stem_name}",
            )

            plt.show()
            plt.close(figure)


        # Docking grid overview
        grid_summary_path = (
            tables_dir
            / "consensus_global_summary.tsv"
        )

        if grid_summary_path.is_file():
            grid_df = pd.read_csv(
                grid_summary_path,
                sep="\t",
            )

            required_grid_columns = {
                "grid_center_x",
                "grid_center_y",
                "grid_size_x",
                "grid_size_y",
            }

            if (
                not grid_df.empty
                and required_grid_columns.issubset(
                    grid_df.columns
                )
            ):
                figure, axis = plt.subplots(
                    figsize=(7, 6)
                )

                for _, row in grid_df.iterrows():
                    try:
                        center_x = float(
                            row["grid_center_x"]
                        )
                        center_y = float(
                            row["grid_center_y"]
                        )
                        size_x = float(
                            row["grid_size_x"]
                        )
                        size_y = float(
                            row["grid_size_y"]
                        )
                    except (
                        TypeError,
                        ValueError,
                    ):
                        continue

                    rectangle = Rectangle(
                        (
                            center_x - size_x / 2.0,
                            center_y - size_y / 2.0,
                        ),
                        size_x,
                        size_y,
                        fill=False,
                        linewidth=1.5,
                    )

                    axis.add_patch(
                        rectangle
                    )

                    axis.scatter(
                        [center_x],
                        [center_y],
                        s=30,
                    )

                    target_name = str(
                        row.get(
                            "target_input",
                            "",
                        )
                    )

                    axis.text(
                        center_x,
                        center_y,
                        f" {target_name}",
                        fontsize=8,
                        ha="left",
                        va="bottom",
                    )

                    try:
                        center_z = float(
                            row.get(
                                "grid_center_z",
                                np.nan,
                            )
                        )
                    except (
                        TypeError,
                        ValueError,
                    ):
                        center_z = np.nan

                    try:
                        size_z = float(
                            row.get(
                                "grid_size_z",
                                np.nan,
                            )
                        )
                    except (
                        TypeError,
                        ValueError,
                    ):
                        size_z = np.nan

                    summary_grid_rows.append(
                        {
                            "target_input": target_name,
                            "grid_center_x": center_x,
                            "grid_center_y": center_y,
                            "grid_center_z": center_z,
                            "grid_size_x": size_x,
                            "grid_size_y": size_y,
                            "grid_size_z": size_z,
                            "n_final_consensus_residues": (
                                row.get(
                                    "n_final_consensus_residues",
                                    np.nan,
                                )
                            ),
                        }
                    )

                configure_axis(
                    axis,
                    title=(
                        "Docking Grid Overview "
                        "(X–Y Projection)"
                    ),
                    xlabel="Grid Center X (Å)",
                    ylabel="Grid Center Y (Å)",
                )

                axis.set_aspect(
                    "equal",
                    adjustable="datalim",
                )

                save_figure(
                    figure,
                    "grid_overview_xy",
                )

                plt.show()
                plt.close(figure)


# ---------------------------------------------------------------------------
# Сводная таблица метрик
# ---------------------------------------------------------------------------

summary_data = {
    "system": SYSTEM_NAME,
    "n_frames_total": int(total_frames),
    "n_frames_analyzed": int(n_analyzed_frames),
    "traj_time_ns": float(time_ns[-1]),
    "analysis_stride": int(ANALYSIS_STRIDE),
    "rmsd_prot_mean_A": float(
        np.mean(rmsd_protein)
    ),
    "rmsd_prot_max_A": float(
        np.max(rmsd_protein)
    ),
    "rmsf_prot_mean_A": float(
        np.mean(rmsf_protein)
    ),
    "rmsf_prot_max_A": float(
        np.max(rmsf_protein)
    ),
    "ligand_rmsd_mean_A": (
        float(np.mean(rmsd_ligand))
        if rmsd_ligand is not None
        else np.nan
    ),
    "ligand_rmsd_max_A": (
        float(np.max(rmsd_ligand))
        if rmsd_ligand is not None
        else np.nan
    ),
    "ligand_protein_com_mean_A": (
        float(
            np.mean(
                ligand_protein_com_distance
            )
        )
        if ligand_protein_com_distance is not None
        else np.nan
    ),
    "ligand_protein_com_max_A": (
        float(
            np.max(
                ligand_protein_com_distance
            )
        )
        if ligand_protein_com_distance is not None
        else np.nan
    ),
    "mmpbsa_delta_g_bind": (
        mmpbsa_results["ΔG_bind"][0]
        if (
            mmpbsa_results
            and "ΔG_bind" in mmpbsa_results
        )
        else np.nan
    ),
    "mmpbsa_delta_g_err": (
        mmpbsa_results["ΔG_bind"][1]
        if (
            mmpbsa_results
            and "ΔG_bind" in mmpbsa_results
        )
        else np.nan
    ),
}

summary_df = pd.DataFrame(
    [summary_data]
)

summary_csv = (
    FIGURES_DIR
    / f"{SYSTEM_NAME}_summary_metrics.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False,
)

if summary_grid_rows:
    grid_summary_csv = (
        FIGURES_DIR
        / f"{SYSTEM_NAME}_grid_overview_summary.csv"
    )

    pd.DataFrame(
        summary_grid_rows
    ).to_csv(
        grid_summary_csv,
        index=False,
    )

    print(
        f"📁 Сохранен сводный файл сетки: "
        f"{grid_summary_csv.name}"
    )

print(
    f"📁 Сохранен сводный CSV: "
    f"{summary_csv.name}"
)

print(
    f"🟢 Анализ завершён. Все результаты: "
    f"{FIGURES_DIR.resolve()}"
)


# ---------------------------------------------------------------------------
# Очистка
# ---------------------------------------------------------------------------

del topology
del reference
del coordinate_sum
del coordinate_squared_sum
gc.collect()

In [ ]:
# @title 📊 27. Сводный анализ docking consensus и целевых метрик ✅

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle


# ---------------------------------------------------------------------------
# Пути
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    globals().get(
        "PROJECT_ROOT",
        globals().get(
            "WORKDIR",
            Path.cwd(),
        ),
    )
).resolve()

WORK_DIR = Path(
    globals().get(
        "WORKDIR",
        PROJECT_ROOT,
    )
).resolve()

ANALYSIS_DIR = WORK_DIR / "analysis_figures"
ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------------
# Вспомогательные функции работы с файлами
# ---------------------------------------------------------------------------

def unique_existing_paths(
    paths: list[Path],
) -> list[Path]:
    """Удаляет дубликаты и оставляет существующие директории."""
    result = []
    seen = set()

    for path in paths:
        path = path.resolve()

        if not path.is_dir():
            continue

        if path in seen:
            continue

        seen.add(path)
        result.append(path)

    return result


SEARCH_ROOTS = unique_existing_paths(
    [
        WORK_DIR,
        PROJECT_ROOT,
        PROJECT_ROOT.parent,
    ]
)


def find_first(
    pattern: str,
) -> Path | None:
    """Находит первый существующий файл по шаблону."""
    for root in SEARCH_ROOTS:
        matches = sorted(
            path
            for path in root.rglob(pattern)
            if path.is_file()
        )

        if matches:
            return matches[0]

    return None


def find_all(
    pattern: str,
) -> list[Path]:
    """Находит все файлы по шаблону без дубликатов."""
    matches = set()

    for root in SEARCH_ROOTS:
        matches.update(
            path
            for path in root.rglob(pattern)
            if path.is_file()
        )

    return sorted(matches)


def load_tsv(
    path: Path | None,
) -> pd.DataFrame:
    """Безопасно загружает TSV-файл."""
    if path is None or not path.is_file():
        return pd.DataFrame()

    try:
        return pd.read_csv(
            path,
            sep="\t",
        )

    except (
        OSError,
        pd.errors.ParserError,
        UnicodeDecodeError,
    ) as exc:
        print(
            f"⚠️ Не удалось прочитать "
            f"{path.name}: {exc}"
        )

        return pd.DataFrame()


def save_figure(
    figure: plt.Figure,
    stem: str,
) -> tuple[Path, Path]:
    """Сохраняет график в PNG 300 DPI и SVG."""
    png_path = ANALYSIS_DIR / f"{stem}.png"
    svg_path = ANALYSIS_DIR / f"{stem}.svg"

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        svg_path,
        bbox_inches="tight",
    )

    print(
        f"📊 Сохранен график: "
        f"{png_path.name}"
    )

    return png_path, svg_path


def numeric_series(
    dataframe: pd.DataFrame,
    column: str,
) -> pd.Series:
    """Преобразует столбец в числовой формат."""
    if column not in dataframe.columns:
        return pd.Series(
            np.nan,
            index=dataframe.index,
            dtype=float,
        )

    return pd.to_numeric(
        dataframe[column],
        errors="coerce",
    )


def matrix_from_long(
    dataframe: pd.DataFrame,
    row_column: str,
    column_column: str,
    value_column: str,
) -> pd.DataFrame | None:
    """Строит матрицу из long-format таблицы."""
    required_columns = {
        row_column,
        column_column,
        value_column,
    }

    if (
        dataframe.empty
        or not required_columns.issubset(
            dataframe.columns
        )
    ):
        return None

    matrix = dataframe.pivot_table(
        index=row_column,
        columns=column_column,
        values=value_column,
        aggfunc="mean",
    )

    if matrix.empty:
        return None

    return matrix.sort_index().sort_index(
        axis=1
    )


def plot_heatmap(
    matrix: pd.DataFrame | None,
    title: str,
    stem: str,
    *,
    vmin: float | None = None,
    vmax: float | None = None,
    center: float | None = None,
    colorbar_label: str = "",
) -> None:
    """Строит и сохраняет heatmap."""
    if matrix is None or matrix.empty:
        return

    values = matrix.to_numpy(
        dtype=float
    )

    finite_values = values[
        np.isfinite(values)
    ]

    if finite_values.size == 0:
        return

    figure_width = max(
        7.0,
        0.45 * matrix.shape[1] + 3.0,
    )

    figure_height = max(
        5.0,
        0.35 * matrix.shape[0] + 2.5,
    )

    figure, axis = plt.subplots(
        figsize=(
            figure_width,
            figure_height,
        )
    )

    image = axis.imshow(
        values,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
    )

    axis.set_title(
        title,
        fontsize=12,
        fontweight="bold",
        pad=12,
    )

    axis.set_xticks(
        np.arange(
            matrix.shape[1]
        )
    )

    axis.set_xticklabels(
        matrix.columns,
        rotation=90,
        fontsize=8,
    )

    axis.set_yticks(
        np.arange(
            matrix.shape[0]
        )
    )

    axis.set_yticklabels(
        matrix.index,
        fontsize=8,
    )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.046,
        pad=0.04,
    )

    if colorbar_label:
        colorbar.set_label(
            colorbar_label
        )

    figure.tight_layout()

    save_figure(
        figure,
        stem,
    )

    plt.show()
    plt.close(figure)


# ---------------------------------------------------------------------------
# Поиск исходных таблиц
# ---------------------------------------------------------------------------

CONSENSUS_SUMMARY_PATH = find_first(
    "consensus_global_summary.tsv"
)

BEST_RESULTS_PATH = find_first(
    "best_per_target.tsv"
)

MD_TOP3_PATH = find_first(
    "md_candidates_top3_per_target.tsv"
)

RANKED_RESULTS_PATH = find_first(
    "ranked_all_successful.tsv"
)

consensus_summary = load_tsv(
    CONSENSUS_SUMMARY_PATH
)

best_results = load_tsv(
    BEST_RESULTS_PATH
)

md_top3 = load_tsv(
    MD_TOP3_PATH
)

ranked_results = load_tsv(
    RANKED_RESULTS_PATH
)

pocket_tables = find_all(
    "*_pocket_summary.tsv"
)

jaccard_tables = find_all(
    "*_jaccard_matrix.tsv"
)

print(
    "📂 Загруженные таблицы:"
)

print(
    f"  • consensus: "
    f"{len(consensus_summary)} строк"
)

print(
    f"  • best: "
    f"{len(best_results)} строк"
)

print(
    f"  • MD top-3: "
    f"{len(md_top3)} строк"
)

print(
    f"  • ranked: "
    f"{len(ranked_results)} строк"
)

print(
    f"  • pocket tables: "
    f"{len(pocket_tables)}"
)

print(
    f"  • Jaccard tables: "
    f"{len(jaccard_tables)}"
)


# ---------------------------------------------------------------------------
# Объединение target-level метрик
# ---------------------------------------------------------------------------

combined = consensus_summary.copy()

if not combined.empty:
    if "target_input" not in combined.columns:
        raise ValueError(
            "В consensus_global_summary.tsv "
            "отсутствует столбец target_input."
        )

    if "methods_used" in combined.columns:
        combined["n_methods_used"] = (
            combined["methods_used"]
            .fillna("")
            .astype(str)
            .map(
                lambda value: len(
                    [
                        item
                        for item in value.split(",")
                        if item.strip()
                    ]
                )
            )
        )
    else:
        combined["n_methods_used"] = np.nan

    grid_columns = (
        "grid_center_x",
        "grid_center_y",
        "grid_center_z",
        "grid_size_x",
        "grid_size_y",
        "grid_size_z",
        "n_final_consensus_residues",
    )

    for column in grid_columns:
        if column not in combined.columns:
            combined[column] = np.nan

    grid_x = numeric_series(
        combined,
        "grid_size_x",
    )

    grid_y = numeric_series(
        combined,
        "grid_size_y",
    )

    grid_z = numeric_series(
        combined,
        "grid_size_z",
    )

    combined["grid_volume_A3"] = (
        grid_x
        * grid_y
        * grid_z
    )

    combined["grid_diag_A"] = np.sqrt(
        grid_x.pow(2)
        + grid_y.pow(2)
        + grid_z.pow(2)
    )

    consensus_residues = numeric_series(
        combined,
        "n_final_consensus_residues",
    )

    combined["consensus_density"] = (
        consensus_residues
        / combined["grid_volume_A3"].replace(
            0,
            np.nan,
        )
    )


# ---------------------------------------------------------------------------
# Статистика всех успешных docking results
# ---------------------------------------------------------------------------

if (
    not combined.empty
    and not ranked_results.empty
    and "target_input" in ranked_results.columns
):
    ranked_numeric = ranked_results.copy()

    for column in (
        "CNNscore",
        "CNNaffinity",
        "minimizedAffinity",
    ):
        if column in ranked_numeric.columns:
            ranked_numeric[column] = pd.to_numeric(
                ranked_numeric[column],
                errors="coerce",
            )

    aggregation = {
        "global_job_index": "size",
    }

    if "CNNscore" in ranked_numeric.columns:
        aggregation["CNNscore"] = [
            "mean",
            "max",
        ]

    if "minimizedAffinity" in ranked_numeric.columns:
        aggregation["minimizedAffinity"] = [
            "mean",
            "min",
        ]

    ranked_agg = (
        ranked_numeric
        .groupby("target_input")
        .agg(aggregation)
        .reset_index()
    )

    flattened_columns = []

    for column in ranked_agg.columns:
        if isinstance(column, tuple):
            metric, statistic = column

            if statistic:
                flattened_columns.append(
                    f"{metric}_{statistic}"
                )
            else:
                flattened_columns.append(
                    metric
                )
        else:
            flattened_columns.append(
                column
            )

    ranked_agg.columns = flattened_columns

    ranked_agg = ranked_agg.rename(
        columns={
            "global_job_index_size":
                "n_successful_docks",
            "CNNscore_mean":
                "mean_ranked_CNNscore",
            "CNNscore_max":
                "max_ranked_CNNscore",
            "minimizedAffinity_mean":
                "mean_ranked_minimizedAffinity",
            "minimizedAffinity_min":
                "best_ranked_minimizedAffinity",
        }
    )

    combined = combined.merge(
        ranked_agg,
        on="target_input",
        how="left",
    )


# ---------------------------------------------------------------------------
# Лучший docking result
# ---------------------------------------------------------------------------

if (
    not combined.empty
    and not best_results.empty
    and "target_input" in best_results.columns
):
    best_columns = [
        "target_input",
        "route_type",
        "CNNscore",
        "CNNaffinity",
        "minimizedAffinity",
    ]

    available_columns = [
        column
        for column in best_columns
        if column in best_results.columns
    ]

    best_subset = best_results[
        available_columns
    ].copy()

    rename_map = {
        column: f"best_{column}"
        for column in available_columns
        if column != "target_input"
    }

    best_subset = best_subset.rename(
        columns=rename_map
    )

    combined = combined.merge(
        best_subset,
        on="target_input",
        how="left",
    )


# ---------------------------------------------------------------------------
# MD top-3 statistics
# ---------------------------------------------------------------------------

if (
    not combined.empty
    and not md_top3.empty
    and "target_input" in md_top3.columns
):
    md_numeric = md_top3.copy()

    for column in (
        "CNNscore",
        "CNNaffinity",
        "minimizedAffinity",
    ):
        if column in md_numeric.columns:
            md_numeric[column] = pd.to_numeric(
                md_numeric[column],
                errors="coerce",
            )

    md_aggregation = {}

    for column in (
        "CNNscore",
        "CNNaffinity",
        "minimizedAffinity",
    ):
        if column not in md_numeric.columns:
            continue

        best_statistic = (
            "min"
            if column == "minimizedAffinity"
            else "max"
        )

        md_aggregation[column] = [
            "mean",
            best_statistic,
        ]

    if md_aggregation:
        md_agg = (
            md_numeric
            .groupby("target_input")
            .agg(md_aggregation)
            .reset_index()
        )

        renamed_columns = []

        for column in md_agg.columns:
            if isinstance(column, tuple):
                metric, statistic = column

                if metric == "CNNscore":
                    prefix = "md3_CNNscore"

                elif metric == "CNNaffinity":
                    prefix = "md3_CNNaffinity"

                else:
                    prefix = (
                        "md3_minimizedAffinity"
                    )

                statistic_name = (
                    "best"
                    if statistic in {"max", "min"}
                    else "mean"
                )

                renamed_columns.append(
                    f"{prefix}_{statistic_name}"
                )

            else:
                renamed_columns.append(
                    column
                )

        md_agg.columns = renamed_columns

        combined = combined.merge(
            md_agg,
            on="target_input",
            how="left",
        )


# ---------------------------------------------------------------------------
# Сохранение общей таблицы
# ---------------------------------------------------------------------------

if not combined.empty:
    combined = combined.sort_values(
        [
            "n_final_consensus_residues",
            "target_input",
        ],
        ascending=[
            False,
            True,
        ],
        na_position="last",
    )

    combined_path = (
        ANALYSIS_DIR
        / "combined_target_metrics.tsv"
    )

    combined.to_csv(
        combined_path,
        sep="\t",
        index=False,
    )

    print(
        f"📊 Сохранена сводная таблица: "
        f"{combined_path.name}"
    )


# ---------------------------------------------------------------------------
# Target × metric heatmap
# ---------------------------------------------------------------------------

metric_columns = []

if not combined.empty:
    preferred_metrics = [
        "n_final_consensus_residues",
        "n_methods_used",
        "grid_center_x",
        "grid_center_y",
        "grid_center_z",
        "grid_size_x",
        "grid_size_y",
        "grid_size_z",
        "grid_volume_A3",
        "grid_diag_A",
        "consensus_density",
        "n_successful_docks",
        "mean_ranked_CNNscore",
        "max_ranked_CNNscore",
        "mean_ranked_minimizedAffinity",
        "best_ranked_minimizedAffinity",
        "best_CNNscore",
        "best_CNNaffinity",
        "best_minimizedAffinity",
        "md3_CNNscore_mean",
        "md3_CNNscore_best",
        "md3_CNNaffinity_mean",
        "md3_CNNaffinity_best",
        "md3_minimizedAffinity_mean",
        "md3_minimizedAffinity_best",
    ]

    metric_columns = [
        column
        for column in preferred_metrics
        if column in combined.columns
    ]

    if metric_columns:
        metric_matrix = (
            combined
            .set_index("target_input")[
                metric_columns
            ]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
        )

        standard_deviation = (
            metric_matrix.std(
                axis=0,
                ddof=0,
            )
            .replace(
                0,
                np.nan,
            )
        )

        z_scores = (
            metric_matrix
            - metric_matrix.mean(
                axis=0
            )
        ) / standard_deviation

        z_scores = (
            z_scores
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .fillna(0.0)
        )

        plot_heatmap(
            z_scores,
            title=(
                "Target × Metric Overview "
                "(Z-scores)"
            ),
            stem=(
                "target_metric_overview_zscore"
            ),
            center=0.0,
            colorbar_label="Z-score",
        )


# ---------------------------------------------------------------------------
# Таблица для отчёта
# ---------------------------------------------------------------------------

if not combined.empty and metric_columns:
    table_columns = [
        "target_input",
        *metric_columns,
    ]

    report_table = combined[
        table_columns
    ].copy()

    figure_width = min(
        22,
        max(
            10,
            0.8 * len(metric_columns) + 4,
        ),
    )

    figure_height = max(
        4,
        0.45 * len(report_table) + 2,
    )

    figure, axis = plt.subplots(
        figsize=(
            figure_width,
            figure_height,
        )
    )

    axis.axis("off")

    table = axis.table(
        cellText=(
            report_table
            .round(3)
            .astype(str)
            .values
        ),
        colLabels=report_table.columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(7)

    table.scale(
        1,
        1.2,
    )

    axis.set_title(
        "Combined Target Metrics Summary",
        fontsize=12,
        fontweight="bold",
        pad=20,
    )

    save_figure(
        figure,
        "combined_target_metrics_table",
    )

    plt.show()
    plt.close(figure)


# ---------------------------------------------------------------------------
# Pocket-level Jaccard analysis
# ---------------------------------------------------------------------------

def plot_target_jaccard(
    pocket_path: Path,
    jaccard_path: Path,
) -> None:
    """Строит pocket-level и method-level Jaccard heatmaps."""
    pockets = load_tsv(
        pocket_path
    )

    jaccard = load_tsv(
        jaccard_path
    )

    if pockets.empty or jaccard.empty:
        return

    if "target_input" in pockets.columns:
        target_values = (
            pockets["target_input"]
            .dropna()
            .astype(str)
        )

        if target_values.empty:
            target = pocket_path.stem.replace(
                "_pocket_summary",
                "",
            )
        else:
            target = target_values.iloc[0]

    else:
        target = pocket_path.stem.replace(
            "_pocket_summary",
            "",
        )

    target_safe = (
        target
        .replace(" ", "_")
        .replace("/", "_")
    )

    # Pocket × pocket Jaccard.
    pocket_matrix = matrix_from_long(
        jaccard,
        "pocket_1",
        "pocket_2",
        "jaccard",
    )

    if pocket_matrix is not None:
        plot_heatmap(
            pocket_matrix,
            title=(
                f"{target}: "
                "Pocket-to-Pocket Jaccard Index"
            ),
            stem=(
                f"{target_safe}_pocket_jaccard"
            ),
            vmin=0.0,
            vmax=1.0,
            colorbar_label="Jaccard Index",
        )

    required_columns = {
        "method",
        "pocket_name",
    }

    if not required_columns.issubset(
        pockets.columns
    ):
        return

    methods_by_pocket = dict(
        zip(
            pockets["pocket_name"].astype(str),
            pockets["method"].astype(str),
        )
    )

    methods = sorted(
        set(
            methods_by_pocket.values()
        )
    )

    if not methods:
        return

    # Симметричная таблица Jaccard.
    jaccard_lookup = {}

    for _, row in jaccard.iterrows():
        pocket_1 = str(
            row["pocket_1"]
        )

        pocket_2 = str(
            row["pocket_2"]
        )

        value = pd.to_numeric(
            row["jaccard"],
            errors="coerce",
        )

        if pd.isna(value):
            continue

        value = float(value)

        jaccard_lookup[
            (pocket_1, pocket_2)
        ] = value

        jaccard_lookup[
            (pocket_2, pocket_1)
        ] = value

    method_matrix = pd.DataFrame(
        np.nan,
        index=methods,
        columns=methods,
        dtype=float,
    )

    for method_1 in methods:
        pockets_1 = [
            pocket
            for pocket, method
            in methods_by_pocket.items()
            if method == method_1
        ]

        for method_2 in methods:
            pockets_2 = [
                pocket
                for pocket, method
                in methods_by_pocket.items()
                if method == method_2
            ]

            values = [
                jaccard_lookup[
                    (pocket_1, pocket_2)
                ]
                for pocket_1 in pockets_1
                for pocket_2 in pockets_2
                if (
                    pocket_1,
                    pocket_2,
                ) in jaccard_lookup
            ]

            if values:
                method_matrix.loc[
                    method_1,
                    method_2,
                ] = max(values)

    plot_heatmap(
        method_matrix,
        title=(
            f"{target}: "
            "Method Consensus (Max Jaccard)"
        ),
        stem=(
            f"{target_safe}_method_consensus_jaccard"
        ),
        vmin=0.0,
        vmax=1.0,
        colorbar_label="Max Jaccard Index",
    )


# ---------------------------------------------------------------------------
# Обработка всех targets
# ---------------------------------------------------------------------------

for pocket_path in pocket_tables:
    target_name = (
        pocket_path.stem
        .replace(
            "_pocket_summary",
            "",
        )
    )

    jaccard_path = next(
        (
            path
            for path in jaccard_tables
            if path.stem.replace(
                "_jaccard_matrix",
                "",
            )
            == target_name
        ),
        None,
    )

    if jaccard_path is None:
        print(
            f"⚠️ Для {target_name} "
            "не найдена Jaccard-таблица."
        )
        continue

    try:
        plot_target_jaccard(
            pocket_path,
            jaccard_path,
        )

    except (
        OSError,
        KeyError,
        ValueError,
        TypeError,
    ) as exc:
        print(
            f"⚠️ Пропущен target "
            f"{target_name}: {exc}"
        )


# ---------------------------------------------------------------------------
# Общая карта docking grids
# ---------------------------------------------------------------------------

required_grid_columns = {
    "grid_center_x",
    "grid_center_y",
    "grid_size_x",
    "grid_size_y",
}

if (
    not combined.empty
    and required_grid_columns.issubset(
        combined.columns
    )
):
    figure, axis = plt.subplots(
        figsize=(8, 7)
    )

    plotted_grids = 0

    for _, row in combined.iterrows():
        try:
            center_x = float(
                row["grid_center_x"]
            )
            center_y = float(
                row["grid_center_y"]
            )
            size_x = float(
                row["grid_size_x"]
            )
            size_y = float(
                row["grid_size_y"]
            )

        except (
            TypeError,
            ValueError,
        ):
            continue

        if not np.isfinite(
            [
                center_x,
                center_y,
                size_x,
                size_y,
            ]
        ).all():
            continue

        if size_x <= 0 or size_y <= 0:
            continue

        rectangle = Rectangle(
            (
                center_x - size_x / 2.0,
                center_y - size_y / 2.0,
            ),
            size_x,
            size_y,
            fill=False,
            linewidth=1.5,
        )

        axis.add_patch(
            rectangle
        )

        axis.scatter(
            [center_x],
            [center_y],
            s=25,
        )

        axis.text(
            center_x,
            center_y,
            f" {row['target_input']}",
            fontsize=8,
            ha="left",
            va="bottom",
        )

        plotted_grids += 1

    if plotted_grids > 0:
        axis.set_title(
            "Docking Grid Overview (X–Y Projection)",
            fontsize=12,
            fontweight="bold",
        )

        axis.set_xlabel(
            "Grid Center X (Å)"
        )

        axis.set_ylabel(
            "Grid Center Y (Å)"
        )

        axis.grid(
            linestyle="-",
            linewidth=0.5,
            alpha=0.7,
        )

        axis.set_aspect(
            "equal",
            adjustable="datalim",
        )

        save_figure(
            figure,
            "grid_overview_xy",
        )

        plt.show()

    plt.close(figure)


# ---------------------------------------------------------------------------
# Итог
# ---------------------------------------------------------------------------

print(
    "\n🟢 Сводный docking-анализ завершен."
)

print(
    f"📁 Все результаты сохранены в: "
    f"{ANALYSIS_DIR.resolve()}"
)

In [ ]:
# @title 🧠 28. Визуализация консенсусного кармана по сходству Жаккара ✅

from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Конфигурация
# ---------------------------------------------------------------------------

WORK_DIR = Path(
    globals().get(
        "WORKDIR",
        Path.cwd(),
    )
).resolve()

ANALYSIS_DIR = WORK_DIR / "analysis_figures"
ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONSENSUS_THRESHOLD = 0.45


# ---------------------------------------------------------------------------
# Работа с файлами
# ---------------------------------------------------------------------------

def find_pocket_tables() -> list[Path]:
    """Находит все таблицы pocket-level анализа."""
    search_roots = [
        WORK_DIR,
        WORK_DIR.parent,
    ]

    tables = set()

    for root in search_roots:
        if not root.is_dir():
            continue

        tables.update(
            path
            for path in root.rglob("*_pocket_summary.tsv")
            if path.is_file()
        )

    return sorted(tables)


def find_jaccard_table(
    target_name: str,
) -> Path | None:
    """Находит Jaccard-таблицу для указанного target."""
    candidates = [
        WORK_DIR / f"{target_name}_jaccard_matrix.tsv",
        WORK_DIR / "analysis_figures"
        / f"{target_name}_jaccard_matrix.tsv",
    ]

    for path in candidates:
        if path.is_file():
            return path

    for root in (
        WORK_DIR,
        WORK_DIR.parent,
    ):
        if not root.is_dir():
            continue

        matches = list(
            root.rglob(
                f"{target_name}_jaccard_matrix.tsv"
            )
        )

        if matches:
            return matches[0]

    return None


def load_tsv(
    path: Path,
) -> pd.DataFrame:
    """Загружает TSV-файл."""
    return pd.read_csv(
        path,
        sep="\t",
    )


# ---------------------------------------------------------------------------
# Сохранение графиков
# ---------------------------------------------------------------------------

def save_figure(
    figure: plt.Figure,
    stem: str,
) -> tuple[Path, Path]:
    """Сохраняет график в PNG 300 DPI и SVG."""
    png_path = ANALYSIS_DIR / f"{stem}.png"
    svg_path = ANALYSIS_DIR / f"{stem}.svg"

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        svg_path,
        bbox_inches="tight",
    )

    print(
        f"📊 Сохранено: "
        f"{png_path.name} + {svg_path.name}"
    )

    return png_path, svg_path


# ---------------------------------------------------------------------------
# Jaccard
# ---------------------------------------------------------------------------

def jaccard_similarity(
    set_a: set,
    set_b: set,
) -> float:
    """Вычисляет индекс Жаккара."""
    union = set_a | set_b

    if not union:
        return 0.0

    return len(set_a & set_b) / len(union)


def build_jaccard_matrix(
    pockets: OrderedDict[str, list[set]],
) -> tuple[
    list[str],
    list[set],
    np.ndarray,
    list[int],
]:
    """Строит полную матрицу Jaccard для всех карманов."""
    labels = []
    pocket_sets = []
    block_sizes = []

    for algorithm, algorithm_pockets in pockets.items():
        block_sizes.append(
            len(algorithm_pockets)
        )

        for index, pocket in enumerate(
            algorithm_pockets,
            start=1,
        ):
            labels.append(
                f"{algorithm}\nP{index}"
            )

            pocket_sets.append(
                set(pocket)
            )

    if not pocket_sets:
        raise ValueError(
            "Не найдено ни одного предсказанного кармана."
        )

    n_pockets = len(pocket_sets)

    matrix = np.zeros(
        (
            n_pockets,
            n_pockets,
        ),
        dtype=float,
    )

    for index_1 in range(n_pockets):
        for index_2 in range(
            index_1,
            n_pockets,
        ):
            similarity = jaccard_similarity(
                pocket_sets[index_1],
                pocket_sets[index_2],
            )

            matrix[
                index_1,
                index_2,
            ] = similarity

            matrix[
                index_2,
                index_1,
            ] = similarity

    return (
        labels,
        pocket_sets,
        matrix,
        block_sizes,
    )


def select_consensus_pocket(
    matrix: np.ndarray,
) -> int:
    """
    Выбирает центральный карман по максимальному
    среднему сходству с остальными карманами.
    """
    if matrix.shape[0] == 1:
        return 0

    similarity_sum = (
        matrix.sum(axis=1)
        - np.diag(matrix)
    )

    similarity_mean = (
        similarity_sum
        / (matrix.shape[0] - 1)
    )

    return int(
        np.argmax(similarity_mean)
    )


# ---------------------------------------------------------------------------
# Загрузка реальных pocket-данных
# ---------------------------------------------------------------------------

def load_real_pockets(
    pocket_path: Path,
) -> tuple[str, OrderedDict[str, list[set]]]:
    """
    Загружает реальные pocket'ы из pocket_summary.tsv.

    Ожидаются столбцы:
    - pocket_name
    - method
    - residue/residue_id/residue_number
    """
    dataframe = load_tsv(
        pocket_path
    )

    required_columns = {
        "pocket_name",
        "method",
    }

    if not required_columns.issubset(
        dataframe.columns
    ):
        raise ValueError(
            f"{pocket_path.name}: отсутствуют "
            f"обязательные столбцы "
            f"{required_columns}."
        )

    residue_column = next(
        (
            column
            for column in (
                "residue",
                "residue_id",
                "residue_number",
                "resSeq",
            )
            if column in dataframe.columns
        ),
        None,
    )

    if residue_column is None:
        raise ValueError(
            f"{pocket_path.name}: не найден столбец "
            "с идентификаторами остатков."
        )

    target = (
        str(
            dataframe["target_input"]
            .dropna()
            .iloc[0]
        )
        if "target_input" in dataframe.columns
        and not dataframe["target_input"]
        .dropna()
        .empty
        else pocket_path.stem.replace(
            "_pocket_summary",
            "",
        )
    )

    pockets_by_algorithm = OrderedDict()

    grouped = dataframe.groupby(
        [
            "method",
            "pocket_name",
        ],
        sort=True,
    )

    for (
        method,
        pocket_name,
    ), group in grouped:
        residues = set(
            pd.to_numeric(
                group[residue_column],
                errors="coerce",
            )
            .dropna()
            .astype(int)
            .tolist()
        )

        if not residues:
            continue

        pockets_by_algorithm.setdefault(
            str(method),
            [],
        ).append(
            residues
        )

    if not pockets_by_algorithm:
        raise ValueError(
            f"{pocket_path.name}: "
            "не найдено непустых карманов."
        )

    return (
        target,
        pockets_by_algorithm,
    )


# ---------------------------------------------------------------------------
# Основная визуализация
# ---------------------------------------------------------------------------

def plot_consensus_pocket(
    pockets_by_algorithm: OrderedDict[str, list[set]],
    target: str,
    save_stem: str,
) -> None:
    """Строит Jaccard heatmap и сходство с центральным карманом."""
    (
        labels,
        pocket_sets,
        matrix,
        block_sizes,
    ) = build_jaccard_matrix(
        pockets_by_algorithm
    )

    consensus_index = select_consensus_pocket(
        matrix
    )

    consensus_label = labels[
        consensus_index
    ]

    consensus_scores = matrix[
        consensus_index
    ].copy()

    order = np.argsort(
        -consensus_scores,
        kind="stable",
    )

    sorted_labels = [
        labels[index]
        for index in order
    ]

    sorted_scores = (
        consensus_scores[order]
    )

    figure = plt.figure(
        figsize=(18, 10)
    )

    grid = figure.add_gridspec(
        1,
        2,
        width_ratios=(2.2, 1.0),
        wspace=0.28,
    )

    # -----------------------------------------------------------------------
    # Полная Jaccard-матрица
    # -----------------------------------------------------------------------

    axis_heatmap = figure.add_subplot(
        grid[0, 0]
    )

    image = axis_heatmap.imshow(
        matrix,
        vmin=0.0,
        vmax=1.0,
        interpolation="nearest",
    )

    colorbar = figure.colorbar(
        image,
        ax=axis_heatmap,
        fraction=0.046,
        pad=0.04,
    )

    colorbar.set_label(
        "Сходство Жаккара"
    )

    positions = np.arange(
        len(labels)
    )

    axis_heatmap.set_xticks(
        positions
    )

    axis_heatmap.set_yticks(
        positions
    )

    axis_heatmap.set_xticklabels(
        labels,
        fontsize=8,
    )

    axis_heatmap.set_yticklabels(
        labels,
        fontsize=8,
    )

    plt.setp(
        axis_heatmap.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    axis_heatmap.set_title(
        "Попарное сходство Жаккара "
        "между предсказанными карманами",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )

    for row_index in range(
        matrix.shape[0]
    ):
        for column_index in range(
            matrix.shape[1]
        ):
            value = matrix[
                row_index,
                column_index,
            ]

            axis_heatmap.text(
                column_index,
                row_index,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=7,
            )

    cumulative_sizes = np.cumsum(
        block_sizes
    )

    for boundary in cumulative_sizes[:-1]:
        position = boundary - 0.5

        axis_heatmap.axhline(
            position,
            linewidth=1.2,
        )

        axis_heatmap.axvline(
            position,
            linewidth=1.2,
        )

    axis_heatmap.add_patch(
        plt.Rectangle(
            (
                consensus_index - 0.5,
                consensus_index - 0.5,
            ),
            1.0,
            1.0,
            fill=False,
            linewidth=2.5,
        )
    )

    # -----------------------------------------------------------------------
    # Ранжирование относительно консенсусного кармана
    # -----------------------------------------------------------------------

    axis_rank = figure.add_subplot(
        grid[0, 1]
    )

    y_positions = np.arange(
        len(sorted_labels)
    )

    bars = axis_rank.barh(
        y_positions,
        sorted_scores,
        linewidth=0.6,
    )

    consensus_sorted_index = (
        sorted_labels.index(
            consensus_label
        )
    )

    bars[
        consensus_sorted_index
    ].set_linewidth(
        1.8
    )

    axis_rank.set_yticks(
        y_positions
    )

    axis_rank.set_yticklabels(
        sorted_labels,
        fontsize=8,
    )

    axis_rank.invert_yaxis()

    axis_rank.set_xlim(
        0.0,
        1.0,
    )

    axis_rank.set_xlabel(
        "Сходство Жаккара"
    )

    axis_rank.set_title(
        "Сходство с центральным "
        "консенсусным карманом",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )

    for index, score in enumerate(
        sorted_scores
    ):
        axis_rank.text(
            min(
                score + 0.02,
                0.97,
            ),
            index,
            f"{score:.2f}",
            va="center",
            fontsize=8,
        )

    axis_rank.axvline(
        CONSENSUS_THRESHOLD,
        linestyle="--",
        linewidth=1.2,
    )

    axis_rank.text(
        CONSENSUS_THRESHOLD + 0.01,
        len(sorted_labels) - 0.5,
        (
            f"порог = "
            f"{CONSENSUS_THRESHOLD:.2f}"
        ),
        fontsize=8,
        va="bottom",
    )

    figure.suptitle(
        (
            f"{target}: визуализация "
            "консенсусного кармана"
        ),
        fontsize=16,
        fontweight="bold",
    )

    figure.tight_layout(
        rect=(0, 0, 1, 0.95)
    )

    save_figure(
        figure,
        save_stem,
    )

    plt.show()
    plt.close(figure)

    # -----------------------------------------------------------------------
    # Текстовый итог
    # -----------------------------------------------------------------------

    high_similarity = [
        (
            label,
            score,
        )
        for label, score in zip(
            sorted_labels,
            sorted_scores,
        )
        if score >= CONSENSUS_THRESHOLD
    ]

    print(
        f"\n🎯 Target: {target}"
    )

    print(
        f"Центральный консенсусный карман: "
        f"{consensus_label.replace(chr(10), ' ')}"
    )

    print(
        "Карманов со сходством "
        f"≥ {CONSENSUS_THRESHOLD:.2f}: "
        f"{len(high_similarity)}"
    )

    if high_similarity:
        print(
            "Карманов, входящих в консенсусную "
            "группу:"
        )

        for label, score in high_similarity:
            print(
                f"  • "
                f"{label.replace(chr(10), ' ')}: "
                f"Jaccard = {score:.3f}"
            )


# ---------------------------------------------------------------------------
# Запуск для всех реальных targets
# ---------------------------------------------------------------------------

print(
    "--- Визуализация консенсусных карманов ---"
)

pocket_tables = find_pocket_tables()

if not pocket_tables:
    raise FileNotFoundError(
        "Не найдены *_pocket_summary.tsv."
    )

print(
    f"Найдено pocket-таблиц: "
    f"{len(pocket_tables)}"
)

processed_targets = 0

for pocket_path in pocket_tables:
    try:
        target, pockets_by_algorithm = (
            load_real_pockets(
                pocket_path
            )
        )

        jaccard_path = find_jaccard_table(
            target
        )

        if jaccard_path is None:
            print(
                f"⚠️ {target}: "
                "Jaccard-таблица не найдена."
            )
            continue

        plot_consensus_pocket(
            pockets_by_algorithm,
            target=target,
            save_stem=(
                f"{target}_consensus_pocket_selection"
            ),
        )

        processed_targets += 1

    except (
        OSError,
        KeyError,
        ValueError,
        TypeError,
    ) as exc:
        print(
            f"⚠️ Пропущен target "
            f"{pocket_path.name}: {exc}"
        )


# ---------------------------------------------------------------------------
# Итог
# ---------------------------------------------------------------------------

print(
    "\n🟢 Визуализация консенсусных карманов завершена."
)

print(
    f"Обработано targets: "
    f"{processed_targets}"
)

print(
    f"📁 Результаты: "
    f"{ANALYSIS_DIR.resolve()}"
)